
# I-GDP-PA: CelebA participant-count sweep

This notebook runs the agreed nested sweep

\[
p\in\{10,30,50,70,90\}.
\]

One deterministic 90-participant master allocation is created. Every participant has 100 identities with a 70/15/15 identity-level train/validation/test split, identities are globally disjoint, and one image per identity remains reserved for privacy linkage. Smaller conditions use strict prefixes of the 90-participant master. The public rank-400 SVD still uses exactly 10,000 identity-exclusive CelebA images.

The centered unit-isometric anchor has \(r=1600\), \(h=400\), \(A^\top A=I_h\), and zero column mean. A validation-only \(p=10\) calibration chooses the main anchor-noise upper bound. Each full condition then contains 100 uniform anchor-noise draws and a separate deterministic \(v=40=\sqrt r\) I-GDP-equivalence control.



## Calibration and test discipline

Calibration opens only participant-1 training and validation identities. It fits C-GDP and I-GDP controls and evaluates AA-I-GDP (anchor noise) at

\[
v\in\{0,0.0625,0.125,0.25,0.5,1,2\}.
\]

All candidate levels scale the same frozen standard-normal anchor-noise tensor. A level is linkage-saturated when its strongest ten-way linkage estimate is within 0.03 of I-GDP, its identity-cluster 95% confidence interval contains the 0.10 chance level, and that interval overlaps the I-GDP interval. The right endpoint of the first two consecutive saturated levels becomes \(v_{\max}\); balanced accuracy must remain at least 0.50 but is otherwise reported as an experimental response rather than used to match I-GDP. Before freezing the bound, validation-only controls require usable public-SVD and clean C-GDP utility, exact C-GDP recovery, and converged exact zero-anchor recovery. If either the adequacy gate or the linkage-saturation rule fails, the notebook stops before opening test pixels.

After calibration, each participant-count condition performs 207 fits: clean C-GDP and I-GDP; 100 C-GDP (private-data noise) draws with \(\lambda_x\sim U[0,0.5]\); 100 AA-I-GDP (anchor noise) draws with \(v\sim U[0,v_{\max}]\); zero and upper endpoints for both noisy families; and one AA-I-GDP control at \(v=40\). The \(v=40\) control is outside the uniform range. GPM cap hits and optimization time remain recorded rather than being discarded.



## Drive-backed persistence and resume

Drive is authoritative for the raw CelebA archive, RGB64 shards, public-SVD basis, 90-participant master allocation, shared anchor and GDP parameter bank, calibration manifest, per-fit records, checkpoints, plots, and runtime summaries. Each participant count has a separate result directory. Rerunning all cells after a runtime disconnect validates every lock and skips completed fit IDs before continuing with the first unfinished fit. The reviewed legacy right-censored calibration is resolved from its nine immutable saved records without refitting; the frozen manifest separately records the record-generation and resolver module hashes.


In [ ]:

#@title 1. Run controls { display-mode: "form" }
EXECUTE_FULL_SWEEP = True #@param {type:"boolean"}
EXECUTE_EXPLORATORY_P70_P90 = False #@param {type:"boolean"}
DEVICE = "auto" #@param ["auto", "cuda", "cpu"]
DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/I-GDP-PA/CelebA_PublicSVD_h400_identity100_p10to90_anchor_sweep_v3" #@param {type:"string"}
DRIVE_CACHE_ROOT = "/content/drive/MyDrive/I-GDP-PA/CelebA_PublicSVD_identity100_cache_v1" #@param {type:"string"}
LOCAL_CACHE_ROOT = "/content/igdp_pa_celeba_p10to90_anchor_sweep_v3" #@param {type:"string"}
SVD_Q = 420 #@param {type:"integer"}
SVD_NITER = 4 #@param {type:"integer"}

P_SWEEP = (10, 30, 50, 70, 90)
if P_SWEEP != (10, 30, 50, 70, 90):
    raise AssertionError("The reviewed participant grid changed")


In [ ]:

# 2. Install dependencies, mount Drive, and verify persistent roots
import os
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "kagglehub", "gdown", "pandas", "scikit-learn", "scikit-image",
        "matplotlib", "pillow",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "facenet-pytorch"
    ])
    from google.colab import drive

    def _drive_ready():
        return os.path.ismount("/content/drive") and Path(
            "/content/drive/MyDrive"
        ).is_dir()

    mount_errors = []
    for attempt in range(3):
        if _drive_ready():
            break
        try:
            drive.mount(
                "/content/drive",
                force_remount=attempt > 0,
                timeout_ms=180_000,
            )
        except ValueError as exc:
            mount_errors.append(f"attempt {attempt + 1}: {exc}")
            if attempt < 2:
                time.sleep(5)
    if not _drive_ready():
        raise RuntimeError(
            "Google Drive did not mount after three attempts: "
            + "; ".join(mount_errors)
        )

RESULTS_ROOT = Path(DRIVE_RESULTS_ROOT)
CACHE_ROOT = Path(DRIVE_CACHE_ROOT)
LOCAL_ROOT = Path(LOCAL_CACHE_ROOT)
if IN_COLAB:
    drive_prefix = Path("/content/drive/MyDrive")
    for label, path in (("results", RESULTS_ROOT), ("cache", CACHE_ROOT)):
        try:
            path.relative_to(drive_prefix)
        except ValueError as exc:
            raise ValueError(f"{label} root must be on mounted My Drive: {path}") from exc
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
print({
    "results": str(RESULTS_ROOT),
    "cache": str(CACHE_ROOT),
    "local_mirror": str(LOCAL_ROOT),
    "drive_mounted": (not IN_COLAB) or _drive_ready(),
})


## 3. Embedded protocol and experiment implementation


In [ ]:
%%writefile core_protocols.py
"""Float64 PyTorch implementations of the I-GDP paper protocols and attacks.

Records are row vectors.  Every GDP transform is therefore a right action:
``X @ O`` and ``A @ O``.  The module intentionally keeps model training out of
scope so its algebraic self-tests can run quickly on CPU or CUDA.  The shared
empirical reduction is the reference's uncentered, unnormalized public Gaussian
random projection; reconstruction uses its Moore--Penrose inverse rather than
assuming an orthonormal basis.
"""

from __future__ import annotations

import os
import random
import time
from dataclasses import dataclass
from typing import List, Literal, Optional, Sequence, Tuple, Union

import torch

Tensor = torch.Tensor
TensorOrBlocks = Union[Tensor, Sequence[Tensor]]
PROTOCOL_DTYPE = torch.float64


@dataclass
class GaussianProjectionReducer:
    """Low-level public Gaussian linear reducer.

    ``F_pinv`` is cached because Gaussian projection columns are not
    orthonormal and hence the decoder is not generally ``F.T``.
    """

    F: Tensor
    singular_values: Tensor
    F_pinv: Tensor

    def transform(self, x: Tensor) -> Tensor:
        return _as_tensor(x, self.F.device) @ self.F

    def transform_blocks(self, blocks: Sequence[Tensor]) -> List[Tensor]:
        return [self.transform(x) for x in blocks]

    def decode(self, z: Tensor) -> Tensor:
        return _as_tensor(z, self.F.device) @ self.F_pinv

@dataclass
class GPMDiagnostics:
    """Numerical diagnostics for PA-I-GDP's block-Gram GPM."""

    iterations: int
    converged: bool
    elapsed_seconds: float
    anchor_singular_values: Tensor
    anchor_sigma_min: float
    anchor_sigma_max: float
    anchor_condition: float
    upload_singular_values: List[Tensor]
    gram_eigenvalues: Tensor
    eigengap: float
    relative_eigengap: float
    multistart_objective_gap: float
    gopp_residual: float
    relative_gopp_residual: float
    fixed_point_residual: float
    objective_history: List[float]
    step_history: List[float]


@dataclass
class ProtocolResult:
    """Protocol transcript plus the matrix supplied to the analyst's model."""

    name: str
    private_uploads: List[Tensor]
    collaboration_blocks: List[Tensor]
    collaboration: Tensor
    rotations: List[Tensor]
    translations: List[Tensor]
    anchor_uploads: Optional[List[Tensor]] = None
    alignment_rotations: Optional[List[Tensor]] = None
    private_noises: Optional[List[Tensor]] = None
    anchor_noises: Optional[List[Tensor]] = None
    diagnostics: Optional[GPMDiagnostics] = None
    reference: Optional[int] = None


@dataclass
class AttackResult:
    """Estimated target GDP parameters and reduced-space reconstruction."""

    rotation: Tensor
    translation: Tensor
    reduced: Tensor
    method: str


def seed_everything(seed: int, deterministic: bool = True) -> None:
    """Seed Python, NumPy when present, and PyTorch deterministically."""

    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.use_deterministic_algorithms(True, warn_only=True)
        if torch.backends.cudnn.is_available():
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = True


def make_generator(seed: int, device: Union[str, torch.device] = "cpu") -> torch.Generator:
    """Return a seeded generator on ``device``."""

    dev = torch.device(device)
    return torch.Generator(device=dev).manual_seed(seed)


def _as_tensor(x: Tensor, device: Optional[torch.device] = None) -> Tensor:
    dev = x.device if device is None and isinstance(x, Tensor) else device
    return torch.as_tensor(x, dtype=PROTOCOL_DTYPE, device=dev)


def _blocks(blocks: Sequence[Tensor], device: Optional[torch.device] = None) -> List[Tensor]:
    if not blocks:
        raise ValueError("At least one participant block is required.")
    dev = torch.device(device) if device is not None else torch.as_tensor(blocks[0]).device
    out = [_as_tensor(x, dev) for x in blocks]
    h = out[0].shape[1]
    if any(x.ndim != 2 or x.shape[1] != h for x in out):
        raise ValueError("All participant blocks must be 2-D with the same feature dimension.")
    return out


def _randn(shape: Tuple[int, ...], ref: Tensor, generator: Optional[torch.Generator]) -> Tensor:
    return torch.randn(shape, dtype=PROTOCOL_DTYPE, device=ref.device, generator=generator)


def haar_orthogonal(
    h: int,
    *,
    generator: Optional[torch.Generator] = None,
    device: Union[str, torch.device] = "cpu",
) -> Tensor:
    """Sample a Haar orthogonal matrix in O(h) by sign-corrected Gaussian QR."""

    z = torch.randn((h, h), dtype=PROTOCOL_DTYPE, device=device, generator=generator)
    q, r = torch.linalg.qr(z)
    signs = torch.where(torch.diagonal(r) >= 0, 1.0, -1.0).to(q)
    return q * signs.unsqueeze(0)


def sample_gdp_parameters(
    num_users: int,
    h: int,
    *,
    translation_scale: float = 1.0,
    generator: Optional[torch.Generator] = None,
    device: Union[str, torch.device] = "cpu",
) -> Tuple[List[Tensor], List[Tensor]]:
    """Sample independent ``(O_i, Psi_i)`` pairs."""

    rotations = [haar_orthogonal(h, generator=generator, device=device) for _ in range(num_users)]
    translations = [
        translation_scale
        * torch.randn((h,), dtype=PROTOCOL_DTYPE, device=device, generator=generator)
        for _ in range(num_users)
    ]
    return rotations, translations


def make_shared_gaussian_projection(
    raw_dimension: int,
    h: int,
    *,
    seed: int,
    device: Union[str, torch.device] = "cpu",
) -> GaussianProjectionReducer:
    """Create the public Gaussian map ``F`` with iid ``N(0, 1)`` entries.

    For row-oriented observations the sole reduction relation is
    ``X_tilde = X @ F``.  A tall Gaussian matrix has full column rank with
    probability one, and its cached decoder is
    ``F^dagger = (F.T @ F)^(-1) @ F.T`` (computed numerically by ``pinv``).
    """

    if raw_dimension < 1:
        raise ValueError("raw_dimension must be positive.")
    if not 1 <= h <= raw_dimension:
        raise ValueError("h must be between 1 and raw_dimension.")
    dev = torch.device(device)
    generator = make_generator(seed, dev)
    F = torch.randn(
        (raw_dimension, h),
        dtype=PROTOCOL_DTYPE,
        device=dev,
        generator=generator,
    )
    singular_values = torch.linalg.svdvals(F)
    tolerance = torch.finfo(PROTOCOL_DTYPE).eps * max(F.shape) * singular_values[0]
    if singular_values[-1] <= tolerance:
        raise RuntimeError("Seeded Gaussian projection was numerically rank deficient.")
    return GaussianProjectionReducer(
        F=F,
        singular_values=singular_values,
        F_pinv=torch.linalg.pinv(F),
    )


def reduce_blocks(blocks: Sequence[Tensor], F: Tensor) -> List[Tensor]:
    """Apply the shared composition ``Xtilde_i = X_i F``."""

    F = _as_tensor(F)
    return [_as_tensor(x, F.device) @ F for x in blocks]


def decode_reduced(z: Tensor, F: Tensor, F_pinv: Optional[Tensor] = None) -> Tensor:
    """Decode a reduced reconstruction as ``Xhat = Xtildehat F^dagger``."""

    F = _as_tensor(F)
    pinv = torch.linalg.pinv(F) if F_pinv is None else _as_tensor(F_pinv, F.device)
    return _as_tensor(z, F.device) @ pinv


def projection_floor(x: Tensor, F: Tensor) -> Tensor:
    """Return the public map's Moore--Penrose linear reconstruction control."""

    F = _as_tensor(F)
    return decode_reduced(_as_tensor(x, F.device) @ F, F)


def orthogonal_polar(matrix: Tensor) -> Tensor:
    """Project a square matrix onto O(h) using ``U V^T``."""

    u, _, vh = torch.linalg.svd(matrix, full_matrices=False)
    return u @ vh


def blockwise_polar(stacked: Tensor, h: int) -> Tensor:
    """Apply the polar projection to each h-by-h vertical block."""

    if stacked.ndim != 2 or stacked.shape[1] != h or stacked.shape[0] % h:
        raise ValueError("Expected a (p*h)-by-h stacked matrix.")
    return torch.cat([orthogonal_polar(b) for b in stacked.split(h, dim=0)], dim=0)


def _parameters(
    p: int,
    h: int,
    ref: Tensor,
    rotations: Optional[Sequence[Tensor]],
    translations: Optional[Sequence[Tensor]],
    generator: Optional[torch.Generator],
    translation_scale: float,
) -> Tuple[List[Tensor], List[Tensor]]:
    if rotations is None or translations is None:
        sampled_o, sampled_p = sample_gdp_parameters(
            p,
            h,
            translation_scale=translation_scale,
            generator=generator,
            device=ref.device,
        )
        rotations = sampled_o if rotations is None else rotations
        translations = sampled_p if translations is None else translations
    if len(rotations) != p or len(translations) != p:
        raise ValueError("One rotation and translation are required per participant.")
    os_ = [_as_tensor(o, ref.device) for o in rotations]
    psis = [_as_tensor(psi, ref.device).reshape(h) for psi in translations]
    eye = torch.eye(h, dtype=PROTOCOL_DTYPE, device=ref.device)
    if any(o.shape != (h, h) or not torch.allclose(o.mT @ o, eye, atol=1e-8, rtol=1e-8) for o in os_):
        raise ValueError("Every O_i must be h-by-h and orthogonal.")
    return os_, psis


def _noise_blocks(
    blocks: Sequence[Tensor],
    scale: float,
    noises: Optional[Sequence[Tensor]],
    generator: Optional[torch.Generator],
) -> List[Tensor]:
    if noises is None:
        return [scale * _randn(tuple(x.shape), x, generator) for x in blocks]
    if len(noises) != len(blocks):
        raise ValueError("One noise matrix is required per participant.")
    out = [_as_tensor(e, x.device) for e, x in zip(noises, blocks)]
    if any(e.shape != x.shape for e, x in zip(out, blocks)):
        raise ValueError("Noise matrices must match their participant block shapes.")
    return out


def _result(
    name: str,
    uploads: List[Tensor],
    blocks: List[Tensor],
    rotations: List[Tensor],
    translations: List[Tensor],
    **kwargs,
) -> ProtocolResult:
    return ProtocolResult(
        name=name,
        private_uploads=uploads,
        collaboration_blocks=blocks,
        collaboration=torch.cat(blocks, dim=0),
        rotations=rotations,
        translations=translations,
        **kwargs,
    )


def c_gdp(
    x_blocks: Sequence[Tensor],
    *,
    rotation: Optional[Tensor] = None,
    translation: Optional[Tensor] = None,
    generator: Optional[torch.Generator] = None,
    translation_scale: float = 1.0,
) -> ProtocolResult:
    """Common GDP: all users upload ``Xtilde_i O + Psi``."""

    xs = _blocks(x_blocks)
    h = xs[0].shape[1]
    os_, psis = _parameters(1, h, xs[0], None if rotation is None else [rotation], None if translation is None else [translation], generator, translation_scale)
    O, psi = os_[0], psis[0]
    uploads = [x @ O + psi for x in xs]
    return _result("C-GDP", uploads, uploads, [O] * len(xs), [psi] * len(xs))


def c_gdp_an(
    x_blocks: Sequence[Tensor],
    sigma: float,
    *,
    rotation: Optional[Tensor] = None,
    translation: Optional[Tensor] = None,
    noises: Optional[Sequence[Tensor]] = None,
    generator: Optional[torch.Generator] = None,
    translation_scale: float = 1.0,
) -> ProtocolResult:
    """Common GDP with iid private-upload Gaussian noise."""

    base = c_gdp(
        x_blocks,
        rotation=rotation,
        translation=translation,
        generator=generator,
        translation_scale=translation_scale,
    )
    noise = _noise_blocks(base.private_uploads, sigma, noises, generator)
    uploads = [y + e for y, e in zip(base.private_uploads, noise)]
    return _result("C-GDP-AN", uploads, uploads, base.rotations, base.translations, private_noises=noise)


def i_gdp(
    x_blocks: Sequence[Tensor],
    *,
    rotations: Optional[Sequence[Tensor]] = None,
    translations: Optional[Sequence[Tensor]] = None,
    generator: Optional[torch.Generator] = None,
    translation_scale: float = 1.0,
) -> ProtocolResult:
    """Independent GDP with no analyst-side alignment."""

    xs = _blocks(x_blocks)
    os_, psis = _parameters(len(xs), xs[0].shape[1], xs[0], rotations, translations, generator, translation_scale)
    uploads = [x @ o + psi for x, o, psi in zip(xs, os_, psis)]
    return _result("I-GDP", uploads, uploads, os_, psis)


def aa_alignment(
    private_uploads: Sequence[Tensor],
    anchor_uploads: Sequence[Tensor],
    reference: int = 0,
) -> Tuple[List[Tensor], List[Tensor], List[Tensor]]:
    """Apply the draft's AA map using ``Bbar_i.T @ Bbar_s``."""

    ys = _blocks(private_uploads)
    bs_upload = _blocks(anchor_uploads, ys[0].device)
    p, h = len(ys), ys[0].shape[1]
    if len(bs_upload) != p or not 0 <= reference < p:
        raise ValueError("Invalid number of anchors or reference index.")
    means = [b.mean(dim=0) for b in bs_upload]
    centered = [b - mean for b, mean in zip(bs_upload, means)]
    rotations: List[Tensor] = []
    eye = torch.eye(h, dtype=PROTOCOL_DTYPE, device=ys[0].device)
    for i in range(p):
        rotations.append(eye if i == reference else orthogonal_polar(centered[i].mT @ centered[reference]))
    ref_mean = means[reference]
    aligned = [(y - mean) @ r + ref_mean for y, mean, r in zip(ys, means, rotations)]
    return aligned, rotations, means


def aa_i_gdp(
    x_blocks: Sequence[Tensor],
    anchor: Tensor,
    *,
    reference: int = 0,
    rotations: Optional[Sequence[Tensor]] = None,
    translations: Optional[Sequence[Tensor]] = None,
    generator: Optional[torch.Generator] = None,
    translation_scale: float = 1.0,
) -> ProtocolResult:
    """Noiseless anchor-aligned independent GDP."""

    xs = _blocks(x_blocks)
    A = _as_tensor(anchor, xs[0].device)
    p, h = len(xs), xs[0].shape[1]
    if A.ndim != 2 or A.shape[1] != h:
        raise ValueError("anchor must be r-by-h.")
    os_, psis = _parameters(p, h, xs[0], rotations, translations, generator, translation_scale)
    ys = [x @ o + psi for x, o, psi in zip(xs, os_, psis)]
    Bs = [A @ o + psi for o, psi in zip(os_, psis)]
    aligned, rs, _ = aa_alignment(ys, Bs, reference)
    return _result(
        "AA-I-GDP",
        ys,
        aligned,
        os_,
        psis,
        anchor_uploads=Bs,
        alignment_rotations=rs,
        reference=reference,
    )


def aa_i_gdp_an(
    x_blocks: Sequence[Tensor],
    anchor: Tensor,
    sigma: float,
    *,
    reference: int = 0,
    rotations: Optional[Sequence[Tensor]] = None,
    translations: Optional[Sequence[Tensor]] = None,
    noises: Optional[Sequence[Tensor]] = None,
    generator: Optional[torch.Generator] = None,
    translation_scale: float = 1.0,
) -> ProtocolResult:
    """AA-I-GDP with Gaussian noise on private uploads only."""

    base = aa_i_gdp(
        x_blocks,
        anchor,
        reference=reference,
        rotations=rotations,
        translations=translations,
        generator=generator,
        translation_scale=translation_scale,
    )
    noise = _noise_blocks(base.private_uploads, sigma, noises, generator)
    ys = [y + e for y, e in zip(base.private_uploads, noise)]
    aligned, rs, _ = aa_alignment(ys, base.anchor_uploads or [], reference)
    return _result(
        "AA-I-GDP-AN",
        ys,
        aligned,
        base.rotations,
        base.translations,
        anchor_uploads=base.anchor_uploads,
        alignment_rotations=rs,
        private_noises=noise,
        reference=reference,
    )


def build_block_gram(centered_anchor_uploads: Sequence[Tensor]) -> Tensor:
    """Build PA-I-GDP blocks ``G_ij = Bbar_i.T @ Bbar_j``."""

    Bs = _blocks(centered_anchor_uploads)
    rows = [torch.cat([bi.mT @ bj for bj in Bs], dim=1) for bi in Bs]
    G = torch.cat(rows, dim=0)
    return 0.5 * (G + G.mT)


def gpm_align(
    centered_anchor_uploads: Sequence[Tensor],
    *,
    raw_anchor: Optional[Tensor] = None,
    max_iters: int = 200,
    tol: float = 1e-10,
    n_random_starts: int = 0,
    restart_seed: int = 0,
) -> Tuple[List[Tensor], GPMDiagnostics]:
    """Run spectral initialization and blockwise-polar GPM.

    The implementation deliberately avoids materializing the
    ``(p*h)``-by-``(p*h)`` block Gram matrix.  With the centered anchor
    uploads concatenated as ``D = [Bbar_1 ... Bbar_p]``, its action is
    ``G @ R = D.T @ (D @ R)``.  The leading eigenspace of ``G = D.T @ D``
    is obtained from a thin SVD of ``D``.
    """

    started = time.perf_counter()
    Bs = _blocks(centered_anchor_uploads)
    p, h = len(Bs), Bs[0].shape[1]
    D = torch.cat(Bs, dim=1)
    if D.shape[0] < h:
        raise ValueError("At least h centered anchor rows are required for spectral initialization.")
    _, singular_values, vh = torch.linalg.svd(D, full_matrices=False)
    gram_eigenvalues = torch.zeros(p * h, dtype=PROTOCOL_DTYPE, device=D.device)
    gram_eigenvalues[: singular_values.numel()] = singular_values.square()
    stacked = blockwise_polar(vh[:h].mT, h)

    def gram_apply(r: Tensor) -> Tensor:
        return D.mT @ (D @ r)

    def objective(r: Tensor) -> float:
        # trace(R.T @ D.T @ D @ R) = ||D @ R||_F^2.
        return float(torch.linalg.norm(D @ r).square().detach().cpu())

    objectives = [objective(stacked)]
    steps: List[float] = []
    converged = False
    for _ in range(max_iters):
        nxt = blockwise_polar(gram_apply(stacked), h)
        step = float(torch.linalg.norm(nxt - stacked).detach().cpu())
        steps.append(step)
        stacked = nxt
        objectives.append(objective(stacked))
        if step <= tol:
            converged = True
            break
    rs = list(stacked.split(h, dim=0))
    template = sum((b @ r for b, r in zip(Bs, rs)), torch.zeros_like(Bs[0])) / p
    residual = sum(float(torch.linalg.norm(template - b @ r).square().detach().cpu()) for b, r in zip(Bs, rs))
    denom = sum(float(torch.linalg.norm(b).square().detach().cpu()) for b in Bs)
    fixed = torch.linalg.norm(blockwise_polar(gram_apply(stacked), h) - stacked) / max(1.0, float(torch.linalg.norm(stacked).detach().cpu()))
    gap = (
        float((gram_eigenvalues[h - 1] - gram_eigenvalues[h]).detach().cpu())
        if gram_eigenvalues.numel() > h
        else float("nan")
    )
    lambda_h = abs(float(gram_eigenvalues[h - 1].detach().cpu()))
    top_scale = max(lambda_h, torch.finfo(PROTOCOL_DTYPE).eps)
    spectral_objective = objectives[-1]
    best_restart_objective = spectral_objective
    if n_random_starts < 0:
        raise ValueError("n_random_starts must be nonnegative.")
    if n_random_starts:
        restart_generator = make_generator(restart_seed, Bs[0].device)
        for _ in range(n_random_starts):
            trial = torch.cat(
                [haar_orthogonal(h, generator=restart_generator, device=Bs[0].device) for _ in range(p)],
                dim=0,
            )
            for _ in range(max_iters):
                trial_next = blockwise_polar(gram_apply(trial), h)
                trial_step = float(torch.linalg.norm(trial_next - trial).detach().cpu())
                trial = trial_next
                if trial_step <= tol:
                    break
            best_restart_objective = max(best_restart_objective, objective(trial))
    if raw_anchor is None:
        anchor_svals = torch.empty(0, dtype=PROTOCOL_DTYPE, device=Bs[0].device)
        anchor_sigma_min = anchor_sigma_max = anchor_condition = float("nan")
    else:
        A = _as_tensor(raw_anchor, Bs[0].device)
        anchor_svals = torch.linalg.svdvals(A - A.mean(dim=0))
        anchor_sigma_max = float(anchor_svals[0].detach().cpu())
        anchor_sigma_min = float(anchor_svals[-1].detach().cpu())
        anchor_condition = anchor_sigma_max / anchor_sigma_min if anchor_sigma_min > 0 else float("inf")
    diagnostics = GPMDiagnostics(
        iterations=len(steps),
        converged=converged,
        elapsed_seconds=time.perf_counter() - started,
        anchor_singular_values=anchor_svals,
        anchor_sigma_min=anchor_sigma_min,
        anchor_sigma_max=anchor_sigma_max,
        anchor_condition=anchor_condition,
        upload_singular_values=[torch.linalg.svdvals(b) for b in Bs],
        gram_eigenvalues=gram_eigenvalues,
        eigengap=gap,
        relative_eigengap=gap / top_scale,
        multistart_objective_gap=max(0.0, best_restart_objective - spectral_objective),
        gopp_residual=residual,
        relative_gopp_residual=residual / max(denom, torch.finfo(PROTOCOL_DTYPE).eps),
        fixed_point_residual=float(fixed.detach().cpu()),
        objective_history=objectives,
        step_history=steps,
    )
    return rs, diagnostics


def pa_i_gdp(
    x_blocks: Sequence[Tensor],
    anchor: Tensor,
    v: float,
    *,
    rotations: Optional[Sequence[Tensor]] = None,
    translations: Optional[Sequence[Tensor]] = None,
    anchor_noises: Optional[Sequence[Tensor]] = None,
    generator: Optional[torch.Generator] = None,
    translation_scale: float = 1.0,
    max_iters: int = 200,
    tol: float = 1e-10,
    n_random_starts: int = 0,
    restart_seed: int = 0,
) -> ProtocolResult:
    """PA-I-GDP with noisy anchors and block-Gram generalized Procrustes."""

    xs = _blocks(x_blocks)
    A = _as_tensor(anchor, xs[0].device)
    p, h = len(xs), xs[0].shape[1]
    if A.ndim != 2 or A.shape[1] != h:
        raise ValueError("anchor must be r-by-h.")
    os_, psis = _parameters(p, h, xs[0], rotations, translations, generator, translation_scale)
    ys = [x @ o + psi for x, o, psi in zip(xs, os_, psis)]
    clean_Bs = [A @ o + psi for o, psi in zip(os_, psis)]
    noise = _noise_blocks(clean_Bs, v, anchor_noises, generator)
    Bs = [b + e for b, e in zip(clean_Bs, noise)]
    means = [b.mean(dim=0) for b in Bs]
    centered = [b - mean for b, mean in zip(Bs, means)]
    rs, diagnostics = gpm_align(
        centered,
        raw_anchor=A,
        max_iters=max_iters,
        tol=tol,
        n_random_starts=n_random_starts,
        restart_seed=restart_seed,
    )
    aligned = [(y - mean) @ r for y, mean, r in zip(ys, means, rs)]
    return _result(
        "PA-I-GDP",
        ys,
        aligned,
        os_,
        psis,
        anchor_uploads=Bs,
        alignment_rotations=rs,
        anchor_noises=noise,
        diagnostics=diagnostics,
    )


def _stack(x: TensorOrBlocks, device: Optional[torch.device] = None) -> Tensor:
    if isinstance(x, Tensor):
        return _as_tensor(x, device)
    return torch.cat(_blocks(x, device), dim=0)


def _center(x: Tensor) -> Tuple[Tensor, Tensor]:
    mean = x.mean(dim=0)
    return mean, x - mean


def attack_c_known_input(
    colluding_x: TensorOrBlocks,
    colluding_y: TensorOrBlocks,
    target_y: Tensor,
    *,
    estimator: Literal["exact", "mp", "op"] = "exact",
) -> AttackResult:
    """Known-input attack against C-GDP; ``mp`` uses a pseudoinverse on decoding."""

    Xq = _stack(colluding_x)
    Yq = _stack(colluding_y, Xq.device)
    if Xq.shape != Yq.shape:
        raise ValueError("Colluding raw/uploaded rows must be paired and shape-compatible.")
    xbar, Xc = _center(Xq)
    ybar, Yc = _center(Yq)
    if estimator == "op":
        O = orthogonal_polar(Xc.mT @ Yc)
    else:
        O = torch.linalg.pinv(Xc) @ Yc
    psi = ybar - O.mT @ xbar
    inverse = torch.linalg.pinv(O) if estimator == "mp" else O.mT
    reduced = (_as_tensor(target_y, Xq.device) - psi) @ inverse
    return AttackResult(O, psi, reduced, f"C-{estimator}")


def invert_known_gdp(target_y: Tensor, rotation: Tensor, translation: Tensor) -> AttackResult:
    """Invert a released GDP matrix when its orthogonal secret is revealed."""

    O = _as_tensor(rotation)
    psi = _as_tensor(translation, O.device)
    return AttackResult(O, psi, (_as_tensor(target_y, O.device) - psi) @ O.mT, "known-secret")


def attack_i_colluder_negative_control(
    target_y: Tensor,
    colluder_rotation: Tensor,
    colluder_translation: Tensor,
) -> AttackResult:
    """Apply a colluder's unrelated I-GDP secret to a noncolluding target."""

    result = invert_known_gdp(target_y, colluder_rotation, colluder_translation)
    result.method = "I-colluder-negative-control"
    return result


def attack_i_distribution_moment(
    colluder_reduced: TensorOrBlocks,
    target_reference_upload: TensorOrBlocks,
    target_y: Tensor,
) -> AttackResult:
    """Unpaired I-GDP distribution-alignment sensitivity attack.

    This attack is not an algebraic inversion theorem.  It uses reduced raw
    records revealed by a colluder as a reference population and estimates a
    non-colluding target's orthogonal transform from the eigensystems of the
    two centered covariance matrices.  Eigenvector signs are resolved with
    standardized third moments, falling back to fifth moments when skewness is
    nearly zero.  It is threat-model compatible for iid participant samples
    but relies on the additional population-matching assumption and must be
    reported as a distribution-prior sensitivity.
    """

    Xq = _stack(colluder_reduced)
    Yref = _stack(target_reference_upload, Xq.device)
    Ytarget = _as_tensor(target_y, Xq.device)
    if Xq.ndim != 2 or Yref.ndim != 2 or Xq.shape[1] != Yref.shape[1]:
        raise ValueError("Reference reduced data and target uploads need the same feature dimension.")
    if Ytarget.ndim != 2 or Ytarget.shape[1] != Xq.shape[1]:
        raise ValueError("Target reconstruction rows need the same feature dimension.")
    if len(Xq) < 2 or len(Yref) < 2:
        raise ValueError("Distribution alignment requires at least two rows per reference sample.")

    xbar, Xc = _center(Xq)
    ybar, Yc = _center(Yref)
    cov_x = Xc.mT @ Xc / max(1, len(Xc) - 1)
    cov_y = Yc.mT @ Yc / max(1, len(Yc) - 1)
    eval_x, Ux = torch.linalg.eigh(cov_x)
    eval_y, Uy = torch.linalg.eigh(cov_y)
    order_x = torch.argsort(eval_x, descending=True)
    order_y = torch.argsort(eval_y, descending=True)
    Ux = Ux[:, order_x]
    Uy = Uy[:, order_y]

    px = Xc @ Ux
    py = Yc @ Uy
    eps = torch.finfo(PROTOCOL_DTYPE).eps**0.5
    px = px / px.square().mean(dim=0).sqrt().clamp_min(eps)
    py = py / py.square().mean(dim=0).sqrt().clamp_min(eps)
    skew_x, skew_y = px.pow(3).mean(dim=0), py.pow(3).mean(dim=0)
    fifth_x, fifth_y = px.pow(5).mean(dim=0), py.pow(5).mean(dim=0)
    use_fifth = (skew_x.abs() + skew_y.abs()) < 1.0e-3
    odd_x = torch.where(use_fifth, fifth_x, skew_x)
    odd_y = torch.where(use_fifth, fifth_y, skew_y)
    signs = torch.where(odd_x * odd_y >= 0, 1.0, -1.0).to(Xq)

    O = Ux @ torch.diag(signs) @ Uy.mT
    psi = ybar - xbar @ O
    reduced = (Ytarget - psi) @ O.mT
    return AttackResult(O, psi, reduced, "I-distribution-moment")


def attack_aa_known_anchor(anchor: Tensor, target_B: Tensor, target_y: Tensor) -> AttackResult:
    """Draft pseudoinverse known-anchor attack against AA-I-GDP or AA-I-GDP-AN."""

    A = _as_tensor(anchor)
    B = _as_tensor(target_B, A.device)
    a, Ac = _center(A)
    b, Bc = _center(B)
    O = torch.linalg.pinv(Ac) @ Bc
    psi = b - O.mT @ a
    reduced = (_as_tensor(target_y, A.device) - psi) @ O.mT
    return AttackResult(O, psi, reduced, "AA-known-anchor")


def attack_pa_mp(anchor: Tensor, target_B: Tensor, target_y: Tensor) -> AttackResult:
    """Unconstrained Moore--Penrose known-anchor attack against PA-I-GDP."""

    A = _as_tensor(anchor)
    B = _as_tensor(target_B, A.device)
    a, Ac = _center(A)
    b, Bc = _center(B)
    O = torch.linalg.pinv(Ac) @ Bc
    psi = b - O.mT @ a
    reduced = (_as_tensor(target_y, A.device) - psi) @ torch.linalg.pinv(O)
    return AttackResult(O, psi, reduced, "PA-MP")


def attack_pa_op(anchor: Tensor, target_B: Tensor, target_y: Tensor) -> AttackResult:
    """Orthogonal-Procrustes known-anchor attack against PA-I-GDP."""

    A = _as_tensor(anchor)
    B = _as_tensor(target_B, A.device)
    a, Ac = _center(A)
    b, Bc = _center(B)
    O = orthogonal_polar(Ac.mT @ Bc)
    psi = b - O.mT @ a
    reduced = (_as_tensor(target_y, A.device) - psi) @ O.mT
    return AttackResult(O, psi, reduced, "PA-OP")


def attack_pa_alignment_calibration(
    anchor: Tensor,
    target_B: Tensor,
    target_y: Tensor,
    target_alignment: Tensor,
    colluder_rotations: Sequence[Tensor],
    colluder_alignments: Sequence[Tensor],
) -> AttackResult:
    """Calibrate PA's global rotation from colluders' revealed local transforms."""

    if not colluder_rotations or len(colluder_rotations) != len(colluder_alignments):
        raise ValueError("Matching nonempty colluder rotation/alignment lists are required.")
    A = _as_tensor(anchor)
    B = _as_tensor(target_B, A.device)
    q_sum = sum(
        (_as_tensor(o, A.device) @ _as_tensor(r, A.device) for o, r in zip(colluder_rotations, colluder_alignments)),
        torch.zeros_like(_as_tensor(colluder_rotations[0], A.device)),
    )
    Q = orthogonal_polar(q_sum)
    Rj = _as_tensor(target_alignment, A.device)
    O = Q @ Rj.mT
    a = A.mean(dim=0)
    b = B.mean(dim=0)
    psi = b - O.mT @ a
    reduced = (_as_tensor(target_y, A.device) - psi) @ O.mT
    return AttackResult(O, psi, reduced, "PA-AM")


def _relative_error(x: Tensor, y: Tensor) -> float:
    num = torch.linalg.norm(x - y)
    den = torch.clamp(torch.linalg.norm(y), min=1.0)
    return float((num / den).detach().cpu())


def affine_orthogonal_registration(
    source: Tensor,
    target: Tensor,
) -> Tuple[Tensor, Tensor, Tensor, float]:
    """Register row embeddings by one right-orthogonal map and common shift."""

    x = _as_tensor(source)
    y = _as_tensor(target, x.device)
    if x.shape != y.shape or x.ndim != 2:
        raise ValueError("source and target must be same-shaped matrices.")
    x_mean = x.mean(dim=0)
    y_mean = y.mean(dim=0)
    rotation = orthogonal_polar((x - x_mean).mT @ (y - y_mean))
    translation = y_mean - x_mean @ rotation
    registered = x @ rotation + translation
    return registered, rotation, translation, _relative_error(registered, y)


def _assert_close(x: Tensor, y: Tensor, name: str, tol: float = 1e-9) -> float:
    error = _relative_error(x, y)
    if error > tol:
        raise AssertionError(f"{name} relative error {error:.3e} exceeds {tol:.3e}.")
    return error


def run_algebraic_self_tests(
    device: Union[str, torch.device] = "cpu",
    seed: int = 20260711,
) -> dict:
    """Run synthetic orientation, attack, GPM, and C/AA equivalence gates."""

    seed_everything(seed)
    dev = torch.device(device)
    gen = make_generator(seed, dev)
    p, raw_d, h, r = 3, 9, 4, 8
    ns = [12, 11, 10]
    raw = [torch.randn((n, raw_d), dtype=PROTOCOL_DTYPE, device=dev, generator=gen) for n in ns]
    reducer = make_shared_gaussian_projection(raw_d, h, seed=seed + 17, device=dev)
    xs = reducer.transform_blocks(raw)

    # A centred, well-conditioned anchor with A^T A proportional to I.
    g = torch.randn((r, h), dtype=PROTOCOL_DTYPE, device=dev, generator=gen)
    q, _ = torch.linalg.qr(g - g.mean(dim=0), mode="reduced")
    A = q * (r**0.5)
    anchor_svals = torch.linalg.svdvals(A - A.mean(dim=0))
    anchor_rank = int(torch.linalg.matrix_rank(A - A.mean(dim=0)).detach().cpu())
    if anchor_rank != h or float(anchor_svals[-1].detach().cpu()) <= 1e-10:
        raise AssertionError("Centered anchor is not full column rank.")
    os_, psis = sample_gdp_parameters(p, h, generator=gen, device=dev)
    s = 0

    aa = aa_i_gdp(xs, A, reference=s, rotations=os_, translations=psis)
    common = c_gdp(xs, rotation=os_[s], translation=psis[s])
    exact_error = _assert_close(aa.collaboration, common.collaboration, "C/AA equality")
    r_error = max(_relative_error(rr, oi.mT @ os_[s]) for rr, oi in zip(aa.alignment_rotations or [], os_))
    if r_error > 1e-9:
        raise AssertionError(f"AA relative-rotation error {r_error:.3e}.")

    sigma = 0.17
    shared_noise = [sigma * _randn(tuple(x.shape), x, gen) for x in xs]
    coupled_local = [g_i @ rr.mT for g_i, rr in zip(shared_noise, aa.alignment_rotations or [])]
    aa_an = aa_i_gdp_an(
        xs,
        A,
        sigma,
        reference=s,
        rotations=os_,
        translations=psis,
        noises=coupled_local,
    )
    common_an = c_gdp_an(
        xs,
        sigma,
        rotation=os_[s],
        translation=psis[s],
        noises=shared_noise,
    )
    noisy_error = _assert_close(aa_an.collaboration, common_an.collaboration, "coupled C-AN/AA-AN equality")

    c_attack = attack_c_known_input(xs[0], common.private_uploads[0], common.private_uploads[1])
    c_attack_error = _assert_close(c_attack.reduced, xs[1], "C attack")
    aa_attack = attack_aa_known_anchor(A, aa.anchor_uploads[1], aa.private_uploads[1])  # type: ignore[index]
    aa_attack_error = _assert_close(aa_attack.reduced, xs[1], "AA attack")
    aa_an_attack = attack_aa_known_anchor(A, aa_an.anchor_uploads[1], aa_an.private_uploads[1])  # type: ignore[index]
    aa_an_endpoint = xs[1] + coupled_local[1] @ os_[1].mT
    aa_an_endpoint_error = _assert_close(
        aa_an_attack.reduced,
        aa_an_endpoint,
        "AA-AN attack noise endpoint",
    )

    independent = i_gdp(xs, rotations=os_, translations=psis)
    i_negative = attack_i_colluder_negative_control(
        independent.private_uploads[1], os_[0], psis[0]
    )
    i_negative_error = _relative_error(i_negative.reduced, xs[1])
    if i_negative_error < 1e-3:
        raise AssertionError("I-GDP colluder negative control unexpectedly reconstructed the target.")

    pa = pa_i_gdp(
        xs,
        A,
        0.0,
        rotations=os_,
        translations=psis,
        generator=gen,
        max_iters=100,
        tol=1e-12,
        n_random_starts=1,
        restart_seed=seed + 1,
    )
    if pa.diagnostics is None:
        raise AssertionError("PA diagnostics are missing.")
    if not pa.diagnostics.converged:
        raise AssertionError("Noiseless PA GPM failed to converge.")
    if pa.diagnostics.relative_gopp_residual > 1e-10:
        raise AssertionError(
            f"PA relative GOPP residual {pa.diagnostics.relative_gopp_residual:.3e}."
        )
    if pa.diagnostics.fixed_point_residual > 1e-10:
        raise AssertionError(
            f"PA fixed-point residual {pa.diagnostics.fixed_point_residual:.3e}."
        )
    if pa.diagnostics.anchor_sigma_min <= 1e-10:
        raise AssertionError("PA diagnostic reports a rank-deficient centered anchor.")
    effective = [o @ rr for o, rr in zip(os_, pa.alignment_rotations or [])]
    pa_sync_error = max(_relative_error(qi, effective[0]) for qi in effective)
    if pa_sync_error > 1e-8:
        raise AssertionError(f"PA noiseless synchronization error {pa_sync_error:.3e}.")
    _, _, _, pa_c_registration_error = affine_orthogonal_registration(
        pa.collaboration, common.collaboration
    )
    if pa_c_registration_error > 1e-8:
        raise AssertionError(
            f"PA v=0/C affine-orthogonal registration error {pa_c_registration_error:.3e}."
        )
    pa_op = attack_pa_op(A, pa.anchor_uploads[1], pa.private_uploads[1])  # type: ignore[index]
    pa_op_error = _assert_close(pa_op.reduced, xs[1], "PA OP attack", tol=1e-8)
    pa_mp = attack_pa_mp(A, pa.anchor_uploads[1], pa.private_uploads[1])  # type: ignore[index]
    pa_mp_error = _assert_close(pa_mp.reduced, xs[1], "PA MP attack", tol=1e-8)
    pa_am = attack_pa_alignment_calibration(
        A,
        pa.anchor_uploads[1],  # type: ignore[index]
        pa.private_uploads[1],
        pa.alignment_rotations[1],  # type: ignore[index]
        [os_[0]],
        [pa.alignment_rotations[0]],  # type: ignore[index]
    )
    pa_am_error = _assert_close(pa_am.reduced, xs[1], "PA AM attack", tol=1e-8)

    decoded = decode_reduced(xs[1], reducer.F, reducer.F_pinv)
    floor = projection_floor(raw[1], reducer.F)
    decode_error = _assert_close(decoded, floor, "Gaussian projection decode")
    return {
        "c_aa_relative_error": exact_error,
        "c_aa_an_coupled_relative_error": noisy_error,
        "aa_rotation_relative_error": r_error,
        "c_attack_relative_error": c_attack_error,
        "aa_attack_relative_error": aa_attack_error,
        "aa_an_attack_endpoint_relative_error": aa_an_endpoint_error,
        "i_colluder_negative_control_relative_error": i_negative_error,
        "pa_sync_relative_error": pa_sync_error,
        "pa_c_affine_registration_relative_error": pa_c_registration_error,
        "pa_mp_relative_error": pa_mp_error,
        "pa_op_relative_error": pa_op_error,
        "pa_am_relative_error": pa_am_error,
        "grp_decode_relative_error": decode_error,
        "grp_matrix_sigma_min": float(reducer.singular_values[-1].detach().cpu()),
        "grp_matrix_condition": float(
            (reducer.singular_values[0] / reducer.singular_values[-1]).detach().cpu()
        ),
        "centered_anchor_rank": anchor_rank,
        "centered_anchor_sigma_min": float(anchor_svals[-1].detach().cpu()),
        "pa_gopp_residual": pa.diagnostics.gopp_residual if pa.diagnostics else float("nan"),
        "pa_relative_gopp_residual": pa.diagnostics.relative_gopp_residual if pa.diagnostics else float("nan"),
        "pa_fixed_point_residual": pa.diagnostics.fixed_point_residual if pa.diagnostics else float("nan"),
        "pa_converged": pa.diagnostics.converged if pa.diagnostics else False,
    }


def run_self_tests(
    device: Union[str, torch.device] = "cpu",
    seed: int = 20260711,
) -> dict:
    """Run all algebra gates and add a notebook-friendly success flag."""

    metrics = run_algebraic_self_tests(device=device, seed=seed)
    return {"all_passed": True, **metrics}


__all__ = [
    "PROTOCOL_DTYPE",
    "GaussianProjectionReducer",
    "GPMDiagnostics",
    "ProtocolResult",
    "AttackResult",
    "seed_everything",
    "make_generator",
    "haar_orthogonal",
    "sample_gdp_parameters",
    "make_shared_gaussian_projection",
    "reduce_blocks",
    "decode_reduced",
    "projection_floor",
    "orthogonal_polar",
    "blockwise_polar",
    "c_gdp",
    "c_gdp_an",
    "i_gdp",
    "aa_alignment",
    "aa_i_gdp",
    "aa_i_gdp_an",
    "build_block_gram",
    "gpm_align",
    "pa_i_gdp",
    "attack_c_known_input",
    "invert_known_gdp",
    "attack_i_colluder_negative_control",
    "attack_i_distribution_moment",
    "attack_aa_known_anchor",
    "attack_pa_mp",
    "attack_pa_op",
    "attack_pa_alignment_calibration",
    "affine_orthogonal_registration",
    "run_algebraic_self_tests",
    "run_self_tests",
]


In [ ]:
%%writefile celeba_public_svd_identity100_runner.py
"""CelebA Public-SVD I-GDP-PA experiment runner.

This module is embedded verbatim in the Colab notebook.  Drive is the
authoritative store.  Runtime-local files are verified mirrors only.
"""

from __future__ import annotations

import copy
import csv
import dataclasses
import hashlib
import json
import math
import os
import random
import shutil
import tempfile
import time
import zipfile
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Any, Iterable, Mapping, Optional, Sequence

import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as torch_f

import core_protocols as cp


DATASET_NAME = "CelebA"
TASK_NAME = "Smiling"
RAW_SHAPE = (3, 64, 64)
RAW_DIM = int(np.prod(RAW_SHAPE))
UTILITY_CHANCE = 0.50
LINKAGE_CHANCE = 0.10
METHODS = ("c_gdp", "c_gdp_an", "i_gdp", "pa_i_gdp")
SCHEMA_VERSION = 1
DATA_CACHE_SCHEMA_VERSION = 1
PREPROCESS_VERSION = "celeba-center-square-rgb64-v1"
REDUCTION_VERSION = "celeba-public-identity-exclusive-uncentered-svd-rank400-v1"
PUBLIC_SVD_IMAGES = 10_000
IDENTITIES_PER_PARTICIPANT = 100
SPLIT_IDENTITY_COUNTS = {"train": 70, "val": 15, "test": 15}
MAX_PARTICIPANTS = 50
ADEQUACY_GATE_VERSION = "c-gdp-exact-reduced-nrmse-v2"
C_GDP_EXACT_NRMSE_MAX = 1.0e-6


def _json_default(value: Any) -> Any:
    if dataclasses.is_dataclass(value):
        return asdict(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Tensor):
        value = value.detach().cpu()
        return value.item() if value.numel() == 1 else value.tolist()
    if isinstance(value, tuple):
        return list(value)
    raise TypeError(f"Cannot JSON-serialize {type(value)!r}")


def stable_hash(value: Any, length: int = 32) -> str:
    payload = json.dumps(
        value, sort_keys=True, separators=(",", ":"), default=_json_default
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:length]


def file_sha256(path: Path, chunk_size: int = 8 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def named_seed(base_seed: int, *parts: Any) -> int:
    digest = hashlib.sha256(
        json.dumps([int(base_seed), *parts], default=_json_default).encode("utf-8")
    ).digest()
    return int.from_bytes(digest[:8], "little") & ((1 << 63) - 1)


def _hash_order(seed: int, label: str, values: Iterable[Any]) -> list[Any]:
    return sorted(values, key=lambda value: stable_hash([seed, label, value], 64))


def seed_everything(seed: int, deterministic: bool = True) -> None:
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=True)


def resolve_device(spec: str = "auto") -> torch.device:
    if spec == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(spec)


def _atomic_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(
        "w", encoding="utf-8", delete=False, dir=path.parent, suffix=".tmp"
    ) as handle:
        json.dump(payload, handle, indent=2, sort_keys=True, default=_json_default)
        handle.write("\n")
        temporary = Path(handle.name)
    os.replace(temporary, path)


def _atomic_torch_save(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + f".{os.getpid()}.tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def _write_or_validate_json(path: Path, payload: Mapping[str, Any], label: str) -> None:
    if path.exists():
        existing = json.loads(path.read_text(encoding="utf-8"))
        if stable_hash(existing) != stable_hash(payload):
            raise RuntimeError(f"Refusing incompatible resume: {label} differs at {path}")
        return
    _atomic_json(path, payload)


@dataclass(frozen=True)
class ExperimentConfig:
    mode: str = "smoke"
    seed: int = 20260713
    data_root: Optional[str] = None
    cache_dir: str = "/content/drive/MyDrive/I-GDP-PA/CelebA_PublicSVD_identity100_cache_v1"
    output_dir: str = "/content/drive/MyDrive/I-GDP-PA/CelebA_PublicSVD_h400_identity100_results_v2"
    local_cache_dir: str = "/content/igdp_pa_celeba_public_svd"
    device: str = "auto"
    deterministic: bool = True
    n_participants: int = 10
    max_participants: int = MAX_PARTICIPANTS
    identities_per_participant: int = IDENTITIES_PER_PARTICIPANT
    public_svd_images: int = PUBLIC_SVD_IMAGES
    projection_side: int = 20
    svd_q: int = 420
    svd_niter: int = 4
    shard_size: int = 512
    batch_size: int = 256
    eval_batch_size: int = 512
    face_batch_size: int = 128
    projection_batch_size: int = 512
    num_workers: int = 2
    downstream_epochs: int = 1
    learning_rate: float = 1.0e-3
    weight_decay: float = 0.0
    smoke_protocol_images_per_identity: Optional[int] = 2
    reference_participant: int = 0
    target_participant: int = 1
    anchor_ratio: float = 4.0
    private_lambda_upper: Optional[float] = None
    anchor_lambda_upper: Optional[float] = None
    pilot_manifest_path: Optional[str] = None
    pilot_manifest_sha256: Optional[str] = None
    pilot_candidate_lambdas: tuple[float, ...] = (
        0.0,
        2.0**-8,
        2.0**-7,
        2.0**-6,
        2.0**-5,
        2.0**-4,
        2.0**-3,
        2.0**-2,
        2.0**-1,
        1.0,
        2.0,
        4.0,
        8.0,
        16.0,
    )
    pa_iterations: int = 500
    pa_tolerance: float = 1.0e-7
    linkage_candidates: int = 10
    linkage_episodes: int = 10
    bootstrap_resamples: int = 2_000
    ssim_max_images: int = 64
    face_model: str = "vggface2"
    run_attacks: bool = True
    module_sha256: Mapping[str, str] = field(default_factory=dict)

    @classmethod
    def for_mode(cls, mode: str, **overrides: Any) -> "ExperimentConfig":
        mode = mode.lower().strip()
        presets: dict[str, dict[str, Any]] = {
            "smoke": {
                "mode": "smoke",
                "downstream_epochs": 1,
                "smoke_protocol_images_per_identity": 2,
                "private_lambda_upper": 0.25,
                "anchor_lambda_upper": 0.25,
                "bootstrap_resamples": 500,
                "ssim_max_images": 32,
            },
            "pilot": {
                "mode": "pilot",
                "downstream_epochs": 5,
                "smoke_protocol_images_per_identity": None,
                "bootstrap_resamples": 2_000,
                "ssim_max_images": 128,
            },
            "full": {
                "mode": "full",
                "downstream_epochs": 15,
                "smoke_protocol_images_per_identity": None,
                "bootstrap_resamples": 10_000,
                "ssim_max_images": 256,
            },
        }
        if mode not in presets:
            raise ValueError("mode must be smoke, pilot, or full")
        return cls(**{**presets[mode], **overrides})

    def __post_init__(self) -> None:
        if self.max_participants != MAX_PARTICIPANTS:
            raise ValueError("The frozen master capacity is exactly 50 participants")
        if not 2 <= self.n_participants <= self.max_participants:
            raise ValueError("n_participants must be between 2 and 50")
        if self.identities_per_participant != IDENTITIES_PER_PARTICIPANT:
            raise ValueError("Every participant must have exactly 100 identities")
        if self.public_svd_images != PUBLIC_SVD_IMAGES:
            raise ValueError("The public SVD fit requires exactly 10,000 images")
        if self.projection_side != 20 or self.reduced_dim != 400:
            raise ValueError("The primary experiment fixes h=400 (20x20)")
        if not self.reduced_dim <= self.svd_q <= self.public_svd_images:
            raise ValueError("svd_q must be between h and 10,000")
        if self.svd_niter < 1:
            raise ValueError("svd_niter must be positive")
        if self.reference_participant != 0 or self.target_participant != 1:
            raise ValueError("The frozen threat model uses colluder 0 and protected participant 1")
        if not 0 <= self.target_participant < self.n_participants:
            raise ValueError("target_participant is out of range")
        if not 0 <= self.reference_participant < self.n_participants:
            raise ValueError("reference_participant is out of range")
        if sum(SPLIT_IDENTITY_COUNTS.values()) != self.identities_per_participant:
            raise AssertionError("Identity split counts must sum to 100")
        if self.linkage_candidates != 10:
            raise ValueError("Privacy evaluation fixes ten-way linkage")
        if self.mode == "pilot" and (self.private_lambda_upper is not None or self.anchor_lambda_upper is not None):
            raise ValueError("Pilot upper bounds must be selected, not supplied")

    @property
    def reduced_dim(self) -> int:
        return self.projection_side**2

    @property
    def anchor_rows(self) -> int:
        return int(math.ceil(self.anchor_ratio * self.reduced_dim))

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


def _coerce_config(config: Mapping[str, Any] | ExperimentConfig) -> ExperimentConfig:
    if isinstance(config, ExperimentConfig):
        return config
    return ExperimentConfig(**dict(config))


def make_config(mode: str = "smoke", **overrides: Any) -> dict[str, Any]:
    return ExperimentConfig.for_mode(mode, **overrides).to_dict()


def print_config(config: Mapping[str, Any] | ExperimentConfig) -> dict[str, Any]:
    cfg = _coerce_config(config)
    payload = cfg.to_dict()
    print(json.dumps(payload, indent=2, sort_keys=True, default=_json_default))
    return payload


def _find_one(root: Path, filename: str) -> Path:
    matches = list(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} below {root}")
    return matches[0]


def _find_optional(root: Path, filenames: Sequence[str]) -> Optional[Path]:
    for filename in filenames:
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]
    return None


def _archive_tree(source: Path, destination: Path) -> dict[str, Any]:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + f".{os.getpid()}.tmp")
    temporary.unlink(missing_ok=True)
    files = sorted(path for path in source.rglob("*") if path.is_file())
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for path in files:
            archive.write(path, path.relative_to(source).as_posix())
    os.replace(temporary, destination)
    return {
        "archive": str(destination),
        "bytes": destination.stat().st_size,
        "sha256": file_sha256(destination),
        "file_count": len(files),
    }


def _resolve_dataset_root(cfg: ExperimentConfig) -> tuple[Path, dict[str, Any]]:
    """Materialize a runtime copy from an authoritative Drive archive."""

    persistent = Path(cfg.cache_dir) / "raw_archive"
    persistent.mkdir(parents=True, exist_ok=True)
    archive_path = persistent / "celeba_kaggle_mirror.zip"
    archive_manifest_path = persistent / "celeba_kaggle_mirror_manifest.json"
    if archive_path.is_file():
        if not archive_manifest_path.is_file():
            raise RuntimeError("Persistent CelebA archive lacks its manifest")
        manifest = json.loads(archive_manifest_path.read_text(encoding="utf-8"))
        if int(manifest["bytes"]) != archive_path.stat().st_size:
            raise RuntimeError("Persistent CelebA archive size mismatch")
    else:
        if cfg.data_root and Path(cfg.data_root).exists():
            downloaded = Path(cfg.data_root)
        else:
            try:
                import kagglehub
            except ImportError as exc:
                raise RuntimeError("Install kagglehub to obtain CelebA") from exc
            downloaded = Path(kagglehub.dataset_download("jessicali9530/celeba-dataset"))
        manifest = {
            "schema_version": DATA_CACHE_SCHEMA_VERSION,
            "source": "jessicali9530/celeba-dataset",
            **_archive_tree(downloaded, archive_path),
        }
        _atomic_json(archive_manifest_path, manifest)
    local_root = Path(cfg.local_cache_dir) / "raw" / str(manifest["sha256"])[:16]
    ready = local_root / ".complete.json"
    if ready.is_file() and _find_optional(local_root, ("list_attr_celeba.csv",)):
        return local_root, manifest
    local_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(local_root)
    if _find_optional(local_root, ("list_attr_celeba.csv",)) is None:
        raise RuntimeError("Extracted CelebA archive lacks attribute metadata")
    _atomic_json(ready, {"archive_sha256": manifest["sha256"], "ready": True})
    return local_root, manifest


def _identity_metadata_file(root: Path, cfg: ExperimentConfig) -> Path:
    names = (
        "identity_CelebA.csv",
        "identity_CelebA.txt",
        "list_identity_celeba.csv",
        "list_identity_celeba.txt",
    )
    local = _find_optional(root, names)
    if local is not None:
        return local
    metadata_dir = Path(cfg.cache_dir) / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)
    destination = metadata_dir / "identity_CelebA.txt"
    expected_md5 = "32bd1bd63d3c78cd57e08160ec5ed1e2"

    def valid(path: Path) -> bool:
        return path.is_file() and hashlib.md5(path.read_bytes()).hexdigest() == expected_md5  # nosec B324

    if valid(destination):
        return destination
    try:
        import gdown
    except ImportError as exc:
        raise RuntimeError("gdown is required for official CelebA identity metadata") from exc
    temporary = destination.with_suffix(".download")
    temporary.unlink(missing_ok=True)
    gdown.download(id="1_ee_0u7vcNLOfNLegJRHmolfH5ICW-XS", output=str(temporary), quiet=False)
    if not valid(temporary):
        raise RuntimeError("Official CelebA identity metadata failed its MD5 check")
    os.replace(temporary, destination)
    return destination


def _load_metadata(root: Path, cfg: ExperimentConfig):
    import pandas as pd

    attr = pd.read_csv(_find_one(root, "list_attr_celeba.csv"))
    identity_path = _identity_metadata_file(root, cfg)
    if identity_path.suffix.lower() == ".txt":
        identity = pd.read_csv(
            identity_path, sep=r"\s+", header=None, names=["image_id", "identity"]
        )
    else:
        identity = pd.read_csv(identity_path)
    if "image_id" not in attr.columns:
        attr = attr.rename(columns={attr.columns[0]: "image_id"})
    if "image_id" not in identity.columns:
        identity = identity.rename(columns={identity.columns[0]: "image_id"})
    identity_col = next((column for column in identity.columns if column != "image_id"), None)
    if identity_col is None or "Smiling" not in attr.columns:
        raise ValueError("CelebA metadata lacks identity or Smiling columns")
    frame = attr[["image_id", "Smiling"]].merge(
        identity[["image_id", identity_col]], on="image_id", validate="one_to_one"
    )
    frame = frame.rename(columns={identity_col: "identity"})
    frame["label"] = (frame["Smiling"].astype(int) > 0).astype(np.int64)
    frame["identity"] = frame["identity"].astype(np.int64)
    frame["source_index"] = np.arange(len(frame), dtype=np.int64)
    if len(frame) != 202_599 or frame["identity"].nunique() != 10_177:
        raise RuntimeError("CelebA merge must preserve 202,599 images and 10,177 identities")
    frame.attrs["identity_metadata_sha256"] = file_sha256(identity_path)
    frame.attrs["attribute_metadata_sha256"] = file_sha256(_find_one(root, "list_attr_celeba.csv"))
    return frame


@dataclass(frozen=True)
class IdentityStat:
    identity: int
    reserved_filename: str
    protocol_filenames: tuple[str, ...]
    protocol_labels: tuple[int, ...]
    protocol_sources: tuple[int, ...]

    @property
    def image_count(self) -> int:
        return len(self.protocol_filenames)

    @property
    def positive_count(self) -> int:
        return int(sum(self.protocol_labels))


def _balanced_identity_allocation(
    stats: Sequence[IdentityStat], cfg: ExperimentConfig
) -> dict[tuple[int, str], list[IdentityStat]]:
    """Fill exact identity-count slots while balancing images and prevalence."""

    required = cfg.max_participants * cfg.identities_per_participant
    if len(stats) != required:
        raise ValueError(f"Expected {required} participant identities, received {len(stats)}")
    total_images = sum(item.image_count for item in stats)
    total_positive = sum(item.positive_count for item in stats)
    prevalence = total_positive / max(1, total_images)
    mean_images = total_images / required
    capacities = {
        (participant, split): SPLIT_IDENTITY_COUNTS[split]
        for participant in range(cfg.max_participants)
        for split in ("train", "val", "test")
    }
    buckets = {key: [] for key in capacities}
    loads = {key: 0 for key in capacities}
    positives = {key: 0 for key in capacities}
    order = sorted(
        stats,
        key=lambda item: (
            -item.image_count,
            -abs(item.positive_count / max(1, item.image_count) - prevalence),
            stable_hash([cfg.seed, "allocation", item.identity], 64),
        ),
    )
    for item in order:
        candidates = [key for key, cap in capacities.items() if len(buckets[key]) < cap]
        if not candidates:
            raise AssertionError("Identity allocation exhausted all capacities early")

        def score(key: tuple[int, str]) -> tuple[float, str]:
            _, split = key
            target_load = mean_images * SPLIT_IDENTITY_COUNTS[split]
            new_load = loads[key] + item.image_count
            new_positive = positives[key] + item.positive_count
            load_error = ((new_load - target_load) / max(1.0, target_load)) ** 2
            prevalence_error = (new_positive / max(1, new_load) - prevalence) ** 2
            fill = (len(buckets[key]) + 1) / capacities[key]
            return (
                load_error + 2.0 * prevalence_error + 1.0e-3 * fill,
                stable_hash([cfg.seed, "allocation-tie", item.identity, key], 64),
            )

        chosen = min(candidates, key=score)
        buckets[chosen].append(item)
        loads[chosen] += item.image_count
        positives[chosen] += item.positive_count
    for (participant, split), items in buckets.items():
        expected = SPLIT_IDENTITY_COUNTS[split]
        if len(items) != expected:
            raise AssertionError(
                f"Participant {participant} {split} has {len(items)} identities, expected {expected}"
            )
    return buckets


def _complete_identity_indices_legacy(
    identities: np.ndarray, size: int, seed: int
) -> np.ndarray:
    """Reproduce the complete-identity cap used by the prior CrossSVD smoke."""

    identities = np.asarray(identities, dtype=np.int64)
    rng = np.random.default_rng(seed)
    row_order = np.argsort(identities, kind="stable")
    sorted_identities = identities[row_order]
    boundaries = np.flatnonzero(np.diff(sorted_identities)) + 1
    groups = list(np.split(row_order, boundaries))
    rng.shuffle(groups)
    chosen: list[np.ndarray] = []
    used = 0
    for indices in groups:
        if used + len(indices) <= size:
            chosen.append(indices)
            used += len(indices)
    return np.concatenate(chosen).astype(np.int64) if chosen else np.empty(0, dtype=np.int64)


def _legacy_cross_svd_test_identities(frame) -> list[int]:
    """Quarantine identities whose test pixels the earlier smoke opened."""

    legacy_seed = 20260712
    identities = np.asarray(sorted(frame["identity"].unique()), dtype=np.int64)
    rng = np.random.default_rng(named_seed(legacy_seed, "identity_disjoint_split"))
    rng.shuffle(identities)
    n_train = int(round(0.70 * len(identities)))
    n_val = int(round(0.15 * len(identities)))
    test_ids = set(identities[n_train + n_val :].tolist())
    subset = frame[frame["identity"].isin(test_ids)].reset_index(drop=True)
    indices = _complete_identity_indices_legacy(
        subset["identity"].to_numpy(),
        2_500,
        named_seed(legacy_seed, "limit", "test"),
    )
    return sorted(map(int, subset.iloc[indices]["identity"].unique().tolist()))


def _build_master_manifest(frame, cfg: ExperimentConfig, archive_manifest: Mapping[str, Any]) -> dict[str, Any]:
    grouped = {int(identity): group.copy() for identity, group in frame.groupby("identity", sort=True)}
    eligible = [identity for identity, group in grouped.items() if len(group) >= 2]
    public_identities: list[int] = []
    public_count = 0
    for identity in _hash_order(cfg.seed, "public-identities", eligible):
        public_identities.append(int(identity))
        public_count += len(grouped[int(identity)])
        if public_count >= cfg.public_svd_images:
            break
    public_set = set(public_identities)
    public_rows = frame[frame["identity"].isin(public_set)]
    public_filenames = _hash_order(
        cfg.seed, "public-images", [str(value) for value in public_rows["image_id"].tolist()]
    )[: cfg.public_svd_images]
    if len(public_filenames) != PUBLIC_SVD_IMAGES:
        raise RuntimeError("Could not select exactly 10,000 public SVD images")

    legacy_test_identities = _legacy_cross_svd_test_identities(frame)
    legacy_quarantine = set(legacy_test_identities)
    remaining = [
        identity
        for identity in eligible
        if identity not in public_set and identity not in legacy_quarantine
    ]
    required = cfg.max_participants * cfg.identities_per_participant
    chosen_identities = _hash_order(cfg.seed, "participant-identities", remaining)[:required]
    if len(chosen_identities) != required:
        raise RuntimeError("Too few eligible non-public CelebA identities")
    stats: list[IdentityStat] = []
    for identity in chosen_identities:
        group = grouped[int(identity)]
        rows = {
            str(row.image_id): (int(row.label), int(row.source_index))
            for row in group.itertuples(index=False)
        }
        ordered = _hash_order(cfg.seed, f"identity-{identity}-images", rows)
        reserved = str(ordered[0])
        protocol = tuple(str(value) for value in ordered[1:])
        if not protocol:
            raise AssertionError("Every selected identity must retain a protocol image")
        stats.append(
            IdentityStat(
                identity=int(identity),
                reserved_filename=reserved,
                protocol_filenames=protocol,
                protocol_labels=tuple(rows[name][0] for name in protocol),
                protocol_sources=tuple(rows[name][1] for name in protocol),
            )
        )
    buckets = _balanced_identity_allocation(stats, cfg)
    assignments = []
    for (participant, split), items in sorted(buckets.items()):
        for item in sorted(items, key=lambda value: value.identity):
            assignments.append(
                {
                    "participant": participant,
                    "split": split,
                    "identity": item.identity,
                    "reserved_filename": item.reserved_filename,
                    "protocol_filenames": list(item.protocol_filenames),
                    "protocol_labels": list(item.protocol_labels),
                    "protocol_sources": list(item.protocol_sources),
                    "protocol_image_count": item.image_count,
                    "protocol_positive_count": item.positive_count,
                }
            )
    participant_set = {int(item["identity"]) for item in assignments}
    if public_set & participant_set:
        raise AssertionError("Public and participant identities overlap")
    if legacy_quarantine & participant_set:
        raise AssertionError("A legacy-opened test identity entered the new participant design")
    reserved = {str(item["reserved_filename"]) for item in assignments}
    protocol = {
        name for item in assignments for name in map(str, item["protocol_filenames"])
    }
    if reserved & protocol:
        raise AssertionError("Reserved references entered protocol data")
    manifest = {
        "schema_version": DATA_CACHE_SCHEMA_VERSION,
        "dataset": DATASET_NAME,
        "seed": cfg.seed,
        "archive": dict(archive_manifest),
        "metadata_hashes": {
            "identity": frame.attrs["identity_metadata_sha256"],
            "attributes": frame.attrs["attribute_metadata_sha256"],
        },
        "eligibility": "at least two available images before reserving one reference",
        "public_svd": {
            "uncentered": True,
            "image_count": len(public_filenames),
            "filenames": public_filenames,
            "identity_ids": sorted(public_set),
            "identity_count": len(public_set),
        },
        "legacy_test_quarantine": {
            "source_notebook": "I_GDP_PA_CelebA_CrossSVD_Colab.ipynb",
            "legacy_seed": 20260712,
            "legacy_test_cap": 2_500,
            "identity_ids": legacy_test_identities,
            "identity_hash": stable_hash(legacy_test_identities),
            "policy": "excluded from all new participant data",
        },
        "participant_design": {
            "master_capacity_participants": cfg.max_participants,
            "identities_per_participant": cfg.identities_per_participant,
            "split_identity_counts": SPLIT_IDENTITY_COUNTS,
            "globally_participant_disjoint": True,
            "reserved_references_excluded_from_protocol": True,
        },
        "assignments": assignments,
    }
    manifest["manifest_hash"] = stable_hash(manifest)
    return manifest


def _load_or_create_master_manifest(
    frame, cfg: ExperimentConfig, archive_manifest: Mapping[str, Any]
) -> dict[str, Any]:
    path = Path(cfg.cache_dir) / "master" / (
        f"master_pmax{cfg.max_participants}_i100_seed{cfg.seed}.json"
    )
    expected = _build_master_manifest(frame, cfg, archive_manifest)
    _write_or_validate_json(path, expected, "CelebA identity/public allocation")
    loaded = json.loads(path.read_text(encoding="utf-8"))
    if loaded.get("manifest_hash") != stable_hash({k: v for k, v in loaded.items() if k != "manifest_hash"}):
        raise RuntimeError("Master manifest self-hash mismatch")
    return loaded


class _FilenameImageDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir: Path, filenames: Sequence[str]) -> None:
        self.image_dir = image_dir
        self.filenames = tuple(map(str, filenames))

    def __len__(self) -> int:
        return len(self.filenames)

    def __getitem__(self, index: int) -> tuple[Tensor, int]:
        from PIL import Image

        with Image.open(self.image_dir / self.filenames[index]) as image:
            image = image.convert("RGB")
            width, height = image.size
            side = min(width, height)
            left = (width - side) // 2
            top = (height - side) // 2
            image = image.crop((left, top, left + side, top + side)).resize(
                (64, 64), Image.Resampling.BILINEAR
            )
            array = np.asarray(image, dtype=np.uint8).copy()
        return torch.from_numpy(array).permute(2, 0, 1).contiguous(), index


class RGB64ShardStore:
    """Drive-backed selected-image cache with verified runtime-local mirrors."""

    def __init__(
        self,
        cfg: ExperimentConfig,
        image_dir: Path,
        scope: str,
        filenames: Sequence[str],
    ) -> None:
        self.cfg = cfg
        self.image_dir = image_dir
        self.scope = scope
        self.filenames = tuple(map(str, filenames))
        self.fingerprint = stable_hash(
            {"preprocess": PREPROCESS_VERSION, "scope": scope, "filenames": self.filenames}
        )
        self.drive_dir = Path(cfg.cache_dir) / "rgb64_shards" / self.fingerprint
        self.local_dir = Path(cfg.local_cache_dir) / "rgb64_shards" / self.fingerprint
        self.manifest_path = self.drive_dir / "manifest.json"
        self._index: Optional[dict[str, tuple[Path, int]]] = None

    def materialize(self) -> dict[str, Any]:
        self.drive_dir.mkdir(parents=True, exist_ok=True)
        records = []
        loader = None
        dataset = _FilenameImageDataset(self.image_dir, self.filenames)
        for shard_index, start in enumerate(range(0, len(self.filenames), self.cfg.shard_size)):
            stop = min(len(self.filenames), start + self.cfg.shard_size)
            names = self.filenames[start:stop]
            path = self.drive_dir / f"shard_{shard_index:05d}.pt"
            if not path.is_file():
                subset = torch.utils.data.Subset(dataset, range(start, stop))
                loader = torch.utils.data.DataLoader(
                    subset,
                    batch_size=min(self.cfg.shard_size, 256),
                    shuffle=False,
                    num_workers=self.cfg.num_workers,
                )
                batches = [images for images, _ in loader]
                images = torch.cat(batches, dim=0)
                _atomic_torch_save(path, {"filenames": names, "images": images})
            payload = torch.load(path, map_location="cpu", weights_only=False)
            if tuple(payload["filenames"]) != tuple(names):
                raise RuntimeError(f"Shard filename mismatch at {path}")
            images = payload["images"]
            if images.dtype != torch.uint8 or tuple(images.shape) != (len(names), *RAW_SHAPE):
                raise RuntimeError(f"Invalid RGB64 shard tensor at {path}")
            records.append(
                {
                    "file": path.name,
                    "count": len(names),
                    "bytes": path.stat().st_size,
                    "sha256": file_sha256(path),
                    "filename_hash": stable_hash(names),
                }
            )
        manifest = {
            "schema_version": DATA_CACHE_SCHEMA_VERSION,
            "scope": self.scope,
            "fingerprint": self.fingerprint,
            "preprocess": PREPROCESS_VERSION,
            "count": len(self.filenames),
            "filenames": list(self.filenames),
            "shards": records,
            "drive_authoritative": True,
        }
        _write_or_validate_json(self.manifest_path, manifest, f"RGB64 {self.scope} shards")
        self._mirror_to_runtime(manifest)
        return manifest

    def _mirror_to_runtime(self, manifest: Mapping[str, Any]) -> None:
        self.local_dir.mkdir(parents=True, exist_ok=True)
        for record in manifest["shards"]:
            source = self.drive_dir / str(record["file"])
            target = self.local_dir / str(record["file"])
            if not target.is_file() or target.stat().st_size != int(record["bytes"]):
                temporary = target.with_suffix(target.suffix + ".tmp")
                shutil.copy2(source, temporary)
                os.replace(temporary, target)
            if file_sha256(target) != str(record["sha256"]):
                target.unlink(missing_ok=True)
                raise RuntimeError(f"Runtime shard mirror failed SHA-256 validation: {target}")
        _atomic_json(
            self.local_dir / "manifest.json",
            {"drive_manifest_hash": stable_hash(manifest), "verified": True},
        )

    def _build_index(self) -> dict[str, tuple[Path, int]]:
        manifest = json.loads(self.manifest_path.read_text(encoding="utf-8"))
        index: dict[str, tuple[Path, int]] = {}
        offset = 0
        for record in manifest["shards"]:
            path = self.local_dir / str(record["file"])
            payload = torch.load(path, map_location="cpu", weights_only=False)
            for local_position, name in enumerate(payload["filenames"]):
                index[str(name)] = (path, local_position)
            offset += int(record["count"])
        if offset != len(self.filenames) or set(index) != set(self.filenames):
            raise RuntimeError("RGB64 shard index is incomplete")
        return index

    def load(self, filenames: Sequence[str]) -> Tensor:
        if self._index is None:
            self._index = self._build_index()
        grouped: dict[Path, list[tuple[int, int]]] = {}
        for output_position, name in enumerate(map(str, filenames)):
            path, local_position = self._index[name]
            grouped.setdefault(path, []).append((output_position, local_position))
        output = torch.empty((len(filenames), *RAW_SHAPE), dtype=torch.uint8)
        for path, positions in grouped.items():
            payload = torch.load(path, map_location="cpu", weights_only=False)
            for output_position, local_position in positions:
                output[output_position] = payload["images"][local_position]
        return output


@dataclass
class PublicSVDReducer:
    F: Tensor
    singular_values: Tensor
    basis_manifest: dict[str, Any]

    @property
    def F_pinv(self) -> Tensor:
        return self.F.mT

    def manifest(self) -> dict[str, Any]:
        return dict(self.basis_manifest)


def _make_public_reducer(
    cfg: ExperimentConfig,
    master: Mapping[str, Any],
    public_store: RGB64ShardStore,
) -> PublicSVDReducer:
    basis_dir = Path(cfg.cache_dir) / "public_svd"
    basis_spec = stable_hash(
        {
            "reduction": REDUCTION_VERSION,
            "master": master["manifest_hash"],
            "q": cfg.svd_q,
            "niter": cfg.svd_niter,
            "h": cfg.reduced_dim,
        }
    )
    basis_path = basis_dir / f"basis_{basis_spec}.pt"
    manifest_path = basis_dir / f"basis_{basis_spec}.json"
    if basis_path.is_file() and manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        payload = torch.load(basis_path, map_location="cpu", weights_only=False)
        F = payload["F"].float()
        singular = payload["singular_values"].double()
        if file_sha256(basis_path) != manifest["basis_file_sha256"]:
            raise RuntimeError("Public SVD basis file hash mismatch")
        if tuple(F.shape) != (RAW_DIM, cfg.reduced_dim):
            raise RuntimeError("Cached Public SVD basis has the wrong shape")
        residual = float(torch.linalg.matrix_norm(F.double().mT @ F.double() - torch.eye(cfg.reduced_dim)))
        if residual > 1.0e-3:
            raise RuntimeError("Cached Public SVD basis is not orthonormal")
        return PublicSVDReducer(F, singular, manifest)

    public_filenames = list(master["public_svd"]["filenames"])
    if len(public_filenames) != PUBLIC_SVD_IMAGES:
        raise RuntimeError("Public basis provenance does not contain exactly 10,000 images")
    images = public_store.load(public_filenames)
    matrix = images.float().flatten(1).div_(255.0)
    device = resolve_device(cfg.device)
    matrix_device = matrix.to(device)
    seed_everything(named_seed(cfg.seed, "public-svd"), cfg.deterministic)
    _, singular, candidate = torch.pca_lowrank(
        matrix_device,
        q=cfg.svd_q,
        center=False,
        niter=cfg.svd_niter,
    )
    # QR only selects a numerically stable orthonormal basis for the same span.
    basis_q, _ = torch.linalg.qr(candidate[:, : cfg.reduced_dim].double().cpu(), mode="reduced")
    F = basis_q.float().contiguous()
    retained = singular[: cfg.reduced_dim].double().cpu()
    residual = float(torch.linalg.matrix_norm(F.double().mT @ F.double() - torch.eye(cfg.reduced_dim)))
    if residual > 1.0e-3:
        raise RuntimeError(f"Public SVD orthogonality residual is {residual}")
    payload = {"F": F, "singular_values": retained}
    _atomic_torch_save(basis_path, payload)
    manifest = {
        "schema_version": DATA_CACHE_SCHEMA_VERSION,
        "reduction": REDUCTION_VERSION,
        "uncentered": True,
        "center_argument": False,
        "input_dim": RAW_DIM,
        "output_dim": cfg.reduced_dim,
        "fit_image_count": len(public_filenames),
        "public_identity_ids": list(master["public_svd"]["identity_ids"]),
        "public_filename_hash": stable_hash(public_filenames),
        "master_manifest_hash": master["manifest_hash"],
        "q": cfg.svd_q,
        "niter": cfg.svd_niter,
        "orthogonality_residual": residual,
        "retained_singular_values": retained.tolist(),
        "basis_file": str(basis_path),
        "basis_file_sha256": file_sha256(basis_path),
        "basis_matrix_sha256": hashlib.sha256(F.numpy().tobytes()).hexdigest(),
    }
    _atomic_json(manifest_path, manifest)
    del matrix, matrix_device, images
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return PublicSVDReducer(F, retained, manifest)


@dataclass
class SplitData:
    raw_uint8: Tensor
    labels: Tensor
    identities: Tensor
    source_indices: Tensor
    participant_ids: Tensor
    reduced: Tensor
    filenames: tuple[str, ...]

    def __len__(self) -> int:
        return int(self.labels.numel())


@dataclass
class ReferenceData:
    raw_uint8: Tensor
    identities: Tensor
    filenames: tuple[str, ...]

    def by_identity(self) -> dict[int, int]:
        return {int(identity): index for index, identity in enumerate(self.identities.tolist())}


@dataclass
class CelebADataBundle:
    train: SplitData
    val: SplitData
    test: SplitData
    references: ReferenceData
    reducer: PublicSVDReducer
    manifest: dict[str, Any]

    def split(self, name: str) -> SplitData:
        if name not in {"train", "val", "test"}:
            raise KeyError(name)
        return getattr(self, name)

    def participant_blocks(
        self, split: str, device: torch.device | str = "cpu"
    ) -> tuple[list[Tensor], list[Tensor], list[Tensor]]:
        data = self.split(split)
        xs: list[Tensor] = []
        ys: list[Tensor] = []
        indices: list[Tensor] = []
        n_participants = int(self.manifest["participant_design"]["active_participants"])
        for participant in range(n_participants):
            idx = torch.nonzero(data.participant_ids == participant, as_tuple=False).flatten()
            indices.append(idx)
            xs.append(data.reduced[idx].to(device=device, dtype=torch.float64))
            ys.append(data.labels[idx])
        return xs, ys, indices


def _assignment_frame(master: Mapping[str, Any], split: str, cfg: ExperimentConfig):
    import pandas as pd

    rows = []
    cap = cfg.smoke_protocol_images_per_identity if cfg.mode == "smoke" else None
    for assignment in master["assignments"]:
        if assignment["split"] != split or int(assignment["participant"]) >= cfg.n_participants:
            continue
        names = list(assignment["protocol_filenames"])
        labels = list(assignment["protocol_labels"])
        sources = list(assignment["protocol_sources"])
        if cap is not None:
            names, labels, sources = names[:cap], labels[:cap], sources[:cap]
        for name, label, source in zip(names, labels, sources):
            rows.append(
                {
                    "image_id": str(name),
                    "label": int(label),
                    "identity": int(assignment["identity"]),
                    "source_index": int(source),
                    "participant": int(assignment["participant"]),
                }
            )
    rows.sort(
        key=lambda row: (
            row["participant"],
            stable_hash([cfg.seed, "protocol-row", split, row["image_id"]], 64),
        )
    )
    return pd.DataFrame(rows)


def _scope_filenames(master: Mapping[str, Any], split: str, cfg: ExperimentConfig) -> list[str]:
    frame = _assignment_frame(master, split, cfg)
    return frame["image_id"].astype(str).tolist()


def _reference_filenames(
    master: Mapping[str, Any], allowed_splits: Sequence[str], cfg: ExperimentConfig
) -> tuple[list[str], list[int]]:
    selected = [
        assignment
        for assignment in master["assignments"]
        if assignment["split"] in set(allowed_splits)
        and int(assignment["participant"]) < cfg.n_participants
    ]
    selected.sort(key=lambda item: int(item["identity"]))
    return (
        [str(item["reserved_filename"]) for item in selected],
        [int(item["identity"]) for item in selected],
    )


def _project_split(
    frame,
    store: RGB64ShardStore,
    reducer: PublicSVDReducer,
    cfg: ExperimentConfig,
    split: str,
) -> SplitData:
    filenames = tuple(frame["image_id"].astype(str).tolist())
    cap_tag = (
        f"cap{cfg.smoke_protocol_images_per_identity}"
        if cfg.mode == "smoke"
        else "all"
    )
    fingerprint = stable_hash(
        {
            "cap": cap_tag,
            "split": split,
            "filenames": filenames,
            "projection": reducer.manifest()["basis_matrix_sha256"],
        }
    )
    path = Path(cfg.cache_dir) / "projected_splits" / f"{split}_{fingerprint}.pt"
    if path.is_file():
        payload = torch.load(path, map_location="cpu", weights_only=False)
        if tuple(payload["filenames"]) != filenames:
            raise RuntimeError(f"Projected {split} cache provenance mismatch")
        return SplitData(**payload)
    raw = store.load(filenames)
    reduced = torch.empty((len(frame), cfg.reduced_dim), dtype=torch.float32)
    device = resolve_device(cfg.device)
    F = reducer.F.to(device)
    for start in range(0, len(raw), cfg.projection_batch_size):
        stop = min(len(raw), start + cfg.projection_batch_size)
        flattened = raw[start:stop].to(device=device, dtype=torch.float32).flatten(1) / 255.0
        reduced[start:stop] = (flattened @ F).cpu()
    reducer.F = reducer.F.cpu()
    result = SplitData(
        raw_uint8=raw,
        labels=torch.as_tensor(frame["label"].to_numpy(), dtype=torch.long),
        identities=torch.as_tensor(frame["identity"].to_numpy(), dtype=torch.long),
        source_indices=torch.as_tensor(frame["source_index"].to_numpy(), dtype=torch.long),
        participant_ids=torch.as_tensor(frame["participant"].to_numpy(), dtype=torch.long),
        reduced=reduced,
        filenames=filenames,
    )
    _atomic_torch_save(path, asdict(result))
    return result


def _empty_split(cfg: ExperimentConfig) -> SplitData:
    return SplitData(
        raw_uint8=torch.empty((0, *RAW_SHAPE), dtype=torch.uint8),
        labels=torch.empty(0, dtype=torch.long),
        identities=torch.empty(0, dtype=torch.long),
        source_indices=torch.empty(0, dtype=torch.long),
        participant_ids=torch.empty(0, dtype=torch.long),
        reduced=torch.empty((0, cfg.reduced_dim), dtype=torch.float32),
        filenames=(),
    )


def _balance_diagnostics(master: Mapping[str, Any], cfg: ExperimentConfig) -> dict[str, Any]:
    rows = []
    for assignment in master["assignments"]:
        if int(assignment["participant"]) >= cfg.n_participants:
            continue
        rows.append(
            {
                "participant": int(assignment["participant"]),
                "split": str(assignment["split"]),
                "images": int(assignment["protocol_image_count"]),
                "positive": int(assignment["protocol_positive_count"]),
            }
        )
    diagnostics: dict[str, Any] = {}
    for split in ("train", "val", "test"):
        values = []
        for participant in sorted({row["participant"] for row in rows}):
            selected = [row for row in rows if row["participant"] == participant and row["split"] == split]
            images = sum(row["images"] for row in selected)
            positive = sum(row["positive"] for row in selected)
            values.append(
                {
                    "participant": participant,
                    "identity_count": len(selected),
                    "image_count": images,
                    "smiling_prevalence": positive / max(1, images),
                }
            )
        counts = [item["image_count"] for item in values]
        prevalences = [item["smiling_prevalence"] for item in values]
        diagnostics[split] = {
            "participants": values,
            "image_count_range": [min(counts), max(counts)],
            "prevalence_range": [min(prevalences), max(prevalences)],
            "image_count_relative_range": (max(counts) - min(counts)) / max(1.0, float(np.mean(counts))),
            "prevalence_absolute_range": max(prevalences) - min(prevalences),
        }
    return diagnostics


def _active_design_checks(master: Mapping[str, Any], cfg: ExperimentConfig) -> dict[str, bool]:
    active = [
        item
        for item in master["assignments"]
        if int(item["participant"]) < cfg.n_participants
    ]
    identities = [int(item["identity"]) for item in active]
    public = set(map(int, master["public_svd"]["identity_ids"]))
    legacy = set(map(int, master["legacy_test_quarantine"]["identity_ids"]))
    exact_split_counts = all(
        sum(
            int(item["participant"]) == participant and item["split"] == split
            for item in active
        )
        == SPLIT_IDENTITY_COUNTS[split]
        for participant in range(cfg.n_participants)
        for split in SPLIT_IDENTITY_COUNTS
    )
    reserved = {str(item["reserved_filename"]) for item in active}
    protocol = {
        str(name) for item in active for name in item["protocol_filenames"]
    }
    return {
        "exact_100_identities_per_participant": len(identities)
        == cfg.n_participants * IDENTITIES_PER_PARTICIPANT,
        "exact_70_15_15_identity_splits": exact_split_counts,
        "globally_participant_disjoint": len(identities) == len(set(identities)),
        "public_identity_exclusive": not bool(public & set(identities)),
        "legacy_test_quarantine_exclusive": not bool(legacy & set(identities)),
        "reserved_images_excluded_from_protocol": not bool(reserved & protocol),
    }


def prepare_celeba_data(
    config: Mapping[str, Any] | ExperimentConfig,
    *,
    include_test: Optional[bool] = None,
) -> CelebADataBundle:
    cfg = _coerce_config(config)
    if include_test is None:
        include_test = cfg.mode == "full"
    if cfg.mode in {"smoke", "pilot"} and include_test:
        raise ValueError("Smoke/pilot must not open test pixels")
    seed_everything(cfg.seed, cfg.deterministic)
    root, archive_manifest = _resolve_dataset_root(cfg)
    metadata = _load_metadata(root, cfg)
    master = _load_or_create_master_manifest(metadata, cfg, archive_manifest)
    candidates = [path for path in root.rglob("img_align_celeba") if path.is_dir()]
    image_dir = next(
        (path for path in candidates if (path / str(metadata.iloc[0]["image_id"])).is_file()),
        None,
    )
    if image_dir is None:
        raise FileNotFoundError("Could not locate aligned CelebA JPEG directory")

    public_names = list(master["public_svd"]["filenames"])
    public_store = RGB64ShardStore(cfg, image_dir, "public_svd", public_names)
    public_shards = public_store.materialize()
    reducer = _make_public_reducer(cfg, master, public_store)

    split_data: dict[str, SplitData] = {}
    shard_manifests: dict[str, Any] = {"public_svd": public_shards}
    for split in ("train", "val", "test"):
        if split == "test" and not include_test:
            split_data[split] = _empty_split(cfg)
            continue
        frame = _assignment_frame(master, split, cfg)
        cap_tag = (
            f"cap{cfg.smoke_protocol_images_per_identity}"
            if cfg.mode == "smoke"
            else "all"
        )
        store = RGB64ShardStore(
            cfg, image_dir, f"protocol_{cap_tag}_{split}", frame["image_id"].tolist()
        )
        shard_manifests[f"protocol_{split}"] = store.materialize()
        split_data[split] = _project_split(frame, store, reducer, cfg, split)

    allowed_reference_splits = ("train", "val", "test") if include_test else ("train", "val")
    reference_names, reference_identities = _reference_filenames(master, allowed_reference_splits, cfg)
    reference_store = RGB64ShardStore(
        cfg, image_dir, f"reserved_{'_'.join(allowed_reference_splits)}", reference_names
    )
    shard_manifests["reserved_references"] = reference_store.materialize()
    references = ReferenceData(
        raw_uint8=reference_store.load(reference_names),
        identities=torch.as_tensor(reference_identities, dtype=torch.long),
        filenames=tuple(reference_names),
    )
    manifest = {
        "schema_version": SCHEMA_VERSION,
        "dataset": DATASET_NAME,
        "task": TASK_NAME,
        "mode": cfg.mode,
        "test_pixels_opened": bool(include_test),
        "master_manifest_hash": master["manifest_hash"],
        "active_participant_identity_ids": sorted(
            int(item["identity"])
            for item in master["assignments"]
            if int(item["participant"]) < cfg.n_participants
        ),
        "legacy_test_quarantine": master["legacy_test_quarantine"],
        "participant_design": {
            **master["participant_design"],
            "active_participants": cfg.n_participants,
        },
        "sizes": {name: len(value) for name, value in split_data.items()},
        "positive_counts": {
            name: int(value.labels.sum()) for name, value in split_data.items()
        },
        "identity_counts": {
            name: int(value.identities.unique().numel()) for name, value in split_data.items()
        },
        "balance": _balance_diagnostics(master, cfg),
        "active_design_checks": _active_design_checks(master, cfg),
        "projection": reducer.manifest(),
        "drive_shards": {
            name: stable_hash(value) for name, value in shard_manifests.items()
        },
        "reserved_reference_count": len(references.identities),
        "reserved_reference_identity_hash": stable_hash(sorted(references.identities.tolist())),
        "runtime_local_files_are_mirrors": True,
    }
    return CelebADataBundle(
        train=split_data["train"],
        val=split_data["val"],
        test=split_data["test"],
        references=references,
        reducer=reducer,
        manifest=manifest,
    )


class BinaryReferenceCNN(nn.Module):
    """Reference two-convolution architecture with one Smiling logit."""

    def __init__(self, side: int = 20) -> None:
        super().__init__()
        self.side = side
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.pool = nn.MaxPool2d(2)
        with torch.no_grad():
            flat = int(self._convolve(torch.zeros(1, 1, side, side)).numel())
        self.fc1 = nn.Linear(flat, 50)
        self.fc2 = nn.Linear(50, 1)

    def _convolve(self, x: Tensor) -> Tensor:
        x = self.pool(torch_f.relu(self.conv1(x)))
        return self.pool(torch_f.relu(self.conv2(x)))

    def forward_features(self, x: Tensor) -> Tensor:
        return torch_f.relu(self.fc1(torch.flatten(self._convolve(x), 1)))

    def forward(self, x: Tensor) -> Tensor:
        return self.fc2(self.forward_features(x)).squeeze(1)


def _binary_metrics(labels: Tensor, probabilities: Tensor) -> dict[str, Any]:
    y = labels.detach().long().flatten().cpu()
    probability = probabilities.detach().float().flatten().cpu()
    pred = (probability >= 0.5).long()
    tn = int(((y == 0) & (pred == 0)).sum())
    fp = int(((y == 0) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum())
    tp = int(((y == 1) & (pred == 1)).sum())
    recalls = [tn / max(1, tn + fp), tp / max(1, tp + fn)]
    f1s = []
    for cls in (0, 1):
        cls_tp = int(((y == cls) & (pred == cls)).sum())
        cls_fp = int(((y != cls) & (pred == cls)).sum())
        cls_fn = int(((y == cls) & (pred != cls)).sum())
        precision = cls_tp / max(1, cls_tp + cls_fp)
        recall = cls_tp / max(1, cls_tp + cls_fn)
        f1s.append(2 * precision * recall / max(1.0e-15, precision + recall))
    from sklearn.metrics import average_precision_score, roc_auc_score

    try:
        auroc = float(roc_auc_score(y.numpy(), probability.numpy()))
        auprc = float(average_precision_score(y.numpy(), probability.numpy()))
    except ValueError:
        auroc = auprc = math.nan
    return {
        "accuracy": float((pred == y).float().mean()),
        "balanced_accuracy": float(np.mean(recalls)),
        "macro_f1": float(np.mean(f1s)),
        "auroc": auroc,
        "auprc": auprc,
        "confusion_matrix": [[tn, fp], [fn, tp]],
        "positive_prevalence": float(y.float().mean()) if len(y) else math.nan,
        "n": int(len(y)),
    }


@torch.no_grad()
def _evaluate_binary(model: nn.Module, x: Tensor, y: Tensor, cfg: ExperimentConfig) -> dict[str, Any]:
    device = resolve_device(cfg.device)
    model.eval().to(device)
    probabilities: list[Tensor] = []
    loss_sum = 0.0
    for start in range(0, len(y), cfg.eval_batch_size):
        stop = min(len(y), start + cfg.eval_batch_size)
        batch_x = x[start:stop].to(device=device, dtype=torch.float32)
        batch_y = y[start:stop].to(device=device, dtype=torch.float32)
        logits = model(batch_x)
        loss_sum += float(torch_f.binary_cross_entropy_with_logits(logits, batch_y, reduction="sum"))
        probabilities.append(torch.sigmoid(logits).cpu())
    metrics = _binary_metrics(y, torch.cat(probabilities) if probabilities else torch.empty(0))
    metrics["loss"] = loss_sum / max(1, len(y))
    return metrics


def _train_binary(
    train_x: Tensor,
    train_y: Tensor,
    val_x: Tensor,
    val_y: Tensor,
    cfg: ExperimentConfig,
    seed: int,
    checkpoint_path: Path,
) -> tuple[BinaryReferenceCNN, dict[str, Any]]:
    """Resume exactly at epoch boundaries from a Drive-backed checkpoint."""

    seed_everything(seed, cfg.deterministic)
    device = resolve_device(cfg.device)
    model = BinaryReferenceCNN(cfg.projection_side).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay
    )
    generator = torch.Generator(device="cpu").manual_seed(named_seed(seed, "minibatches"))
    best_state = copy.deepcopy(model.state_dict())
    best_score = -math.inf
    best_epoch = 0
    history: list[dict[str, Any]] = []
    completed_epoch = 0
    if checkpoint_path.is_file():
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        best_state = checkpoint["best_state"]
        best_score = float(checkpoint["best_score"])
        best_epoch = int(checkpoint["best_epoch"])
        history = list(checkpoint["history"])
        completed_epoch = int(checkpoint["completed_epoch"])
        generator.set_state(checkpoint["minibatch_generator_state"].cpu())
    x = train_x.to(device=device, dtype=torch.float32)
    y = train_y.to(device=device, dtype=torch.float32)
    for epoch in range(completed_epoch + 1, cfg.downstream_epochs + 1):
        model.train()
        permutation = torch.randperm(len(y), generator=generator)
        running = 0.0
        for start in range(0, len(y), cfg.batch_size):
            idx = permutation[start : start + cfg.batch_size].to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x[idx])
            loss = torch_f.binary_cross_entropy_with_logits(logits, y[idx])
            loss.backward()
            optimizer.step()
            running += float(loss) * len(idx)
        metrics = _evaluate_binary(model, val_x, val_y, cfg)
        history.append(
            {
                "epoch": epoch,
                "train_loss": running / max(1, len(y)),
                **{f"val_{key}": value for key, value in metrics.items()},
            }
        )
        if metrics["balanced_accuracy"] > best_score:
            best_score = float(metrics["balanced_accuracy"])
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
        _atomic_torch_save(
            checkpoint_path,
            {
                "schema_version": SCHEMA_VERSION,
                "completed_epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_state": best_state,
                "best_score": best_score,
                "best_epoch": best_epoch,
                "history": history,
                "minibatch_generator_state": generator.get_state(),
                "complete": epoch == cfg.downstream_epochs,
            },
        )
    model.load_state_dict(best_state)
    return model, {
        "best_epoch": best_epoch,
        "best_val_balanced_accuracy": best_score,
        "history": history,
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_sha256": file_sha256(checkpoint_path),
    }


def _load_face_embedder(cfg: ExperimentConfig) -> tuple[nn.Module, dict[str, Any]]:
    torch_home = Path(cfg.cache_dir) / "torch_home"
    torch_home.mkdir(parents=True, exist_ok=True)
    os.environ["TORCH_HOME"] = str(torch_home)
    from facenet_pytorch import InceptionResnetV1

    model = InceptionResnetV1(pretrained=cfg.face_model).eval().to(resolve_device(cfg.device))
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    digest = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        digest.update(name.encode("utf-8"))
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return model, {
        "architecture": "InceptionResnetV1",
        "pretrained": cfg.face_model,
        "embedding_dim": 512,
        "preprocessing": "RGB [0,1], bilinear 160x160, (255*x-127.5)/128",
        "state_sha256": digest.hexdigest(),
        "frozen": True,
        "drive_backed_torch_home": str(torch_home),
    }


@torch.no_grad()
def _face_embeddings(model: nn.Module, images: Tensor, cfg: ExperimentConfig) -> Tensor:
    device = resolve_device(cfg.device)
    outputs = []
    divide_by_255 = images.dtype == torch.uint8
    for start in range(0, len(images), cfg.face_batch_size):
        batch = images[start : start + cfg.face_batch_size].to(device=device, dtype=torch.float32)
        if divide_by_255:
            batch = batch / 255.0
        batch = torch_f.interpolate(batch, size=(160, 160), mode="bilinear", align_corners=False)
        batch = (batch * 255.0 - 127.5) / 128.0
        outputs.append(model(batch).cpu())
    return torch_f.normalize(torch.cat(outputs), dim=1) if outputs else torch.empty((0, 512))


CandidateToken = tuple[str, int]


@dataclass(frozen=True)
class FaceLinkageManifest:
    query_splits: tuple[str, ...]
    query_split_indices: tuple[int, ...]
    query_target_positions: tuple[int, ...]
    query_identities: tuple[int, ...]
    source_galleries: tuple[tuple[tuple[CandidateToken, ...], ...], ...]
    cross_galleries: tuple[tuple[tuple[CandidateToken, ...], ...], ...]
    seed: int
    n_candidates: int = 10
    episodes: int = 10

    @property
    def manifest_hash(self) -> str:
        return stable_hash(asdict(self))

    def to_dict(self) -> dict[str, Any]:
        return {**asdict(self), "manifest_hash": self.manifest_hash}


def make_face_linkage_manifest(
    data: CelebADataBundle,
    blocks: Mapping[str, tuple[list[Tensor], list[Tensor], list[Tensor]]],
    cfg: ExperimentConfig,
    include_test: bool,
) -> FaceLinkageManifest:
    """One protected query per identity; the bootstrap unit is identity."""

    query_splits = ("train", "val", "test") if include_test else ("train", "val")
    candidates = []
    for split_name in query_splits:
        split = data.split(split_name)
        target_global = blocks[split_name][2][cfg.target_participant]
        by_identity: dict[int, list[tuple[int, int]]] = {}
        for target_position, global_index in enumerate(target_global.tolist()):
            identity = int(split.identities[global_index])
            by_identity.setdefault(identity, []).append((target_position, global_index))
        for identity, positions in by_identity.items():
            ordered = _hash_order(
                cfg.seed,
                f"linkage-query-{split_name}-{identity}",
                positions,
            )
            target_position, global_index = ordered[0]
            candidates.append((split_name, int(global_index), int(target_position), identity))
    candidates.sort(key=lambda value: (value[0], value[3]))
    expected = 100 if include_test else 85
    if len(candidates) != expected:
        raise RuntimeError(
            f"Protected participant {cfg.target_participant} supplied {len(candidates)} linkage "
            f"identities; expected {expected}"
        )
    reference_identities = sorted(map(int, data.references.identities.tolist()))
    reference_set = set(reference_identities)
    if any(identity not in reference_set for *_, identity in candidates):
        raise RuntimeError("A protected linkage identity lacks its reserved reference")
    rng = np.random.default_rng(named_seed(cfg.seed, "reserved-reference-linkage", include_test))
    source_all = []
    cross_all = []
    for split_name, global_index, _, identity in candidates:
        impostor_pool = np.asarray(
            [value for value in reference_identities if value != identity], dtype=np.int64
        )
        source_episodes = []
        cross_episodes = []
        for _ in range(cfg.linkage_episodes):
            impostor_ids = rng.choice(
                impostor_pool, cfg.linkage_candidates - 1, replace=False
            ).tolist()
            source_gallery: list[CandidateToken] = [
                (f"source_{split_name}", int(global_index)),
                *(("reference", int(value)) for value in impostor_ids),
            ]
            cross_gallery: list[CandidateToken] = [
                ("reference", int(identity)),
                *(("reference", int(value)) for value in impostor_ids),
            ]
            rng.shuffle(source_gallery)
            rng.shuffle(cross_gallery)
            source_episodes.append(tuple(source_gallery))
            cross_episodes.append(tuple(cross_gallery))
        source_all.append(tuple(source_episodes))
        cross_all.append(tuple(cross_episodes))
    return FaceLinkageManifest(
        query_splits=tuple(value[0] for value in candidates),
        query_split_indices=tuple(value[1] for value in candidates),
        query_target_positions=tuple(value[2] for value in candidates),
        query_identities=tuple(value[3] for value in candidates),
        source_galleries=tuple(source_all),
        cross_galleries=tuple(cross_all),
        seed=named_seed(cfg.seed, "reserved-reference-linkage", include_test),
        n_candidates=cfg.linkage_candidates,
        episodes=cfg.linkage_episodes,
    )


def _identity_bootstrap_ci(
    per_identity: Sequence[float], cfg: ExperimentConfig, seed: int
) -> list[float]:
    values = np.asarray(per_identity, dtype=np.float64)
    if not len(values):
        return [math.nan, math.nan]
    rng = np.random.default_rng(seed)
    means = []
    remaining = max(200, cfg.bootstrap_resamples)
    while remaining:
        batch = min(500, remaining)
        indices = rng.integers(0, len(values), size=(batch, len(values)))
        means.append(values[indices].mean(axis=1))
        remaining -= batch
    return [float(value) for value in np.quantile(np.concatenate(means), [0.025, 0.975])]


def _candidate_image(data: CelebADataBundle, token: CandidateToken) -> Tensor:
    kind, key = token
    if kind == "reference":
        position = data.references.by_identity()[int(key)]
        return data.references.raw_uint8[position]
    if kind.startswith("source_"):
        split = kind.removeprefix("source_")
        return data.split(split).raw_uint8[int(key)]
    raise KeyError(token)


def _make_gallery_embedding_cache(
    data: CelebADataBundle,
    manifest: FaceLinkageManifest,
    face_model: nn.Module,
    cfg: ExperimentConfig,
) -> dict[CandidateToken, Tensor]:
    unique = sorted(
        {
            token
            for task in (manifest.source_galleries, manifest.cross_galleries)
            for episodes in task
            for gallery in episodes
            for token in gallery
        }
    )
    images = torch.stack([_candidate_image(data, token) for token in unique])
    embeddings = _face_embeddings(face_model, images, cfg)
    return {token: embeddings[position] for position, token in enumerate(unique)}


def _evaluate_face_linkage(
    recovered_queries: Tensor,
    manifest: FaceLinkageManifest,
    embedding_cache: Mapping[CandidateToken, Tensor],
    face_model: nn.Module,
    cfg: ExperimentConfig,
) -> dict[str, Any]:
    recovered_embeddings = _face_embeddings(face_model, recovered_queries.clamp(0, 1), cfg)
    result: dict[str, Any] = {
        "chance_top1": LINKAGE_CHANCE,
        "n_identities": len(manifest.query_identities),
        "episodes_per_identity": manifest.episodes,
        "manifest_hash": manifest.manifest_hash,
        "bootstrap_unit": "identity",
    }
    for task, episode_groups in (
        ("source_image", manifest.source_galleries),
        ("cross_image_identity", manifest.cross_galleries),
    ):
        per_identity = []
        ranks = []
        for position, galleries in enumerate(episode_groups):
            identity = int(manifest.query_identities[position])
            split = manifest.query_splits[position]
            source_index = int(manifest.query_split_indices[position])
            genuine: CandidateToken = (
                (f"source_{split}", source_index)
                if task == "source_image"
                else ("reference", identity)
            )
            successes = []
            for gallery in galleries:
                candidates = torch.stack([embedding_cache[token] for token in gallery])
                similarities = candidates @ recovered_embeddings[position]
                true_position = gallery.index(genuine)
                order = torch.argsort(similarities, descending=True)
                rank = int(torch.nonzero(order == true_position, as_tuple=False)[0]) + 1
                ranks.append(rank)
                successes.append(int(rank == 1))
            per_identity.append(float(np.mean(successes)))
        rank_array = np.asarray(ranks, dtype=np.int64)
        result[task] = {
            "top1": float(np.mean(rank_array == 1)),
            "top3": float(np.mean(rank_array <= 3)),
            "mrr": float(np.mean(1.0 / rank_array)),
            "mean_rank": float(rank_array.mean()),
            "identity_top1": per_identity,
            "ci95_identity_cluster": _identity_bootstrap_ci(
                per_identity, cfg, named_seed(cfg.seed, "linkage-ci", task)
            ),
            "n_episodes": int(len(rank_array)),
        }
    return result


def _reconstruction_metrics(
    original_uint8: Tensor,
    recovered_flat: Tensor,
    cfg: ExperimentConfig,
    seed: int,
) -> dict[str, Any]:
    original = original_uint8.float().flatten(1) / 255.0
    recovered = recovered_flat.float().reshape_as(original)
    error = recovered - original
    mse = error.square().mean(dim=1)
    nrmse = float(
        torch.linalg.vector_norm(error)
        / torch.linalg.vector_norm(original).clamp_min(1.0e-12)
    )
    clipped = recovered.clamp(0, 1)
    clipped_mse = (clipped - original).square().mean(dim=1)
    psnr = 10.0 * torch.log10(1.0 / clipped_mse.clamp_min(1.0e-12))
    count = min(len(original), cfg.ssim_max_images)
    rng = np.random.default_rng(seed)
    chosen = (
        np.sort(rng.choice(len(original), count, replace=False))
        if count
        else np.empty(0, dtype=int)
    )
    ssim_values = []
    try:
        from skimage.metrics import structural_similarity

        for index in chosen:
            left = original[index].reshape(*RAW_SHAPE).permute(1, 2, 0).numpy()
            right = clipped[index].reshape(*RAW_SHAPE).permute(1, 2, 0).numpy()
            ssim_values.append(
                float(structural_similarity(left, right, data_range=1.0, channel_axis=2))
            )
        backend = "skimage"
    except ImportError:
        backend = "unavailable"
    return {
        "mse": float(mse.mean()),
        "rmse": float(torch.sqrt(mse.mean())),
        "nrmse": nrmse,
        "psnr_db": float(psnr.mean()),
        "ssim": float(np.mean(ssim_values)) if ssim_values else math.nan,
        "ssim_n": len(ssim_values),
        "ssim_backend": backend,
        "clipped_fraction": float(((recovered < 0) | (recovered > 1)).float().mean()),
        "n": int(len(original)),
    }


def _reduced_metrics(original: Tensor, recovered: Tensor) -> dict[str, float]:
    error = recovered.float() - original.float()
    return {
        "mse": float(error.square().mean()),
        "rmse": float(torch.sqrt(error.square().mean())),
        "nrmse": float(
            torch.linalg.vector_norm(error)
            / torch.linalg.vector_norm(original.float()).clamp_min(1.0e-12)
        ),
    }


def _relative_error(left: Tensor, right: Tensor) -> float:
    left_cpu = left.detach().double().cpu()
    right_cpu = right.detach().double().cpu()
    return float(
        torch.linalg.vector_norm(left_cpu - right_cpu)
        / torch.linalg.vector_norm(right_cpu).clamp_min(1.0e-15)
    )


def _fixed_pa(blocks: Sequence[Tensor], template: cp.ProtocolResult) -> cp.ProtocolResult:
    means = [block.mean(0) for block in template.anchor_uploads or []]
    uploads = [
        x @ rotation + translation
        for x, rotation, translation in zip(blocks, template.rotations, template.translations)
    ]
    aligned = [
        (upload - mean) @ rotation
        for upload, mean, rotation in zip(
            uploads, means, template.alignment_rotations or []
        )
    ]
    return cp.ProtocolResult(
        name="PA-I-GDP",
        private_uploads=uploads,
        collaboration_blocks=aligned,
        collaboration=torch.cat(aligned),
        rotations=template.rotations,
        translations=template.translations,
        anchor_uploads=template.anchor_uploads,
        alignment_rotations=template.alignment_rotations,
        anchor_noises=template.anchor_noises,
        diagnostics=template.diagnostics,
    )


def _noise_tensor(
    cfg: ExperimentConfig,
    shape: Sequence[int],
    *,
    outer: int,
    draw: int,
    split: str,
    participant: int,
    family: str,
    device: torch.device,
) -> Tensor:
    generator = torch.Generator(device=device).manual_seed(
        named_seed(cfg.seed, "noise", family, outer, draw, split, participant)
    )
    return torch.randn(tuple(shape), generator=generator, device=device, dtype=torch.float64)


def _diag(
    result: cp.ProtocolResult, cfg: ExperimentConfig, anchor_rms: float
) -> Optional[dict[str, Any]]:
    diagnostics = result.diagnostics
    if diagnostics is None:
        return None
    p, h, rows = cfg.n_participants, cfg.reduced_dim, cfg.anchor_rows
    empirical_sigma = (
        1.89
        * diagnostics.anchor_sigma_min
        * math.sqrt(p)
        / (math.sqrt(p * h) + math.sqrt(rows - 1) + 2.0 * math.sqrt(p * math.log(p)))
    )
    return {
        "iterations": diagnostics.iterations,
        "converged": diagnostics.converged,
        "termination_reason": "step_tolerance" if diagnostics.converged else "iteration_cap",
        "final_step": diagnostics.step_history[-1] if diagnostics.step_history else math.nan,
        "elapsed_seconds": diagnostics.elapsed_seconds,
        "anchor_sigma_min": diagnostics.anchor_sigma_min,
        "anchor_sigma_max": diagnostics.anchor_sigma_max,
        "anchor_condition": diagnostics.anchor_condition,
        "anchor_sigma_min_over_sqrt_r": diagnostics.anchor_sigma_min / math.sqrt(rows),
        "eigengap": diagnostics.eigengap,
        "relative_eigengap": diagnostics.relative_eigengap,
        "relative_gopp_residual": diagnostics.relative_gopp_residual,
        "fixed_point_residual": diagnostics.fixed_point_residual,
        "multistart_objective_gap": diagnostics.multistart_objective_gap,
        "objective_history": diagnostics.objective_history,
        "step_history": diagnostics.step_history,
        "empirical_phase_transition_lambda": empirical_sigma / max(anchor_rms, 1.0e-15),
        "empirical_phase_transition_status": "Ling empirical scaling; heuristic, not theorem",
    }


class AtomicRunStore:
    def __init__(self, output_dir: Path, stem: str = "celeba_results") -> None:
        self.output_dir = output_dir
        self.run_dir = output_dir / "runs"
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self.stem = stem

    def path(self, run_id: str) -> Path:
        return self.run_dir / f"{run_id}.json"

    def contains(self, run_id: str) -> bool:
        return self.path(run_id).is_file()

    def append(self, record: Mapping[str, Any]) -> None:
        path = self.path(str(record["run_id"]))
        if not path.exists():
            _atomic_json(path, {"schema_version": SCHEMA_VERSION, **dict(record)})

    def records(self) -> list[dict[str, Any]]:
        return [
            json.loads(path.read_text(encoding="utf-8"))
            for path in sorted(self.run_dir.glob("*.json"))
        ]

    def aggregate(self) -> tuple[Path, Path]:
        records = self.records()
        json_path = self.output_dir / f"{self.stem}.json"
        csv_path = self.output_dir / f"{self.stem}.csv"
        _atomic_json(json_path, {"schema_version": SCHEMA_VERSION, "records": records})

        def flatten(value: Mapping[str, Any], prefix: str = "") -> dict[str, Any]:
            output: dict[str, Any] = {}
            for key, item in value.items():
                name = f"{prefix}.{key}" if prefix else str(key)
                if isinstance(item, Mapping):
                    output.update(flatten(item, name))
                elif isinstance(item, (list, tuple)):
                    output[name] = json.dumps(item, separators=(",", ":"))
                else:
                    output[name] = item
            return output

        flattened = [flatten(record) for record in records]
        fields = sorted({key for row in flattened for key in row})
        with tempfile.NamedTemporaryFile(
            "w", encoding="utf-8", newline="", delete=False, dir=self.output_dir, suffix=".tmp"
        ) as handle:
            writer = csv.DictWriter(handle, fieldnames=fields)
            writer.writeheader()
            writer.writerows(flattened)
            temporary = Path(handle.name)
        os.replace(temporary, csv_path)
        return json_path, csv_path


@dataclass
class _Context:
    cfg: ExperimentConfig
    out: Path
    data: CelebADataBundle
    blocks: dict[str, tuple[list[Tensor], list[Tensor], list[Tensor]]]
    labels: dict[str, Tensor]
    face_model: nn.Module
    face_meta: dict[str, Any]
    linkage_manifest: FaceLinkageManifest
    gallery_embedding_cache: dict[CandidateToken, Tensor]
    decoder: Tensor
    original_query_images: Tensor
    original_query_reduced: Tensor
    original_image_control: dict[str, Any]
    reduction_floor_control: dict[str, Any]


def _query_tensor_from_blocks(
    manifest: FaceLinkageManifest,
    blocks: Mapping[str, tuple[list[Tensor], list[Tensor], list[Tensor]]],
    participant: int,
) -> Tensor:
    rows = [
        blocks[split][0][participant][position]
        for split, position in zip(manifest.query_splits, manifest.query_target_positions)
    ]
    return torch.stack(rows) if rows else torch.empty((0, 0), dtype=torch.float64)


def _query_raw_images(data: CelebADataBundle, manifest: FaceLinkageManifest) -> Tensor:
    return torch.stack(
        [
            data.split(split).raw_uint8[index]
            for split, index in zip(manifest.query_splits, manifest.query_split_indices)
        ]
    )


def _make_context(cfg: ExperimentConfig, out: Path, include_test: bool) -> _Context:
    data = prepare_celeba_data(cfg, include_test=include_test)
    if cfg.mode in {"smoke", "pilot"} and data.manifest["test_pixels_opened"]:
        raise AssertionError("Validation-only stage opened test pixels")
    _write_or_validate_json(out / "data_manifest.json", data.manifest, "data manifest")
    device = resolve_device(cfg.device)
    blocks = {
        name: data.participant_blocks(name, device)
        for name in ("train", "val", "test")
    }
    labels = {name: torch.cat(blocks[name][1]) for name in blocks}
    linkage = make_face_linkage_manifest(data, blocks, cfg, include_test)
    _write_or_validate_json(
        out / "face_linkage_manifest.json", linkage.to_dict(), "linkage manifest"
    )
    face_model, face_meta = _load_face_embedder(cfg)
    _write_or_validate_json(
        out / "face_embedder_manifest.json", face_meta, "face embedder manifest"
    )
    gallery_cache = _make_gallery_embedding_cache(data, linkage, face_model, cfg)
    decoder = data.reducer.F_pinv.to(device)
    original_queries = _query_raw_images(data, linkage)
    reduced_queries = _query_tensor_from_blocks(linkage, blocks, cfg.target_participant).float().cpu()
    floor_flat = reduced_queries.to(device) @ decoder
    floor_queries = floor_flat.reshape(-1, *RAW_SHAPE).cpu()
    original_control = {
        "role": "unperturbed face-auditor ceiling",
        "face_linkage": _evaluate_face_linkage(
            original_queries.float() / 255.0,
            linkage,
            gallery_cache,
            face_model,
            cfg,
        ),
    }
    floor_control = {
        "role": "public rank-400 Moore-Penrose reconstruction floor",
        "pixel": _reconstruction_metrics(
            original_queries,
            floor_flat.cpu(),
            cfg,
            named_seed(cfg.seed, "public-svd-floor"),
        ),
        "face_linkage": _evaluate_face_linkage(
            floor_queries,
            linkage,
            gallery_cache,
            face_model,
            cfg,
        ),
    }
    _atomic_json(out / "original_image_control.json", original_control)
    _atomic_json(out / "public_svd_floor_control.json", floor_control)
    return _Context(
        cfg=cfg,
        out=out,
        data=data,
        blocks=blocks,
        labels=labels,
        face_model=face_model,
        face_meta=face_meta,
        linkage_manifest=linkage,
        gallery_embedding_cache=gallery_cache,
        decoder=decoder,
        original_query_images=original_queries,
        original_query_reduced=reduced_queries,
        original_image_control=original_control,
        reduction_floor_control=floor_control,
    )


def _deployment(ctx: _Context, outer: int = 0) -> dict[str, Any]:
    cfg = ctx.cfg
    device = resolve_device(cfg.device)
    generator = cp.make_generator(named_seed(cfg.seed, "gdp-parameters", outer), device)
    rotations, translations = cp.sample_gdp_parameters(
        cfg.n_participants, cfg.reduced_dim, generator=generator, device=device
    )
    anchor_generator = torch.Generator(device="cpu").manual_seed(
        named_seed(cfg.seed, "anchor", outer)
    )
    anchor = torch.randn(
        (cfg.anchor_rows, cfg.reduced_dim), generator=anchor_generator, dtype=torch.float64
    ).to(device)
    centered = anchor - anchor.mean(0)
    return {
        "rotations": rotations,
        "translations": translations,
        "anchor": anchor,
        "private_rms": float(torch.sqrt(ctx.data.train.reduced.double().square().mean())),
        "anchor_rms": float(torch.sqrt(centered.square().mean())),
    }


def _build_transcripts(
    ctx: _Context,
    deployment: Mapping[str, Any],
    method: str,
    outer: int,
    draw: int,
    level: float,
    *,
    multistart: bool = False,
) -> tuple[dict[str, cp.ProtocolResult], float, float]:
    cfg = ctx.cfg
    rotations = deployment["rotations"]
    translations = deployment["translations"]
    reference = cfg.reference_participant
    sigma = level * float(deployment["private_rms"]) if method == "c_gdp_an" else 0.0
    anchor_sigma = level * float(deployment["anchor_rms"]) if method == "pa_i_gdp" else 0.0
    transcripts: dict[str, cp.ProtocolResult] = {}
    if method == "pa_i_gdp":
        anchor = deployment["anchor"]
        noises = [
            anchor_sigma
            * _noise_tensor(
                cfg,
                anchor.shape,
                outer=outer,
                draw=draw,
                split="anchor",
                participant=participant,
                family="pa",
                device=anchor.device,
            )
            for participant in range(cfg.n_participants)
        ]
        train = cp.pa_i_gdp(
            ctx.blocks["train"][0],
            anchor,
            anchor_sigma,
            rotations=rotations,
            translations=translations,
            anchor_noises=noises,
            max_iters=cfg.pa_iterations,
            tol=cfg.pa_tolerance,
            n_random_starts=4 if multistart else 0,
            restart_seed=named_seed(cfg.seed, "gpm-restart", outer, draw),
        )
        transcripts["train"] = train
        for split in ("val", "test"):
            transcripts[split] = _fixed_pa(ctx.blocks[split][0], train)
        return transcripts, sigma, anchor_sigma

    for split in ("train", "val", "test"):
        xs = ctx.blocks[split][0]
        if method == "c_gdp":
            transcripts[split] = cp.c_gdp(
                xs, rotation=rotations[reference], translation=translations[reference]
            )
        elif method == "c_gdp_an":
            noises = [
                sigma
                * _noise_tensor(
                    cfg,
                    x.shape,
                    outer=outer,
                    draw=draw,
                    split=split,
                    participant=participant,
                    family="c-gdp-an",
                    device=x.device,
                )
                for participant, x in enumerate(xs)
            ]
            transcripts[split] = cp.c_gdp_an(
                xs,
                sigma,
                rotation=rotations[reference],
                translation=translations[reference],
                noises=noises,
            )
        elif method == "i_gdp":
            transcripts[split] = cp.i_gdp(
                xs, rotations=rotations, translations=translations
            )
        else:
            raise KeyError(method)
    return transcripts, sigma, anchor_sigma


def _utility(
    ctx: _Context,
    transcripts: Mapping[str, cp.ProtocolResult],
    seed: int,
    checkpoint_path: Path,
) -> dict[str, Any]:
    cfg = ctx.cfg
    train_x = transcripts["train"].collaboration.float().reshape(
        -1, 1, cfg.projection_side, cfg.projection_side
    )
    val_x = transcripts["val"].collaboration.float().reshape(
        -1, 1, cfg.projection_side, cfg.projection_side
    )
    model, training = _train_binary(
        train_x,
        ctx.labels["train"],
        val_x,
        ctx.labels["val"],
        cfg,
        seed,
        checkpoint_path,
    )
    utility = dict(training)
    validation = _evaluate_binary(model, val_x, ctx.labels["val"], cfg)
    utility.update({f"validation_{key}": value for key, value in validation.items()})
    if len(ctx.labels["test"]):
        test_x = transcripts["test"].collaboration.float().reshape(
            -1, 1, cfg.projection_side, cfg.projection_side
        )
        test = _evaluate_binary(model, test_x, ctx.labels["test"], cfg)
        utility.update({f"test_{key}": value for key, value in test.items()})
    return utility


def _svd_utility_control(ctx: _Context) -> dict[str, Any]:
    """Train the same CNN on Z=XF without any GDP transformation."""

    cfg = ctx.cfg
    train_x = torch.cat(ctx.blocks["train"][0]).float().reshape(
        -1, 1, cfg.projection_side, cfg.projection_side
    )
    val_x = torch.cat(ctx.blocks["val"][0]).float().reshape(
        -1, 1, cfg.projection_side, cfg.projection_side
    )
    checkpoint = ctx.out / "checkpoints" / "svd_utility_control.pt"
    model, training = _train_binary(
        train_x,
        ctx.labels["train"],
        val_x,
        ctx.labels["val"],
        cfg,
        named_seed(cfg.seed, "fixed-noise-sensitivity-model", 0),
        checkpoint,
    )
    result = {
        **training,
        **{
            f"validation_{key}": value
            for key, value in _evaluate_binary(model, val_x, ctx.labels["val"], cfg).items()
        },
        "role": "utility-only public-SVD control; no GDP",
    }
    if len(ctx.labels["test"]):
        test_x = torch.cat(ctx.blocks["test"][0]).float().reshape(
            -1, 1, cfg.projection_side, cfg.projection_side
        )
        result.update(
            {
                f"test_{key}": value
                for key, value in _evaluate_binary(
                    model, test_x, ctx.labels["test"], cfg
                ).items()
            }
        )
    _atomic_json(ctx.out / "svd_utility_control.json", result)
    return result


def _concatenated_private_queries(
    transcripts: Mapping[str, cp.ProtocolResult],
    manifest: FaceLinkageManifest,
    participant: int,
) -> Tensor:
    rows = [
        transcripts[split].private_uploads[participant][position]
        for split, position in zip(manifest.query_splits, manifest.query_target_positions)
    ]
    return torch.stack(rows)


def _attack_results(
    ctx: _Context,
    method: str,
    transcripts: Mapping[str, cp.ProtocolResult],
    deployment: Mapping[str, Any],
    preview_path: Optional[Path] = None,
) -> tuple[list[dict[str, Any]], Optional[dict[str, Any]]]:
    cfg = ctx.cfg
    colluder, target = cfg.reference_participant, cfg.target_participant
    query_uploads = _concatenated_private_queries(
        transcripts, ctx.linkage_manifest, target
    )
    train_transcript = transcripts["train"]
    candidates: list[tuple[cp.AttackResult, str]] = []
    if method in {"c_gdp", "c_gdp_an"}:
        candidates.append(
            (
                cp.attack_c_known_input(
                    ctx.blocks["train"][0][colluder],
                    train_transcript.private_uploads[colluder],
                    query_uploads,
                    estimator="mp" if method == "c_gdp_an" else "exact",
                ),
                "primary",
            )
        )
        if method == "c_gdp_an":
            candidates.append(
                (
                    cp.attack_c_known_input(
                        ctx.blocks["train"][0][colluder],
                        train_transcript.private_uploads[colluder],
                        query_uploads,
                        estimator="op",
                    ),
                    "primary",
                )
            )
            candidates.append(
                (
                    cp.invert_known_gdp(
                        query_uploads,
                        train_transcript.rotations[colluder],
                        train_transcript.translations[colluder],
                    ),
                    "primary-known-secret",
                )
            )
    elif method == "i_gdp":
        candidates.append(
            (
                cp.attack_i_distribution_moment(
                    ctx.blocks["train"][0][colluder],
                    train_transcript.private_uploads[target],
                    query_uploads,
                ),
                "primary-distribution-sensitivity",
            )
        )
        candidates.append(
            (
                cp.attack_i_colluder_negative_control(
                    query_uploads,
                    train_transcript.rotations[colluder],
                    train_transcript.translations[colluder],
                ),
                "negative-control",
            )
        )
        oracle = cp.invert_known_gdp(
            query_uploads,
            train_transcript.rotations[target],
            train_transcript.translations[target],
        )
        oracle.method = "I-target-secret-oracle"
        candidates.append((oracle, "oracle-outside-threat-model"))
    elif method == "pa_i_gdp":
        anchor = deployment["anchor"]
        candidates.extend(
            [
                (
                    cp.attack_pa_mp(
                        anchor,
                        train_transcript.anchor_uploads[target],
                        query_uploads,
                    ),
                    "primary",
                ),
                (
                    cp.attack_pa_op(
                        anchor,
                        train_transcript.anchor_uploads[target],
                        query_uploads,
                    ),
                    "primary",
                ),
                (
                    cp.attack_pa_alignment_calibration(
                        anchor,
                        train_transcript.anchor_uploads[target],
                        query_uploads,
                        train_transcript.alignment_rotations[target],
                        [train_transcript.rotations[colluder]],
                        [train_transcript.alignment_rotations[colluder]],
                    ),
                    "primary",
                ),
            ]
        )
    evaluated: list[dict[str, Any]] = []
    preview_images: list[tuple[str, Tensor]] = []
    for attack, role in candidates:
        recovered_reduced = attack.reduced.float().cpu()
        decoded_flat = (
            recovered_reduced.to(ctx.decoder.device) @ ctx.decoder
        ).cpu()
        decoded_images = decoded_flat.reshape(-1, *RAW_SHAPE).clamp(0, 1)
        reduced = _reduced_metrics(ctx.original_query_reduced, recovered_reduced)
        pixel = _reconstruction_metrics(
            ctx.original_query_images,
            decoded_flat,
            cfg,
            named_seed(cfg.seed, "reconstruction", method, attack.method),
        )
        linkage = _evaluate_face_linkage(
            decoded_images,
            ctx.linkage_manifest,
            ctx.gallery_embedding_cache,
            ctx.face_model,
            cfg,
        )
        true_rotation = train_transcript.rotations[target]
        true_translation = train_transcript.translations[target].reshape(-1)
        metrics = {
            "reduced": reduced,
            "pixel": pixel,
            "face_linkage": linkage,
            "rotation_relative_error": _relative_error(attack.rotation, true_rotation),
            "translation_relative_error": float(
                torch.linalg.vector_norm(
                    attack.translation.reshape(-1).to(true_translation) - true_translation
                )
                / torch.linalg.vector_norm(true_translation).clamp_min(1.0e-15)
            ),
            "exact_reduced_recovery_1e-6": reduced["nrmse"] <= 1.0e-6,
        }
        evaluated.append({"name": attack.method, "role": role, "metrics": metrics})
        preview_images.append((attack.method, decoded_images[:10]))
    valid = [item for item in evaluated if item["role"].startswith("primary")]
    best = (
        max(
            valid,
            key=lambda item: (
                item["metrics"]["face_linkage"]["cross_image_identity"]["top1"],
                item["metrics"]["face_linkage"]["source_image"]["top1"],
                -item["metrics"]["pixel"]["nrmse"],
            ),
        )
        if valid
        else None
    )
    if preview_path is not None and preview_images:
        _save_preview(ctx, preview_images, evaluated, method, preview_path)
        if best is not None:
            best["preview_path"] = str(preview_path)
    return evaluated, best


def _save_preview(
    ctx: _Context,
    attack_images: Sequence[tuple[str, Tensor]],
    evaluated: Sequence[Mapping[str, Any]],
    method: str,
    path: Path,
) -> None:
    import matplotlib.pyplot as plt

    count = min(10, len(ctx.original_query_images))
    originals = ctx.original_query_images[:count].float() / 255.0
    floor = (
        ctx.original_query_reduced[:count].to(ctx.decoder.device) @ ctx.decoder
    ).cpu().reshape(-1, *RAW_SHAPE).clamp(0, 1)
    original_linkage = ctx.original_image_control["face_linkage"]
    floor_linkage = ctx.reduction_floor_control["face_linkage"]
    rows: list[tuple[str, Tensor]] = [
        (
            "Original\n"
            f"Cross={original_linkage['cross_image_identity']['top1']:.3f}; "
            f"source={original_linkage['source_image']['top1']:.3f}",
            originals,
        ),
        (
            "Public-SVD floor\n"
            f"Cross={floor_linkage['cross_image_identity']['top1']:.3f}; "
            f"source={floor_linkage['source_image']['top1']:.3f}",
            floor,
        ),
    ]
    by_name = {item["name"]: item for item in evaluated}
    for name, images in attack_images:
        cross = by_name[name]["metrics"]["face_linkage"]["cross_image_identity"]["top1"]
        source = by_name[name]["metrics"]["face_linkage"]["source_image"]["top1"]
        rows.append((f"{name}\nCross={cross:.3f}; source={source:.3f}", images[:count]))
    figure, axes = plt.subplots(
        len(rows),
        count + 1,
        figsize=(3.2 + 1.45 * count, 1.45 * len(rows)),
        gridspec_kw={"width_ratios": [3.0] + [1.0] * count},
        squeeze=False,
    )
    for row_index, (label, images) in enumerate(rows):
        axes[row_index, 0].axis("off")
        axes[row_index, 0].text(
            0.02,
            0.5,
            label,
            transform=axes[row_index, 0].transAxes,
            ha="left",
            va="center",
            fontsize=9.5,
        )
        for column in range(count):
            axes[row_index, column + 1].imshow(
                images[column].permute(1, 2, 0).clamp(0, 1)
            )
            axes[row_index, column + 1].axis("off")
    figure.suptitle(
        f"{method}: protected participant {ctx.cfg.target_participant}; "
        "reserved-reference 10-way linkage",
        fontsize=12,
    )
    figure.subplots_adjust(
        left=0.01,
        right=0.995,
        bottom=0.01,
        top=0.94,
        wspace=0.06,
        hspace=0.08,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp.png")
    figure.savefig(temporary, dpi=180, bbox_inches="tight")
    plt.close(figure)
    os.replace(temporary, path)


def _execute_fit(
    ctx: _Context,
    store: AtomicRunStore,
    deployment: Mapping[str, Any],
    *,
    method: str,
    outer: int,
    draw: int,
    run_role: str,
    level: float,
) -> dict[str, Any]:
    cfg = ctx.cfg
    run_identity = {
        "config": stable_hash(cfg.to_dict()),
        "data": stable_hash(ctx.data.manifest),
        "method": method,
        "outer": outer,
        "draw": draw,
        "run_role": run_role,
        "lambda": float(level),
    }
    run_id = stable_hash(run_identity, 24)
    if store.contains(run_id):
        return json.loads(store.path(run_id).read_text(encoding="utf-8"))
    started = time.perf_counter()
    transcripts, sigma, anchor_sigma = _build_transcripts(
        ctx,
        deployment,
        method,
        outer,
        draw,
        level,
        multistart=run_role in {"smoke", "endpoint_zero", "endpoint_upper"},
    )
    checkpoint = ctx.out / "checkpoints" / f"{run_id}.pt"
    utility = _utility(
        ctx,
        transcripts,
        named_seed(cfg.seed, "fixed-noise-sensitivity-model", outer),
        checkpoint,
    )
    preview = (
        ctx.out / "previews" / f"{method}_{run_role}_seed{outer}_draw{draw}.png"
        if run_role
        in {
            "smoke",
            "clean",
            "endpoint_zero",
            "endpoint_upper",
            "i_gdp_equivalence_control",
        }
        else None
    )
    attacks, best = (
        _attack_results(ctx, method, transcripts, deployment, preview)
        if cfg.run_attacks
        else ([], None)
    )
    train_result = transcripts["train"]
    record = {
        "run_id": run_id,
        "dataset": DATASET_NAME,
        "task": TASK_NAME,
        "protocol": method,
        "protocol_name": train_result.name,
        "outer_seed": outer,
        "draw": draw,
        "run_role": run_role,
        "h": cfg.reduced_dim,
        "lambda": float(level),
        "private_lambda": float(level) if method == "c_gdp_an" else 0.0,
        "anchor_lambda": float(level) if method == "pa_i_gdp" else 0.0,
        "noise_sigma": sigma,
        "anchor_noise_sigma": anchor_sigma,
        "utility": utility,
        "attacks": attacks,
        "best_attack": best,
        "privacy_query_splits": sorted(set(ctx.linkage_manifest.query_splits)),
        "privacy_identity_clusters": len(ctx.linkage_manifest.query_identities),
        "gpm": _diag(train_result, cfg, float(deployment["anchor_rms"])),
        "checkpoint": {
            "path": str(checkpoint),
            "sha256": file_sha256(checkpoint),
        },
        "runtime_seconds": time.perf_counter() - started,
        "runtime": {
            "device": str(resolve_device(cfg.device)),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "torch": torch.__version__,
            "cuda": torch.version.cuda,
        },
    }
    store.append(record)
    print(
        f"completed {method} role={run_role} draw={draw} "
        f"lambda={level:.6g} runtime={record['runtime_seconds']:.1f}s"
    )
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return record


def _stage_config(
    config: Mapping[str, Any] | ExperimentConfig,
    output_dir: str | Path,
    mode: str,
) -> tuple[ExperimentConfig, Path]:
    cfg = _coerce_config(config)
    if cfg.mode != mode:
        cfg = replace(cfg, mode=mode)
    base = Path(output_dir)
    out = base / mode
    out.mkdir(parents=True, exist_ok=True)
    cfg = replace(cfg, output_dir=str(out))
    _write_or_validate_json(out / "config.json", cfg.to_dict(), "resolved configuration")
    return cfg, out


def _render_stage(payload: Mapping[str, Any], out: Path) -> dict[str, Any]:
    import matplotlib.pyplot as plt

    records = list(payload.get("records", []))
    summary_path = out / "summary.json"
    report_path = out / "report.md"
    utility_path = out / "celeba_public_svd_utility_noise.png"
    privacy_path = out / "celeba_public_svd_privacy_utility.png"
    gpm_path = out / "celeba_public_svd_gpm_diagnostics.png"
    _atomic_json(
        summary_path,
        {
            "schema_version": SCHEMA_VERSION,
            "mode": payload.get("mode"),
            "status": payload.get("status"),
            "n_records": len(records),
            "methods": sorted({row.get("protocol") for row in records}),
            "context": payload.get("context", {}),
        },
    )
    report_lines = [
        "# CelebA I-GDP-PA public identity-exclusive SVD report",
        "",
        f"Mode: {payload.get('mode')}",
        f"Completed records: {len(records)}",
        "",
        "Utility is Smiling balanced accuracy. Privacy is reserved-reference cross-image ten-way identity linkage.",
        "Smoke and pilot use participant-1 train+validation identities only; test pixels remain unopened.",
    ]
    context = payload.get("context", {})
    svd_control = context.get("svd_utility_control", {}) if isinstance(context, Mapping) else {}
    if svd_control:
        report_lines.extend(
            [
                "",
                f"Public-SVD control validation balanced accuracy: {float(svd_control['validation_balanced_accuracy']):.4f}.",
            ]
        )
    adequacy = context.get("adequacy", {}) if isinstance(context, Mapping) else {}
    if adequacy:
        recovery = adequacy.get("c_gdp_exact_reduced_recovery", {})
        if recovery:
            report_lines.append(
                "C-GDP exact reduced-space recovery NRMSE: "
                f"{float(recovery['nrmse']):.3e}."
            )
        report_lines.append(
            "Descriptive public-SVD minus zero-noise C-GDP balanced accuracy: "
            f"{float(adequacy['svd_minus_c_gdp_balanced_accuracy']):+.4f}."
        )
    temporary_report = report_path.with_suffix(".tmp")
    temporary_report.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
    os.replace(temporary_report, report_path)

    noisy = [row for row in records if row.get("protocol") in {"c_gdp_an", "pa_i_gdp"}]
    figure, axis = plt.subplots(figsize=(7.2, 4.4))
    for method, label in (("c_gdp_an", "C-GDP-AN"), ("pa_i_gdp", "PA-I-GDP")):
        rows = sorted(
            [row for row in noisy if row.get("protocol") == method],
            key=lambda row: float(row["lambda"]),
        )
        if rows:
            axis.scatter(
                [row["lambda"] for row in rows],
                [row["utility"]["validation_balanced_accuracy"] for row in rows],
                s=22,
                alpha=0.7,
                label=label,
            )
    axis.set(
        xlabel="Normalized noise level",
        ylabel="Validation balanced accuracy",
        title="CelebA validation utility response",
    )
    axis.grid(alpha=0.25)
    if noisy:
        axis.legend()
    figure.tight_layout()
    temporary = utility_path.with_suffix(".tmp.png")
    figure.savefig(temporary, dpi=180)
    plt.close(figure)
    os.replace(temporary, utility_path)

    figure, axis = plt.subplots(figsize=(6.3, 4.8))
    for method, label in (("c_gdp_an", "C-GDP-AN"), ("pa_i_gdp", "PA-I-GDP")):
        rows = [
            row
            for row in noisy
            if row.get("protocol") == method and row.get("best_attack")
        ]
        if rows:
            axis.scatter(
                [
                    row["best_attack"]["metrics"]["face_linkage"]
                    ["cross_image_identity"]["top1"]
                    for row in rows
                ],
                [row["utility"]["validation_balanced_accuracy"] for row in rows],
                s=22,
                alpha=0.7,
                label=label,
            )
    axis.axvline(LINKAGE_CHANCE, color="black", linestyle="--", linewidth=1)
    axis.set(
        xlabel="Reserved-reference cross-image top-1",
        ylabel="Validation balanced accuracy",
        title="Privacy--utility observations",
    )
    axis.grid(alpha=0.25)
    if noisy:
        axis.legend()
    figure.tight_layout()
    temporary = privacy_path.with_suffix(".tmp.png")
    figure.savefig(temporary, dpi=180)
    plt.close(figure)
    os.replace(temporary, privacy_path)

    pa_rows = [
        row for row in records if row.get("protocol") == "pa_i_gdp" and row.get("gpm")
    ]
    figure, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
    if pa_rows:
        normalized = [
            float(row["lambda"])
            / max(float(row["gpm"]["empirical_phase_transition_lambda"]), 1.0e-15)
            for row in pa_rows
        ]
        axes[0].scatter(normalized, [row["gpm"]["iterations"] for row in pa_rows], s=18)
        axes[1].scatter(
            normalized, [row["gpm"]["fixed_point_residual"] for row in pa_rows], s=18
        )
        axes[1].set_yscale("log")
    axes[0].set(xlabel="lambda / empirical boundary", ylabel="GPM iterations")
    axes[1].set(xlabel="lambda / empirical boundary", ylabel="Fixed-point residual")
    for axis in axes:
        axis.axvline(1.0, color="black", linestyle="--", linewidth=1)
        axis.grid(alpha=0.25)
    figure.tight_layout()
    temporary = gpm_path.with_suffix(".tmp.png")
    figure.savefig(temporary, dpi=180)
    plt.close(figure)
    os.replace(temporary, gpm_path)

    previews = [
        Path(str(row.get("best_attack", {}).get("preview_path")))
        for row in records
        if row.get("best_attack", {}).get("preview_path")
    ]
    reconstruction = next((path for path in previews if path.is_file()), None)
    reconstruction_grids = [str(path) for path in previews if path.is_file()]
    return {
        **dict(payload),
        "summary_path": str(summary_path),
        "report_path": str(report_path),
        "utility_plot": str(utility_path),
        "privacy_utility_plot": str(privacy_path),
        "gpm_diagnostic_plot": str(gpm_path),
        "reconstruction_grid": None if reconstruction is None else str(reconstruction),
        "reconstruction_grids": reconstruction_grids,
    }


def run_smoke(
    config: Mapping[str, Any] | ExperimentConfig,
    output_dir: str | Path,
) -> dict[str, Any]:
    cfg, out = _stage_config(config, output_dir, "smoke")
    ctx = _make_context(cfg, out, include_test=False)
    if len(ctx.linkage_manifest.query_identities) != 85:
        raise AssertionError("Smoke must use all 85 protected train+validation identities")
    svd_utility = _svd_utility_control(ctx)
    store = AtomicRunStore(out)
    deployment = _deployment(ctx)
    levels = {
        "c_gdp": 0.0,
        "c_gdp_an": float(cfg.private_lambda_upper or 0.25),
        "i_gdp": 0.0,
        "pa_i_gdp": float(cfg.anchor_lambda_upper or 0.25),
    }
    for draw, method in enumerate(METHODS):
        _execute_fit(
            ctx,
            store,
            deployment,
            method=method,
            outer=0,
            draw=draw,
            run_role="smoke",
            level=levels[method],
        )
    json_path, csv_path = store.aggregate()
    return _render_stage(
        {
            "mode": "smoke",
            "status": "complete-validation-only",
            "output_dir": str(out),
            "records": store.records(),
            "context": {
                "config": cfg.to_dict(),
                "data_manifest_hash": stable_hash(ctx.data.manifest),
                "test_pixels_opened": False,
                "face_embedder": ctx.face_meta,
                "original_image_control": ctx.original_image_control,
                "reduction_floor_control": ctx.reduction_floor_control,
                "svd_utility_control": svd_utility,
            },
            "json_path": str(json_path),
            "csv_path": str(csv_path),
        },
        out,
    )


def _pilot_evidence(
    records: Sequence[Mapping[str, Any]],
    method: str,
    ctx: _Context,
) -> tuple[Optional[float], list[dict[str, Any]], str]:
    rows = sorted(
        [row for row in records if row.get("protocol") == method],
        key=lambda row: float(row["lambda"]),
    )
    zero = next((row for row in rows if float(row["lambda"]) == 0.0), None)
    if zero is None:
        return None, [], "missing-zero-control"
    baseline = float(zero["utility"]["validation_balanced_accuracy"])
    evidence = []
    for row in rows:
        utility = float(row["utility"]["validation_balanced_accuracy"])
        retained = (
            (utility - UTILITY_CHANCE) / (baseline - UTILITY_CHANCE)
            if baseline > UTILITY_CHANCE
            else -math.inf
        )
        linkage = float(
            row["best_attack"]["metrics"]["face_linkage"]["cross_image_identity"]["top1"]
        )
        interval = row["best_attack"]["metrics"]["face_linkage"]
        interval = interval["cross_image_identity"]["ci95_identity_cluster"]
        privacy_trigger = bool(
            linkage <= 0.12
            and float(interval[0]) <= LINKAGE_CHANCE <= float(interval[1])
        )
        utility_trigger = retained <= 0.15
        evidence.append(
            {
                "lambda": float(row["lambda"]),
                "validation_balanced_accuracy": utility,
                "baseline_balanced_accuracy": baseline,
                "retained_utility": retained,
                "cross_image_top1": linkage,
                "cross_image_ci95": interval,
                "privacy_trigger": privacy_trigger,
                "utility_trigger": utility_trigger,
                "trigger": privacy_trigger or utility_trigger,
            }
        )
    positive = [item for item in evidence if item["lambda"] > 0]
    for left, right in zip(positive, positive[1:]):
        if left["trigger"] and right["trigger"]:
            return float(left["lambda"]), evidence, "first-of-two-consecutive-triggers"
    return None, evidence, "right-censored-no-two-consecutive-triggers"


def _pilot_adequacy(
    ctx: _Context,
    records: Sequence[Mapping[str, Any]],
    svd_utility: Mapping[str, Any],
) -> dict[str, Any]:
    zero_rows = {
        row["protocol"]: row
        for row in records
        if float(row["lambda"]) == 0.0
        and row["protocol"] in {"c_gdp", "c_gdp_an", "pa_i_gdp"}
    }
    baseline = {
        method: float(row["utility"]["validation_balanced_accuracy"])
        for method, row in zero_rows.items()
    }
    c_gdp_zero = baseline.get("c_gdp", math.nan)
    c_gdp_an_zero = baseline.get("c_gdp_an", math.nan)
    svd_balanced_accuracy = float(svd_utility["validation_balanced_accuracy"])
    svd_c_gdp_delta = svd_balanced_accuracy - c_gdp_zero
    c_gdp_row = zero_rows.get("c_gdp", {})
    exact_attacks = [
        attack
        for attack in c_gdp_row.get("attacks", [])
        if attack.get("name") == "C-exact" and attack.get("role") == "primary"
    ]
    try:
        c_gdp_exact_nrmse = (
            float(exact_attacks[0]["metrics"]["reduced"]["nrmse"])
            if len(exact_attacks) == 1
            else math.nan
        )
    except (KeyError, TypeError, ValueError):
        c_gdp_exact_nrmse = math.nan
    checks = {
        **ctx.data.manifest["active_design_checks"],
        "test_pixels_unopened": not bool(ctx.data.manifest["test_pixels_opened"]),
        "public_svd_exactly_10000": int(
            ctx.data.manifest["projection"]["fit_image_count"]
        )
        == PUBLIC_SVD_IMAGES,
        "protected_linkage_identity_clusters_85": len(ctx.linkage_manifest.query_identities) == 85,
        "c_gdp_zero_at_least_0.70": math.isfinite(c_gdp_zero) and c_gdp_zero >= 0.70,
        "c_gdp_and_zero_noise_variant_match": math.isfinite(c_gdp_an_zero)
        and abs(c_gdp_zero - c_gdp_an_zero) <= 1.0e-6,
        "c_gdp_exact_reduced_nrmse_at_most_1e-6": math.isfinite(c_gdp_exact_nrmse)
        and c_gdp_exact_nrmse <= C_GDP_EXACT_NRMSE_MAX,
        "pa_zero_above_chance": baseline.get("pa_i_gdp", 0.0) > 0.55,
        "original_linkage_above_chance": float(
            ctx.original_image_control["face_linkage"]["cross_image_identity"]["top1"]
        )
        > 0.12,
        "public_participant_identity_disjoint": not bool(
            set(ctx.data.manifest["projection"]["public_identity_ids"])
            & {
                int(identity)
                for identity in ctx.data.manifest["active_participant_identity_ids"]
            }
        ),
    }
    return {
        "gate_version": ADEQUACY_GATE_VERSION,
        "checks": checks,
        "baseline_validation_balanced_accuracy": baseline,
        "svd_validation_balanced_accuracy": svd_balanced_accuracy,
        "svd_minus_c_gdp_balanced_accuracy": svd_c_gdp_delta,
        "descriptive_controls": {
            "svd_minus_c_gdp_balanced_accuracy": svd_c_gdp_delta,
            "svd_vs_c_gdp_accuracy_is_gating": False,
        },
        "c_gdp_exact_reduced_recovery": {
            "attack": "C-exact",
            "query_splits": list(c_gdp_row.get("privacy_query_splits", [])),
            "identity_clusters": int(c_gdp_row.get("privacy_identity_clusters", 0)),
            "nrmse": c_gdp_exact_nrmse,
        },
        "paper_grade_thresholds": {
            "c_gdp_min": 0.70,
            "c_gdp_exact_reduced_nrmse_max": C_GDP_EXACT_NRMSE_MAX,
        },
        "adequacy_passed": all(checks.values()),
    }


def run_validation_pilot(
    config: Mapping[str, Any] | ExperimentConfig,
    output_dir: str | Path,
) -> dict[str, Any]:
    cfg, out = _stage_config(config, output_dir, "pilot")
    ctx = _make_context(cfg, out, include_test=False)
    if ctx.data.manifest["test_pixels_opened"]:
        raise AssertionError("Pilot opened test pixels")
    store = AtomicRunStore(out, stem="pilot_results")
    svd_utility = _svd_utility_control(ctx)
    deployment = _deployment(ctx)
    _execute_fit(
        ctx,
        store,
        deployment,
        method="c_gdp",
        outer=0,
        draw=0,
        run_role="pilot-clean-adequacy",
        level=0.0,
    )
    status: dict[str, Any] = {}
    for method in ("c_gdp_an", "pa_i_gdp"):
        for draw, level in enumerate(cfg.pilot_candidate_lambdas):
            _execute_fit(
                ctx,
                store,
                deployment,
                method=method,
                outer=0,
                draw=draw,
                run_role="pilot",
                level=float(level),
            )
            upper, evidence, reason = _pilot_evidence(store.records(), method, ctx)
            status[method] = {"upper": upper, "evidence": evidence, "reason": reason}
            if upper is not None:
                break
    records = store.records()
    adequacy = _pilot_adequacy(ctx, records, svd_utility)
    private_upper = status["c_gdp_an"]["upper"]
    anchor_upper = status["pa_i_gdp"]["upper"]
    calibration_passed = bool(
        adequacy["adequacy_passed"]
        and private_upper is not None
        and anchor_upper is not None
    )
    manifest = {
        "schema_version": SCHEMA_VERSION,
        "stage": "validation-only-pilot",
        "calibration_passed": calibration_passed,
        "adequacy": adequacy,
        "private_lambda_upper": private_upper,
        "anchor_lambda_upper": anchor_upper,
        "selection": status,
        "test_pixels_opened": False,
        "privacy_query_splits": ["train", "val"],
        "privacy_identity_clusters": len(ctx.linkage_manifest.query_identities),
        "bootstrap_unit": "identity",
        "master_manifest_hash": ctx.data.manifest["master_manifest_hash"],
        "active_participants": cfg.n_participants,
        "projection": ctx.data.manifest["projection"],
        "data_manifest_hash": stable_hash(ctx.data.manifest),
        "config_hash": stable_hash(cfg.to_dict()),
        "module_sha256": dict(cfg.module_sha256),
        "record_ids": sorted(str(row["run_id"]) for row in records),
        "record_hash": stable_hash(records),
    }
    manifest_path = out / (
        "frozen_pilot_manifest.json"
        if calibration_passed
        else "pilot_diagnostic_manifest.json"
    )
    _atomic_json(manifest_path, manifest)
    json_path, csv_path = store.aggregate()
    result = _render_stage(
        {
            "mode": "pilot",
            "status": "frozen" if calibration_passed else "diagnostic-not-frozen",
            "output_dir": str(out),
            "records": records,
            "context": {
                "test_pixels_opened": False,
                "adequacy": adequacy,
                "pilot_manifest": manifest,
                "svd_utility_control": svd_utility,
            },
            "pilot_manifest_path": str(manifest_path),
            "json_path": str(json_path),
            "csv_path": str(csv_path),
        },
        out,
    )
    return result


def _load_frozen_pilot(cfg: ExperimentConfig, base: Path) -> dict[str, Any]:
    path = Path(cfg.pilot_manifest_path or base / "pilot" / "frozen_pilot_manifest.json")
    if not path.is_file():
        raise FileNotFoundError("A passed frozen pilot manifest is required before full mode")
    if cfg.pilot_manifest_sha256 and file_sha256(path) != cfg.pilot_manifest_sha256:
        raise RuntimeError("Frozen pilot manifest SHA-256 mismatch")
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if not manifest.get("calibration_passed") or manifest.get("test_pixels_opened"):
        raise RuntimeError("Pilot is not a passed validation-only calibration")
    if int(manifest.get("active_participants", -1)) != cfg.n_participants:
        raise RuntimeError("Full mode participant count differs from the frozen pilot")
    if dict(manifest.get("module_sha256", {})) != dict(cfg.module_sha256):
        raise RuntimeError("Full mode embedded modules differ from the frozen pilot")
    return manifest


def _lambda_draw(cfg: ExperimentConfig, upper: float, draw: int, family: str) -> float:
    rng = np.random.default_rng(named_seed(cfg.seed, "uniform-noise", family, draw))
    return float(rng.uniform(0.0, upper))


def run_full_after_review(
    config: Mapping[str, Any] | ExperimentConfig,
    output_dir: str | Path,
) -> dict[str, Any]:
    """Explicit post-review entry point; never called by ``run_pipeline``."""

    incoming = _coerce_config(config)
    base = Path(output_dir)
    pilot = _load_frozen_pilot(incoming, base)
    cfg, out = _stage_config(
        replace(
            incoming,
            private_lambda_upper=float(pilot["private_lambda_upper"]),
            anchor_lambda_upper=float(pilot["anchor_lambda_upper"]),
        ),
        base,
        "full",
    )
    ctx = _make_context(cfg, out, include_test=True)
    if len(ctx.linkage_manifest.query_identities) != 100:
        raise AssertionError("Full mode must add the 15 untouched test identities")
    if ctx.data.manifest["master_manifest_hash"] != pilot["master_manifest_hash"]:
        raise RuntimeError("Full and pilot master manifests differ")
    if ctx.data.manifest["projection"]["basis_matrix_sha256"] != pilot["projection"]["basis_matrix_sha256"]:
        raise RuntimeError("Full and pilot public SVD bases differ")
    store = AtomicRunStore(out)
    deployment = _deployment(ctx)
    schedule = [
        {"method": "c_gdp", "outer": 0, "draw": 0, "run_role": "clean", "level": 0.0},
        {"method": "i_gdp", "outer": 0, "draw": 0, "run_role": "clean", "level": 0.0},
    ]
    for draw in range(100):
        schedule.extend(
            [
                {
                    "method": "c_gdp_an",
                    "outer": 0,
                    "draw": draw,
                    "run_role": "random",
                    "level": _lambda_draw(cfg, float(cfg.private_lambda_upper), draw, "private"),
                },
                {
                    "method": "pa_i_gdp",
                    "outer": 0,
                    "draw": draw,
                    "run_role": "random",
                    "level": _lambda_draw(cfg, float(cfg.anchor_lambda_upper), draw, "anchor"),
                },
            ]
        )
    schedule.extend(
        [
            {"method": "c_gdp_an", "outer": 0, "draw": 100, "run_role": "endpoint_zero", "level": 0.0},
            {"method": "pa_i_gdp", "outer": 0, "draw": 100, "run_role": "endpoint_zero", "level": 0.0},
            {"method": "c_gdp_an", "outer": 0, "draw": 101, "run_role": "endpoint_upper", "level": float(cfg.private_lambda_upper)},
            {"method": "pa_i_gdp", "outer": 0, "draw": 101, "run_role": "endpoint_upper", "level": float(cfg.anchor_lambda_upper)},
        ]
    )
    if len(schedule) != 206:
        raise AssertionError("Full schedule must contain exactly 206 fits")
    _write_or_validate_json(
        out / "full_run_manifest.json",
        {
            "schema_version": SCHEMA_VERSION,
            "schedule": schedule,
            "pilot_manifest_sha256": file_sha256(
                Path(cfg.pilot_manifest_path or base / "pilot" / "frozen_pilot_manifest.json")
            ),
        },
        "full schedule",
    )
    for specification in schedule:
        _execute_fit(ctx, store, deployment, **specification)
    json_path, csv_path = store.aggregate()
    return _render_stage(
        {
            "mode": "full",
            "status": {"completed": len(store.records()), "planned": 206},
            "output_dir": str(out),
            "records": store.records(),
            "context": {"pilot": pilot, "test_pixels_opened": True},
            "json_path": str(json_path),
            "csv_path": str(csv_path),
        },
        out,
    )


def render_report(
    result: Mapping[str, Any] | str | Path,
    output_dir: str | Path | None = None,
) -> dict[str, Any]:
    if isinstance(result, (str, Path)):
        payload = json.loads(Path(result).read_text(encoding="utf-8"))
    else:
        payload = dict(result)
    out = Path(payload.get("output_dir") or output_dir or ".")
    return _render_stage(payload, out)


def run_experiment(
    config: Mapping[str, Any] | ExperimentConfig,
    output_dir: str | Path | None = None,
) -> dict[str, Any]:
    cfg = _coerce_config(config)
    base = Path(output_dir or cfg.output_dir)
    if cfg.mode == "smoke":
        return run_smoke(cfg, base)
    if cfg.mode == "pilot":
        return run_validation_pilot(cfg, base)
    if cfg.mode == "full":
        return run_full_after_review(cfg, base)
    raise ValueError(cfg.mode)


def run_pipeline(base_dir: str | Path, **overrides: Any) -> dict[str, Any]:
    """Run validation-only smoke and pilot, then deliberately stop."""

    smoke = run_smoke(ExperimentConfig.for_mode("smoke", **overrides), base_dir)
    pilot = run_validation_pilot(ExperimentConfig.for_mode("pilot", **overrides), base_dir)
    frozen = bool(pilot.get("context", {}).get("pilot_manifest", {}).get("calibration_passed"))
    return {
        "mode": "pipeline",
        "smoke": smoke,
        "pilot": pilot,
        "status": "stopped-after-frozen-pilot" if frozen else "stopped-after-diagnostic-pilot",
        "full_executed": False,
        "next_action": "Review adequacy, bounds, grids, and plots before explicitly calling run_full_after_review.",
    }


def run_public_svd_static_self_tests() -> dict[str, Any]:
    cfg = ExperimentConfig.for_mode("smoke", n_participants=10)
    synthetic = []
    for identity in range(MAX_PARTICIPANTS * IDENTITIES_PER_PARTICIPANT):
        count = 1 + (identity % 7)
        labels = tuple((identity + offset) % 2 for offset in range(count))
        synthetic.append(
            IdentityStat(
                identity=identity,
                reserved_filename=f"r{identity}.jpg",
                protocol_filenames=tuple(f"p{identity}_{offset}.jpg" for offset in range(count)),
                protocol_labels=labels,
                protocol_sources=tuple(range(count)),
            )
        )
    buckets = _balanced_identity_allocation(synthetic, cfg)
    nested_buckets = _balanced_identity_allocation(
        synthetic, replace(cfg, n_participants=25)
    )
    exact_counts = all(
        len(buckets[(participant, split)]) == SPLIT_IDENTITY_COUNTS[split]
        for participant in range(MAX_PARTICIPANTS)
        for split in SPLIT_IDENTITY_COUNTS
    )
    identity_values = [
        item.identity for values in buckets.values() for item in values
    ]
    globally_disjoint = len(identity_values) == len(set(identity_values))
    nested_master_invariant = all(
        [item.identity for item in buckets[key]]
        == [item.identity for item in nested_buckets[key]]
        for key in buckets
    )
    rejected_51 = False
    try:
        ExperimentConfig(n_participants=51)
    except ValueError:
        rejected_51 = True
    report = {
        "master_capacity_participants": MAX_PARTICIPANTS,
        "exact_70_15_15_counts": exact_counts,
        "globally_disjoint": globally_disjoint,
        "nested_master_invariant_across_active_p": nested_master_invariant,
        "rejects_more_than_50": rejected_51,
        "public_svd_images_fixed_10000": cfg.public_svd_images == PUBLIC_SVD_IMAGES,
        "shared_data_cache_schema_is_v1": DATA_CACHE_SCHEMA_VERSION == 1,
        "pipeline_stops_before_full": "run_full_after_review" not in (
            run_pipeline.__doc__ or ""
        ),
    }
    report["all_passed"] = all(bool(value) for key, value in report.items() if key != "master_capacity_participants")
    if not report["all_passed"]:
        raise AssertionError(report)
    return report


__all__ = [
    "ExperimentConfig",
    "make_config",
    "print_config",
    "prepare_celeba_data",
    "run_public_svd_static_self_tests",
    "run_smoke",
    "run_validation_pilot",
    "run_full_after_review",
    "run_pipeline",
    "run_experiment",
    "render_report",
]


In [ ]:
%%writefile celeba_public_svd_identity100_unit_isometric_runner.py
"""Unit-isometric-anchor overlay for the CelebA public-SVD experiment.

The reviewed data preparation, models, attacks, and checkpointing remain in
``celeba_public_svd_identity100_runner``.  This versioned overlay changes only
the anchor experiment: it freezes a centered anchor with orthonormal columns,
uses the anchor-noise standard deviation ``v`` directly, and records enough
anchor and timing metadata to make a cross-runtime resume fail closed.
"""

from __future__ import annotations

import hashlib
import inspect
import json
import math
import os
import sys
import time
from dataclasses import dataclass, replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping, Optional, Sequence

import torch

import celeba_public_svd_identity100_runner as _base


SCHEMA_VERSION = 2
ANCHOR_DESIGN = "centered_unit_isometric_v2"
ANCHOR_SCALE_RULE = "unit_singular_values"
ANCHOR_NOISE_PARAMETERIZATION = "absolute_v"

_BASE_CONFIG = getattr(_base, "_unit_anchor_base_config", _base.ExperimentConfig)
_ORIGINAL_DIAG = getattr(_base, "_unit_anchor_original_diag", _base._diag)
_ORIGINAL_EXECUTE_FIT = getattr(
    _base, "_unit_anchor_original_execute_fit", _base._execute_fit
)
_ORIGINAL_LOAD_FROZEN_PILOT = getattr(
    _base, "_unit_anchor_original_load_frozen_pilot", _base._load_frozen_pilot
)
_ORIGINAL_RENDER_STAGE = getattr(
    _base, "_unit_anchor_original_render_stage", _base._render_stage
)
_ORIGINAL_STATIC_TESTS = getattr(
    _base,
    "_unit_anchor_original_static_tests",
    _base.run_public_svd_static_self_tests,
)

_base._unit_anchor_base_config = _BASE_CONFIG
_base._unit_anchor_original_diag = _ORIGINAL_DIAG
_base._unit_anchor_original_execute_fit = _ORIGINAL_EXECUTE_FIT
_base._unit_anchor_original_load_frozen_pilot = _ORIGINAL_LOAD_FROZEN_PILOT
_base._unit_anchor_original_render_stage = _ORIGINAL_RENDER_STAGE
_base._unit_anchor_original_static_tests = _ORIGINAL_STATIC_TESTS


PRIVATE_PILOT_LEVELS = (
    0.0,
    2.0**-8,
    2.0**-7,
    2.0**-6,
    2.0**-5,
    2.0**-4,
    2.0**-3,
    2.0**-2,
    2.0**-1,
    1.0,
    2.0,
    4.0,
    8.0,
    16.0,
)

# The old Gaussian anchor had sigma_min about 20 and selected v about 4.  Its
# unit-isometric scale analogue is therefore around 0.2.  The denser local grid
# brackets that region while retaining room for a data-driven upper endpoint.
ANCHOR_PILOT_VS = (
    0.0,
    2.0**-9,
    2.0**-8,
    2.0**-7,
    2.0**-6,
    2.0**-5,
    3.0 / 64.0,
    2.0**-4,
    3.0 / 32.0,
    2.0**-3,
    3.0 / 16.0,
    2.0**-2,
    3.0 / 8.0,
    2.0**-1,
    0.75,
    1.0,
    2.0,
)


@dataclass(frozen=True)
class ExperimentConfig(_BASE_CONFIG):
    output_dir: str = (
        "/content/drive/MyDrive/I-GDP-PA/"
        "CelebA_PublicSVD_h400_identity100_unit_isometric_anchor_results_v2"
    )
    local_cache_dir: str = "/content/igdp_pa_celeba_public_svd_unit_anchor"
    anchor_alpha: float = 1.0
    anchor_design: str = ANCHOR_DESIGN
    anchor_scale_rule: str = ANCHOR_SCALE_RULE
    anchor_noise_parameterization: str = ANCHOR_NOISE_PARAMETERIZATION
    anchor_v_upper: Optional[float] = None
    pilot_private_candidate_lambdas: tuple[float, ...] = PRIVATE_PILOT_LEVELS
    pilot_anchor_candidate_vs: tuple[float, ...] = ANCHOR_PILOT_VS

    def __post_init__(self) -> None:
        super().__post_init__()
        if self.anchor_rows - 1 < self.reduced_dim:
            raise ValueError("A centered r-by-h anchor requires r-1 >= h")
        if self.anchor_alpha != 1.0:
            raise ValueError("This experiment fixes anchor_alpha=1")
        if self.anchor_design != ANCHOR_DESIGN:
            raise ValueError(f"anchor_design must be {ANCHOR_DESIGN}")
        if self.anchor_scale_rule != ANCHOR_SCALE_RULE:
            raise ValueError(f"anchor_scale_rule must be {ANCHOR_SCALE_RULE}")
        if self.anchor_noise_parameterization != ANCHOR_NOISE_PARAMETERIZATION:
            raise ValueError("Anchor noise must be parameterized directly by v")
        if self.mode == "pilot" and self.anchor_v_upper is not None:
            raise ValueError("Pilot anchor_v_upper must be selected, not supplied")
        if not self.pilot_private_candidate_lambdas:
            raise ValueError("Private-data pilot grid cannot be empty")
        if not self.pilot_anchor_candidate_vs:
            raise ValueError("Anchor-noise pilot grid cannot be empty")
        if self.pilot_private_candidate_lambdas[0] != 0.0:
            raise ValueError("Private-data pilot lower bound must be zero")
        if self.pilot_anchor_candidate_vs[0] != 0.0:
            raise ValueError("Anchor-noise pilot lower bound must be zero")


def _anchor_tensor_sha256(anchor: torch.Tensor) -> str:
    array = anchor.detach().cpu().to(torch.float64).contiguous().numpy()
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()


def _tensor_sequence_sha256(tensors: Sequence[torch.Tensor]) -> str:
    digest = hashlib.sha256()
    for tensor in tensors:
        array = tensor.detach().cpu().to(torch.float64).contiguous().numpy()
        digest.update(str(tuple(array.shape)).encode("ascii"))
        digest.update(array.tobytes(order="C"))
    return digest.hexdigest()


def _anchor_diagnostics(
    anchor: torch.Tensor, cfg: ExperimentConfig, outer: int
) -> dict[str, Any]:
    cpu = anchor.detach().cpu().to(torch.float64).contiguous()
    h = cfg.reduced_dim
    identity = torch.eye(h, dtype=torch.float64)
    gram = cpu.mT @ cpu
    eigenvalues = torch.linalg.eigvalsh(gram).clamp_min(0.0)
    singular = torch.sqrt(eigenvalues)
    sigma_min = float(singular[0])
    sigma_max = float(singular[-1])
    diagnostics = {
        "design": cfg.anchor_design,
        "scale_rule": cfg.anchor_scale_rule,
        "noise_parameterization": cfg.anchor_noise_parameterization,
        "alpha": float(cfg.anchor_alpha),
        "seed": int(
            _base.named_seed(cfg.seed, "centered_unit_isometric_anchor", outer)
        ),
        "outer": int(outer),
        "rows": int(cfg.anchor_rows),
        "dimension": int(h),
        "mean_residual_l2": float(torch.linalg.vector_norm(cpu.mean(0))),
        "gram_relative_error": float(
            torch.linalg.matrix_norm(gram - identity) / math.sqrt(h)
        ),
        "sigma_min": sigma_min,
        "sigma_max": sigma_max,
        "condition_number": sigma_max / max(sigma_min, 1.0e-15),
        "entrywise_rms": float(torch.sqrt(cpu.square().mean())),
        "tensor_sha256_float64_c_order": _anchor_tensor_sha256(cpu),
    }
    if (
        diagnostics["mean_residual_l2"] > 1.0e-10
        or diagnostics["gram_relative_error"] > 1.0e-10
        or abs(sigma_min - 1.0) > 1.0e-10
        or abs(sigma_max - 1.0) > 1.0e-10
    ):
        raise RuntimeError(f"Unit-isometric anchor validation failed: {diagnostics}")
    return diagnostics


def _make_centered_unit_isometric_anchor(
    cfg: ExperimentConfig, outer: int = 0
) -> tuple[torch.Tensor, dict[str, Any]]:
    generator = torch.Generator(device="cpu").manual_seed(
        _base.named_seed(cfg.seed, "centered_unit_isometric_anchor", outer)
    )
    raw = torch.randn(
        (cfg.anchor_rows, cfg.reduced_dim),
        generator=generator,
        dtype=torch.float64,
    )
    centered = raw - raw.mean(dim=0, keepdim=True)
    q_factor, r_factor = torch.linalg.qr(centered, mode="reduced")
    diagonal = torch.diagonal(r_factor)
    signs = torch.where(diagonal < 0.0, -torch.ones_like(diagonal), torch.ones_like(diagonal))
    anchor = (q_factor * signs.unsqueeze(0)).contiguous()
    return anchor, _anchor_diagnostics(anchor, cfg, outer)


def _persist_anchor(
    anchor: torch.Tensor,
    diagnostics: Mapping[str, Any],
    directory: Path,
    cfg: ExperimentConfig,
) -> dict[str, Any]:
    anchor_dir = directory / "anchors"
    anchor_dir.mkdir(parents=True, exist_ok=True)
    tensor_path = anchor_dir / f"anchor_outer{int(diagnostics['outer'])}.pt"
    if not tensor_path.exists():
        temporary = tensor_path.with_suffix(".tmp.pt")
        torch.save(
            {
                "schema_version": SCHEMA_VERSION,
                "anchor": anchor.detach().cpu().to(torch.float64),
                "diagnostics": dict(diagnostics),
            },
            temporary,
        )
        os.replace(temporary, tensor_path)
    payload = torch.load(tensor_path, map_location="cpu", weights_only=False)
    stored = payload["anchor"].to(torch.float64)
    verified = _anchor_diagnostics(stored, cfg, int(diagnostics["outer"]))
    if verified["tensor_sha256_float64_c_order"] != diagnostics[
        "tensor_sha256_float64_c_order"
    ]:
        raise RuntimeError("Persisted anchor tensor differs from the generated anchor")
    enriched = dict(diagnostics)
    enriched.update(
        {
            "tensor_path": str(tensor_path),
            "tensor_file_sha256": _base.file_sha256(tensor_path),
        }
    )
    _base._atomic_json(anchor_dir / "anchor_manifest.json", enriched)
    return enriched


def _gdp_parameter_diagnostics(
    rotations: Sequence[torch.Tensor],
    translations: Sequence[torch.Tensor],
    cfg: ExperimentConfig,
    outer: int,
) -> dict[str, Any]:
    cpu_rotations = [item.detach().cpu().to(torch.float64).contiguous() for item in rotations]
    cpu_translations = [
        item.detach().cpu().to(torch.float64).contiguous() for item in translations
    ]
    if len(cpu_rotations) != cfg.n_participants or len(cpu_translations) != cfg.n_participants:
        raise RuntimeError("GDP parameter count differs from the participant count")
    identity = torch.eye(cfg.reduced_dim, dtype=torch.float64)
    orthogonality = max(
        float(
            torch.linalg.matrix_norm(rotation.mT @ rotation - identity)
            / math.sqrt(cfg.reduced_dim)
        )
        for rotation in cpu_rotations
    )
    if any(tuple(item.shape) != (cfg.reduced_dim, cfg.reduced_dim) for item in cpu_rotations):
        raise RuntimeError("GDP rotation shape mismatch")
    if any(tuple(item.shape) != (cfg.reduced_dim,) for item in cpu_translations):
        raise RuntimeError("GDP translation shape mismatch")
    if orthogonality > 1.0e-10:
        raise RuntimeError(f"GDP rotation orthogonality check failed: {orthogonality}")
    return {
        "design": "frozen_cpu_float64_gdp_parameters_v1",
        "seed": int(_base.named_seed(cfg.seed, "gdp-parameters", outer)),
        "outer": int(outer),
        "participants": int(cfg.n_participants),
        "dimension": int(cfg.reduced_dim),
        "max_rotation_orthogonality_error": orthogonality,
        "rotations_sha256_float64_c_order": _tensor_sequence_sha256(cpu_rotations),
        "translations_sha256_float64_c_order": _tensor_sequence_sha256(cpu_translations),
    }


def _persist_gdp_parameters(
    rotations: Sequence[torch.Tensor],
    translations: Sequence[torch.Tensor],
    diagnostics: Mapping[str, Any],
    directory: Path,
    cfg: ExperimentConfig,
) -> dict[str, Any]:
    deployment_dir = directory / "deployment"
    deployment_dir.mkdir(parents=True, exist_ok=True)
    tensor_path = deployment_dir / f"gdp_parameters_outer{int(diagnostics['outer'])}.pt"
    if not tensor_path.exists():
        temporary = tensor_path.with_suffix(".tmp.pt")
        torch.save(
            {
                "schema_version": SCHEMA_VERSION,
                "rotations": [item.detach().cpu().to(torch.float64) for item in rotations],
                "translations": [
                    item.detach().cpu().to(torch.float64) for item in translations
                ],
                "diagnostics": dict(diagnostics),
            },
            temporary,
        )
        os.replace(temporary, tensor_path)
    payload = torch.load(tensor_path, map_location="cpu", weights_only=False)
    verified = _gdp_parameter_diagnostics(
        payload["rotations"], payload["translations"], cfg, int(diagnostics["outer"])
    )
    for key in (
        "design",
        "seed",
        "outer",
        "participants",
        "dimension",
        "rotations_sha256_float64_c_order",
        "translations_sha256_float64_c_order",
    ):
        if verified[key] != diagnostics[key]:
            raise RuntimeError(f"Persisted GDP deployment differs at {key}")
    enriched = {
        **dict(diagnostics),
        "tensor_path": str(tensor_path),
        "tensor_file_sha256": _base.file_sha256(tensor_path),
    }
    _base._atomic_json(deployment_dir / "gdp_parameters_manifest.json", enriched)
    return enriched


def _validate_loaded_gdp_parameters(
    rotations: Sequence[torch.Tensor],
    translations: Sequence[torch.Tensor],
    cfg: ExperimentConfig,
    outer: int,
    expected: Mapping[str, Any],
) -> dict[str, Any]:
    actual = _gdp_parameter_diagnostics(rotations, translations, cfg, outer)
    required = (
        "design",
        "seed",
        "outer",
        "participants",
        "dimension",
        "rotations_sha256_float64_c_order",
        "translations_sha256_float64_c_order",
    )
    mismatches = {
        key: {"expected": expected.get(key), "actual": actual.get(key)}
        for key in required
        if expected.get(key) != actual.get(key)
    }
    if mismatches:
        raise RuntimeError(f"Frozen GDP deployment mismatch: {mismatches}")
    return {
        **actual,
        **{
            key: expected[key]
            for key in ("tensor_path", "tensor_file_sha256")
        },
    }


def _validate_loaded_anchor(
    anchor: torch.Tensor,
    cfg: ExperimentConfig,
    outer: int,
    expected: Mapping[str, Any],
) -> dict[str, Any]:
    actual = _anchor_diagnostics(anchor, cfg, outer)
    required = (
        "design",
        "scale_rule",
        "noise_parameterization",
        "alpha",
        "seed",
        "outer",
        "rows",
        "dimension",
        "tensor_sha256_float64_c_order",
    )
    mismatches = {
        key: {"expected": expected.get(key), "actual": actual.get(key)}
        for key in required
        if expected.get(key) != actual.get(key)
    }
    if mismatches:
        raise RuntimeError(f"Frozen pilot anchor mismatch: {mismatches}")
    return {**actual, **{key: expected[key] for key in ("tensor_path", "tensor_file_sha256")}}


def _pilot_manifest_path(cfg: ExperimentConfig) -> Path:
    if cfg.pilot_manifest_path:
        return Path(cfg.pilot_manifest_path)
    output = Path(cfg.output_dir)
    base = output.parent if output.name in {"smoke", "pilot", "full"} else output
    return base / "pilot" / "frozen_pilot_manifest.json"


def _deployment(ctx: Any, outer: int = 0) -> dict[str, Any]:
    cfg = ctx.cfg
    device = _base.resolve_device(cfg.device)
    if cfg.mode == "full":
        manifest_path = _pilot_manifest_path(cfg)
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        expected = manifest.get("anchor", {})
        tensor_path = Path(str(expected.get("tensor_path", "")))
        if not tensor_path.is_file():
            raise FileNotFoundError("Frozen pilot anchor tensor is missing")
        if _base.file_sha256(tensor_path) != expected.get("tensor_file_sha256"):
            raise RuntimeError("Frozen pilot anchor file SHA-256 mismatch")
        payload = torch.load(tensor_path, map_location="cpu", weights_only=False)
        anchor_cpu = payload["anchor"].to(torch.float64).contiguous()
        anchor_spec = _validate_loaded_anchor(anchor_cpu, cfg, outer, expected)
        expected_deployment = manifest.get("deployment", {})
        deployment_path = Path(str(expected_deployment.get("tensor_path", "")))
        if not deployment_path.is_file():
            raise FileNotFoundError("Frozen pilot GDP deployment tensor is missing")
        if _base.file_sha256(deployment_path) != expected_deployment.get(
            "tensor_file_sha256"
        ):
            raise RuntimeError("Frozen pilot GDP deployment file SHA-256 mismatch")
        deployment_payload = torch.load(
            deployment_path, map_location="cpu", weights_only=False
        )
        rotations_cpu = [
            item.to(torch.float64).contiguous()
            for item in deployment_payload["rotations"]
        ]
        translations_cpu = [
            item.to(torch.float64).contiguous()
            for item in deployment_payload["translations"]
        ]
        deployment_spec = _validate_loaded_gdp_parameters(
            rotations_cpu,
            translations_cpu,
            cfg,
            outer,
            expected_deployment,
        )
    else:
        generator = _base.cp.make_generator(
            _base.named_seed(cfg.seed, "gdp-parameters", outer), "cpu"
        )
        rotations_cpu, translations_cpu = _base.cp.sample_gdp_parameters(
            cfg.n_participants,
            cfg.reduced_dim,
            generator=generator,
            device="cpu",
        )
        raw_deployment_spec = _gdp_parameter_diagnostics(
            rotations_cpu, translations_cpu, cfg, outer
        )
        deployment_spec = _persist_gdp_parameters(
            rotations_cpu,
            translations_cpu,
            raw_deployment_spec,
            Path(ctx.out),
            cfg,
        )
        anchor_cpu, raw_spec = _make_centered_unit_isometric_anchor(cfg, outer)
        anchor_spec = _persist_anchor(anchor_cpu, raw_spec, Path(ctx.out), cfg)
    return {
        "rotations": [item.to(device) for item in rotations_cpu],
        "translations": [item.to(device) for item in translations_cpu],
        "anchor": anchor_cpu.to(device),
        "private_rms": float(
            torch.sqrt(ctx.data.train.reduced.double().square().mean())
        ),
        "anchor_rms": float(anchor_spec["entrywise_rms"]),
        "anchor_spec": anchor_spec,
        "deployment_spec": deployment_spec,
    }


def _cpu_noise_tensor(
    cfg: ExperimentConfig,
    shape: Sequence[int],
    *,
    outer: int,
    draw: int,
    split: str,
    participant: int,
    family: str,
    device: torch.device,
) -> torch.Tensor:
    """Generate a device-independent standard-normal tensor and then transfer it."""

    generator = torch.Generator(device="cpu").manual_seed(
        _base.named_seed(
            cfg.seed, "noise", family, outer, draw, split, participant
        )
    )
    cpu = torch.randn(
        tuple(shape), generator=generator, device="cpu", dtype=torch.float64
    )
    return cpu.to(device)


def _build_transcripts(
    ctx: Any,
    deployment: Mapping[str, Any],
    method: str,
    outer: int,
    draw: int,
    level: float,
    *,
    multistart: bool = False,
) -> tuple[dict[str, Any], float, float]:
    cfg = ctx.cfg
    rotations = deployment["rotations"]
    translations = deployment["translations"]
    reference = cfg.reference_participant
    sigma = level * float(deployment["private_rms"]) if method == "c_gdp_an" else 0.0
    anchor_sigma = float(level) if method == "pa_i_gdp" else 0.0
    # Pilot candidates share one standard-normal path; only the amplitude varies.
    # Full random iterations retain independent paths through their draw index.
    noise_draw = 0 if cfg.mode == "pilot" else draw
    transcripts: dict[str, Any] = {}
    if method == "pa_i_gdp":
        anchor = deployment["anchor"]
        noises = [
            anchor_sigma
            * _cpu_noise_tensor(
                cfg,
                anchor.shape,
                outer=outer,
                draw=noise_draw,
                split="anchor",
                participant=participant,
                family="pa-unit-isometric",
                device=anchor.device,
            )
            for participant in range(cfg.n_participants)
        ]
        train = _base.cp.pa_i_gdp(
            ctx.blocks["train"][0],
            anchor,
            anchor_sigma,
            rotations=rotations,
            translations=translations,
            anchor_noises=noises,
            max_iters=cfg.pa_iterations,
            tol=cfg.pa_tolerance,
            n_random_starts=4 if multistart else 0,
            restart_seed=_base.named_seed(cfg.seed, "gpm-restart", outer, draw),
        )
        transcripts["train"] = train
        for split in ("val", "test"):
            transcripts[split] = _base._fixed_pa(ctx.blocks[split][0], train)
        return transcripts, sigma, anchor_sigma

    for split in ("train", "val", "test"):
        xs = ctx.blocks[split][0]
        if method == "c_gdp":
            transcripts[split] = _base.cp.c_gdp(
                xs,
                rotation=rotations[reference],
                translation=translations[reference],
            )
        elif method == "c_gdp_an":
            noises = [
                sigma
                * _cpu_noise_tensor(
                    cfg,
                    x.shape,
                    outer=outer,
                    draw=noise_draw,
                    split=split,
                    participant=participant,
                    family="c-gdp-private-noise",
                    device=x.device,
                )
                for participant, x in enumerate(xs)
            ]
            transcripts[split] = _base.cp.c_gdp_an(
                xs,
                sigma,
                rotation=rotations[reference],
                translation=translations[reference],
                noises=noises,
            )
        elif method == "i_gdp":
            transcripts[split] = _base.cp.i_gdp(
                xs, rotations=rotations, translations=translations
            )
        else:
            raise KeyError(method)
    return transcripts, sigma, anchor_sigma


def _diag(result: Any, cfg: ExperimentConfig, _anchor_rms: float) -> Optional[dict[str, Any]]:
    # Passing one prevents the legacy normalization by entrywise anchor RMS.
    diagnostics = _ORIGINAL_DIAG(result, cfg, 1.0)
    if diagnostics is None:
        return None
    empirical_v = float(diagnostics["empirical_phase_transition_lambda"])
    diagnostics.update(
        {
            "ling_empirical_v_pt": empirical_v,
            "empirical_phase_transition_v": empirical_v,
            "empirical_phase_transition_lambda": empirical_v,
            "empirical_phase_transition_parameterization": "absolute_v",
            "empirical_phase_transition_status": (
                "Ling empirical scaling specialized to the centered unit-isometric "
                "anchor; heuristic, not theorem"
            ),
        }
    )
    return diagnostics


def _execute_fit(*args: Any, **kwargs: Any) -> dict[str, Any]:
    deployment = args[2] if len(args) >= 3 else kwargs["deployment"]
    method = str(kwargs["method"])
    level = float(kwargs["level"])
    store = args[1] if len(args) >= 2 else kwargs["store"]
    record = dict(_ORIGINAL_EXECUTE_FIT(*args, **kwargs))
    record.pop("schema_version", None)
    anchor_v = level if method == "pa_i_gdp" else 0.0
    record.update(
        {
            "schema_version": SCHEMA_VERSION,
            "noise_level": level,
            "anchor_v": anchor_v,
            "anchor_noise_parameterization": ANCHOR_NOISE_PARAMETERIZATION,
            "noise_parameterization": (
                ANCHOR_NOISE_PARAMETERIZATION
                if method == "pa_i_gdp"
                else (
                    "relative_to_reduced_train_rms"
                    if method == "c_gdp_an"
                    else "none"
                )
            ),
            "noise_rng": "torch_cpu_standard_normal_seeded_v1",
            "anchor": dict(deployment["anchor_spec"]),
            "deployment": dict(deployment["deployment_spec"]),
        }
    )
    if method == "pa_i_gdp":
        if not math.isclose(
            float(record["anchor_noise_sigma"]), anchor_v, rel_tol=0.0, abs_tol=1.0e-15
        ):
            raise AssertionError("Recorded anchor-noise sigma is not the supplied absolute v")
        if record.get("gpm"):
            boundary = float(record["gpm"]["ling_empirical_v_pt"])
            record["gpm"]["v_over_ling_empirical_v_pt"] = anchor_v / max(
                boundary, 1.0e-15
            )
    _base._atomic_json(store.path(str(record["run_id"])), record)
    return record


def _validate_frozen_anchor_manifest(
    cfg: ExperimentConfig, manifest: Mapping[str, Any]
) -> None:
    if manifest.get("anchor_noise_parameterization") != ANCHOR_NOISE_PARAMETERIZATION:
        raise RuntimeError("Frozen pilot does not use absolute anchor-noise v")
    if manifest.get("anchor_design") != ANCHOR_DESIGN:
        raise RuntimeError("Frozen pilot does not use the unit-isometric anchor design")
    expected = manifest.get("anchor", {})
    tensor_path = Path(str(expected.get("tensor_path", "")))
    if not tensor_path.is_file():
        raise FileNotFoundError("Frozen pilot anchor tensor is unavailable")
    if _base.file_sha256(tensor_path) != expected.get("tensor_file_sha256"):
        raise RuntimeError("Frozen pilot anchor tensor file hash differs")
    payload = torch.load(tensor_path, map_location="cpu", weights_only=False)
    _validate_loaded_anchor(payload["anchor"], cfg, 0, expected)
    expected_deployment = manifest.get("deployment", {})
    deployment_path = Path(str(expected_deployment.get("tensor_path", "")))
    if not deployment_path.is_file():
        raise FileNotFoundError("Frozen pilot GDP deployment tensor is unavailable")
    if _base.file_sha256(deployment_path) != expected_deployment.get(
        "tensor_file_sha256"
    ):
        raise RuntimeError("Frozen pilot GDP deployment tensor file hash differs")
    deployment_payload = torch.load(
        deployment_path, map_location="cpu", weights_only=False
    )
    _validate_loaded_gdp_parameters(
        deployment_payload["rotations"],
        deployment_payload["translations"],
        cfg,
        0,
        expected_deployment,
    )


def _load_frozen_pilot(cfg: ExperimentConfig, base: Path) -> dict[str, Any]:
    manifest = _ORIGINAL_LOAD_FROZEN_PILOT(cfg, base)
    _validate_frozen_anchor_manifest(cfg, manifest)
    return manifest


def _rewrite_unit_anchor_plots(
    payload: Mapping[str, Any], rendered: Mapping[str, Any]
) -> None:
    import matplotlib.pyplot as plt

    records = list(payload.get("records", []))
    noisy = [row for row in records if row.get("protocol") in {"c_gdp_an", "pa_i_gdp"}]
    utility_path = Path(str(rendered["utility_plot"]))
    figure, axes = plt.subplots(1, 2, figsize=(10.8, 4.25), sharey=True)
    panels = (
        ("c_gdp_an", "C-GDP (private-data noise)", r"Private-data level $\lambda_x$"),
        ("pa_i_gdp", "AA-I-GDP (anchor noise)", r"Anchor-noise s.d. $v$"),
    )
    for axis, (method, title, xlabel) in zip(axes, panels):
        rows = sorted(
            [row for row in noisy if row.get("protocol") == method],
            key=lambda row: float(row["noise_level"]),
        )
        if rows:
            axis.scatter(
                [row["noise_level"] for row in rows],
                [row["utility"]["validation_balanced_accuracy"] for row in rows],
                s=22,
                alpha=0.75,
            )
        axis.set(xlabel=xlabel, title=title)
        axis.grid(alpha=0.25)
    axes[0].set_ylabel("Validation balanced accuracy")
    figure.suptitle("CelebA validation utility response")
    figure.tight_layout()
    temporary = utility_path.with_suffix(".tmp.png")
    figure.savefig(temporary, dpi=180)
    plt.close(figure)
    os.replace(temporary, utility_path)

    privacy_path = Path(str(rendered["privacy_utility_plot"]))
    figure, axis = plt.subplots(figsize=(6.4, 4.8))
    for method, label in (
        ("c_gdp_an", "C-GDP (private-data noise)"),
        ("pa_i_gdp", "AA-I-GDP (anchor noise)"),
    ):
        rows = [
            row
            for row in noisy
            if row.get("protocol") == method and row.get("best_attack")
        ]
        if rows:
            axis.scatter(
                [
                    row["best_attack"]["metrics"]["face_linkage"]
                    ["cross_image_identity"]["top1"]
                    for row in rows
                ],
                [row["utility"]["validation_balanced_accuracy"] for row in rows],
                s=22,
                alpha=0.75,
                label=label,
            )
    axis.axvline(_base.LINKAGE_CHANCE, color="black", linestyle="--", linewidth=1)
    axis.set(
        xlabel="Reserved-reference cross-image top-1",
        ylabel="Validation balanced accuracy",
        title="Privacy--utility observations",
    )
    axis.grid(alpha=0.25)
    if noisy:
        axis.legend()
    figure.tight_layout()
    temporary = privacy_path.with_suffix(".tmp.png")
    figure.savefig(temporary, dpi=180)
    plt.close(figure)
    os.replace(temporary, privacy_path)

    gpm_path = Path(str(rendered["gpm_diagnostic_plot"]))
    pa_rows = [
        row for row in records if row.get("protocol") == "pa_i_gdp" and row.get("gpm")
    ]
    figure, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
    if pa_rows:
        normalized = [
            float(row["anchor_v"])
            / max(float(row["gpm"]["ling_empirical_v_pt"]), 1.0e-15)
            for row in pa_rows
        ]
        axes[0].scatter(normalized, [row["gpm"]["iterations"] for row in pa_rows], s=18)
        axes[1].scatter(
            normalized,
            [row["gpm"]["fixed_point_residual"] for row in pa_rows],
            s=18,
        )
        axes[1].set_yscale("log")
    axes[0].set(xlabel=r"$v/v_{\mathrm{emp}}$", ylabel="GPM iterations")
    axes[1].set(xlabel=r"$v/v_{\mathrm{emp}}$", ylabel="Fixed-point residual")
    for axis in axes:
        axis.axvline(1.0, color="black", linestyle="--", linewidth=1)
        axis.grid(alpha=0.25)
    figure.tight_layout()
    temporary = gpm_path.with_suffix(".tmp.png")
    figure.savefig(temporary, dpi=180)
    plt.close(figure)
    os.replace(temporary, gpm_path)


def _render_stage(payload: Mapping[str, Any], out: Path) -> dict[str, Any]:
    rendered = _ORIGINAL_RENDER_STAGE(payload, out)
    _rewrite_unit_anchor_plots(payload, rendered)
    return rendered


def _utc_now() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _runtime_environment_lock(
    config: Mapping[str, Any] | ExperimentConfig,
) -> dict[str, Any]:
    cfg = _base._coerce_config(config)
    resolved = _base.resolve_device(cfg.device)
    cuda = resolved.type == "cuda"
    return {
        "schema_version": SCHEMA_VERSION,
        "python_major_minor": f"{sys.version_info.major}.{sys.version_info.minor}",
        "torch": torch.__version__,
        "resolved_device_type": resolved.type,
        "gpu": torch.cuda.get_device_name(0) if cuda else None,
        "cuda": torch.version.cuda if cuda else None,
        "cudnn": torch.backends.cudnn.version() if cuda else None,
        "noise_rng": "torch_cpu_standard_normal_seeded_v1",
        "gdp_parameter_generation": "frozen_cpu_float64_gdp_parameters_v1",
    }


def _run_timed_stage(
    mode: str,
    config: Mapping[str, Any] | ExperimentConfig,
    output_dir: str | Path,
    operation: Any,
) -> dict[str, Any]:
    stage_dir = Path(output_dir) / mode
    stage_dir.mkdir(parents=True, exist_ok=True)
    _base._write_or_validate_json(
        stage_dir / "runtime_environment_lock.json",
        _runtime_environment_lock(config),
        "runtime environment lock",
    )
    path = stage_dir / "stage_runtime.json"
    existing = json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {}
    started_at = _utc_now()
    timing = {
        "schema_version": SCHEMA_VERSION,
        "mode": mode,
        "first_started_at_utc": existing.get("first_started_at_utc", started_at),
        "latest_attempt_started_at_utc": started_at,
        "latest_attempt_completed_at_utc": None,
        "latest_attempt_wall_seconds": None,
        "completed": False,
        "attempts": int(existing.get("attempts", 0)) + 1,
    }
    _base._atomic_json(path, timing)
    tick = time.perf_counter()
    try:
        result = operation(config, output_dir)
    except Exception as exc:
        timing.update(
            {
                "latest_attempt_completed_at_utc": _utc_now(),
                "latest_attempt_wall_seconds": time.perf_counter() - tick,
                "last_error": f"{type(exc).__name__}: {exc}",
            }
        )
        _base._atomic_json(path, timing)
        raise
    records = list(result.get("records", []))
    timing.update(
        {
            "latest_attempt_completed_at_utc": _utc_now(),
            "latest_attempt_wall_seconds": time.perf_counter() - tick,
            "summed_fit_runtime_seconds": float(
                sum(float(row.get("runtime_seconds", 0.0)) for row in records)
            ),
            "completed_records": len(records),
            "completed": True,
        }
    )
    _base._atomic_json(path, timing)
    result["stage_runtime_path"] = str(path)
    result["stage_runtime"] = timing
    return result


def _run_validation_pilot_core(
    config: Mapping[str, Any] | ExperimentConfig,
    output_dir: str | Path,
) -> dict[str, Any]:
    cfg, out = _base._stage_config(config, output_dir, "pilot")
    ctx = _base._make_context(cfg, out, include_test=False)
    if ctx.data.manifest["test_pixels_opened"]:
        raise AssertionError("Pilot opened test pixels")
    store = _base.AtomicRunStore(out, stem="pilot_results")
    svd_utility = _base._svd_utility_control(ctx)
    deployment = _deployment(ctx)
    _execute_fit(
        ctx,
        store,
        deployment,
        method="c_gdp",
        outer=0,
        draw=0,
        run_role="pilot-clean-adequacy",
        level=0.0,
    )
    status: dict[str, Any] = {}
    grids: dict[str, Sequence[float]] = {
        "c_gdp_an": cfg.pilot_private_candidate_lambdas,
        "pa_i_gdp": cfg.pilot_anchor_candidate_vs,
    }
    for method in ("c_gdp_an", "pa_i_gdp"):
        for draw, level in enumerate(grids[method]):
            _execute_fit(
                ctx,
                store,
                deployment,
                method=method,
                outer=0,
                draw=draw,
                run_role="pilot",
                level=float(level),
            )
            upper, evidence, reason = _base._pilot_evidence(store.records(), method, ctx)
            status[method] = {"upper": upper, "evidence": evidence, "reason": reason}
            if upper is not None:
                break
    records = store.records()
    adequacy = _base._pilot_adequacy(ctx, records, svd_utility)
    private_upper = status["c_gdp_an"]["upper"]
    anchor_upper = status["pa_i_gdp"]["upper"]
    calibration_passed = bool(
        adequacy["adequacy_passed"]
        and private_upper is not None
        and anchor_upper is not None
    )
    manifest = {
        "schema_version": SCHEMA_VERSION,
        "stage": "validation-only-pilot",
        "calibration_passed": calibration_passed,
        "adequacy": adequacy,
        "private_lambda_lower": 0.0,
        "private_lambda_upper": private_upper,
        "anchor_v_lower": 0.0,
        "anchor_v_upper": anchor_upper,
        "anchor_lambda_upper": anchor_upper,
        "anchor_design": ANCHOR_DESIGN,
        "anchor_scale_rule": ANCHOR_SCALE_RULE,
        "anchor_noise_parameterization": ANCHOR_NOISE_PARAMETERIZATION,
        "anchor": dict(deployment["anchor_spec"]),
        "deployment": dict(deployment["deployment_spec"]),
        "selection": status,
        "pilot_noise_path": "fixed_standard_normal_per_family_amplitude_only",
        "noise_rng": "torch_cpu_standard_normal_seeded_v1",
        "test_pixels_opened": False,
        "privacy_query_splits": ["train", "val"],
        "privacy_identity_clusters": len(ctx.linkage_manifest.query_identities),
        "bootstrap_unit": "identity",
        "master_manifest_hash": ctx.data.manifest["master_manifest_hash"],
        "active_participants": cfg.n_participants,
        "projection": ctx.data.manifest["projection"],
        "data_manifest_hash": _base.stable_hash(ctx.data.manifest),
        "config_hash": _base.stable_hash(cfg.to_dict()),
        "module_sha256": dict(cfg.module_sha256),
        "record_ids": sorted(str(row["run_id"]) for row in records),
        "record_hash": _base.stable_hash(records),
    }
    manifest_path = out / (
        "frozen_pilot_manifest.json"
        if calibration_passed
        else "pilot_diagnostic_manifest.json"
    )
    _base._atomic_json(manifest_path, manifest)
    json_path, csv_path = store.aggregate()
    return _render_stage(
        {
            "mode": "pilot",
            "status": "frozen" if calibration_passed else "diagnostic-not-frozen",
            "output_dir": str(out),
            "records": records,
            "context": {
                "test_pixels_opened": False,
                "adequacy": adequacy,
                "pilot_manifest": manifest,
                "svd_utility_control": svd_utility,
                "anchor": dict(deployment["anchor_spec"]),
                "deployment": dict(deployment["deployment_spec"]),
            },
            "pilot_manifest_path": str(manifest_path),
            "json_path": str(json_path),
            "csv_path": str(csv_path),
        },
        out,
    )


def run_smoke(
    config: Mapping[str, Any] | ExperimentConfig,
    output_dir: str | Path,
) -> dict[str, Any]:
    return _run_timed_stage("smoke", config, output_dir, _base.run_smoke)


def run_validation_pilot(
    config: Mapping[str, Any] | ExperimentConfig,
    output_dir: str | Path,
) -> dict[str, Any]:
    return _run_timed_stage("pilot", config, output_dir, _run_validation_pilot_core)


def run_full_after_review(
    config: Mapping[str, Any] | ExperimentConfig,
    output_dir: str | Path,
) -> dict[str, Any]:
    incoming = _base._coerce_config(config)
    pilot = _load_frozen_pilot(incoming, Path(output_dir))
    incoming = replace(incoming, anchor_v_upper=float(pilot["anchor_v_upper"]))
    lock = {
        "schema_version": SCHEMA_VERSION,
        "pilot_manifest_path": str(_pilot_manifest_path(incoming)),
        "pilot_manifest_sha256": _base.file_sha256(_pilot_manifest_path(incoming)),
        "anchor": pilot["anchor"],
        "deployment": pilot["deployment"],
        "anchor_v_upper": pilot["anchor_v_upper"],
        "anchor_noise_parameterization": ANCHOR_NOISE_PARAMETERIZATION,
    }
    full_dir = Path(output_dir) / "full"
    full_dir.mkdir(parents=True, exist_ok=True)
    _base._write_or_validate_json(full_dir / "full_anchor_lock.json", lock, "full anchor lock")
    return _run_timed_stage("full", incoming, output_dir, _base.run_full_after_review)


def run_pipeline(base_dir: str | Path, **overrides: Any) -> dict[str, Any]:
    """Run validation-only smoke and pilot, then deliberately stop."""

    smoke = run_smoke(ExperimentConfig.for_mode("smoke", **overrides), base_dir)
    pilot = run_validation_pilot(ExperimentConfig.for_mode("pilot", **overrides), base_dir)
    frozen = bool(pilot.get("context", {}).get("pilot_manifest", {}).get("calibration_passed"))
    return {
        "mode": "pipeline",
        "smoke": smoke,
        "pilot": pilot,
        "status": "stopped-after-frozen-pilot" if frozen else "stopped-after-diagnostic-pilot",
        "full_executed": False,
        "next_action": (
            "Review adequacy, direct-v bounds, anchor hash, grids, and plots before "
            "explicitly calling run_full_after_review."
        ),
    }


def run_experiment(
    config: Mapping[str, Any] | ExperimentConfig,
    output_dir: str | Path | None = None,
) -> dict[str, Any]:
    cfg = _base._coerce_config(config)
    base_dir = Path(output_dir or cfg.output_dir)
    if cfg.mode == "smoke":
        return run_smoke(cfg, base_dir)
    if cfg.mode == "pilot":
        return run_validation_pilot(cfg, base_dir)
    if cfg.mode == "full":
        return run_full_after_review(cfg, base_dir)
    raise ValueError(cfg.mode)


def run_public_svd_static_self_tests() -> dict[str, Any]:
    base_report = _ORIGINAL_STATIC_TESTS()
    cfg = ExperimentConfig.for_mode("smoke", n_participants=10, device="cpu")
    anchor, first = _make_centered_unit_isometric_anchor(cfg, 0)
    _, second = _make_centered_unit_isometric_anchor(cfg, 0)
    checks = {
        "base_public_svd_design": bool(base_report.get("all_passed")),
        "anchor_is_centered": first["mean_residual_l2"] <= 1.0e-10,
        "anchor_columns_are_orthonormal": first["gram_relative_error"] <= 1.0e-10,
        "anchor_sigma_min_is_one": abs(first["sigma_min"] - 1.0) <= 1.0e-10,
        "anchor_sigma_max_is_one": abs(first["sigma_max"] - 1.0) <= 1.0e-10,
        "anchor_condition_is_one": abs(first["condition_number"] - 1.0) <= 1.0e-10,
        "anchor_hash_is_deterministic": (
            first["tensor_sha256_float64_c_order"]
            == second["tensor_sha256_float64_c_order"]
        ),
        "anchor_entrywise_rms_is_one_over_sqrt_r": math.isclose(
            first["entrywise_rms"],
            1.0 / math.sqrt(cfg.anchor_rows),
            rel_tol=1.0e-10,
            abs_tol=1.0e-12,
        ),
        "absolute_v_is_not_scaled_by_anchor_rms": (
            "anchor_sigma = float(level)" in inspect.getsource(_build_transcripts)
            and "level * float(deployment[\"anchor_rms\"])" not in inspect.getsource(_build_transcripts)
        ),
        "noise_is_generated_on_cpu_before_transfer": (
            'torch.Generator(device="cpu")' in inspect.getsource(_cpu_noise_tensor)
            and "return cpu.to(device)" in inspect.getsource(_cpu_noise_tensor)
        ),
        "gdp_parameters_are_frozen_from_cpu": (
            'device="cpu"' in inspect.getsource(_deployment)
            and "_persist_gdp_parameters" in inspect.getsource(_deployment)
            and "_validate_loaded_gdp_parameters" in inspect.getsource(_deployment)
        ),
        "separate_pilot_grids": (
            cfg.pilot_private_candidate_lambdas != cfg.pilot_anchor_candidate_vs
        ),
        "new_result_root": "unit_isometric_anchor_results_v2" in cfg.output_dir,
        "anchor_shape": tuple(anchor.shape) == (1600, 400),
        "shared_data_cache_schema_is_v1": _base.DATA_CACHE_SCHEMA_VERSION == 1,
        "unit_result_schema_is_v2": _base.SCHEMA_VERSION == SCHEMA_VERSION == 2,
    }
    report = {
        "base": base_report,
        "unit_isometric_anchor": first,
        **checks,
    }
    report["all_passed"] = all(checks.values())
    if not report["all_passed"]:
        raise AssertionError(report)
    return report


# Patch the reviewed implementation's internal global lookups.  Public entry
# points below remain versioned in this module and the old notebook is untouched.
_base.SCHEMA_VERSION = SCHEMA_VERSION
_base.ExperimentConfig = ExperimentConfig
_base._deployment = _deployment
_base._build_transcripts = _build_transcripts
_base._diag = _diag
_base._execute_fit = _execute_fit
_base._load_frozen_pilot = _load_frozen_pilot
_base._render_stage = _render_stage

make_config = _base.make_config
print_config = _base.print_config
prepare_celeba_data = _base.prepare_celeba_data
render_report = _base.render_report
resolve_device = _base.resolve_device


__all__ = [
    "ExperimentConfig",
    "make_config",
    "print_config",
    "prepare_celeba_data",
    "run_public_svd_static_self_tests",
    "run_smoke",
    "run_validation_pilot",
    "run_full_after_review",
    "run_pipeline",
    "run_experiment",
    "render_report",
    "resolve_device",
]


In [ ]:
%%writefile celeba_public_svd_identity100_p_sweep_runner.py
"""Nested participant-count sweep for the CelebA unit-isometric experiment.

This overlay keeps the reviewed data, model, attack, and protocol code intact
while adding three pieces of experiment orchestration:

* one globally identity-disjoint master allocation for 90 participants;
* validation-only calibration of the anchor-noise sensitivity interval at p=10;
* resumable full runs for p in {10, 30, 50, 70, 90}, including a separate
  v=sqrt(r)=40 I-GDP-equivalence stress control.

The shared anchor and GDP parameter bank are generated once on CPU in float64.
Every active-p deployment is a strict prefix of that bank.  Scalar noise draws
and participant noise tensors are likewise prefix-invariant across p.
"""

from __future__ import annotations

import csv
import gc
import json
import math
import os
import tempfile
import time
from dataclasses import dataclass, replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping, Optional, Sequence

import torch

import celeba_public_svd_identity100_runner as _base
import celeba_public_svd_identity100_unit_isometric_runner as _unit


SCHEMA_VERSION = 3
MAX_PARTICIPANTS = 90
P_SWEEP = (10, 30, 50, 70, 90)
PRIVATE_LAMBDA_UPPER = 0.5
ANCHOR_CALIBRATION_LEVELS = (0.0, 0.0625, 0.125, 0.25, 0.5, 1.0, 2.0)
ANCHOR_EQUIVALENCE_V = 40.0
RANDOM_DRAWS_PER_FAMILY = 100
CALIBRATION_RULE_VERSION = "i-gdp-linkage-saturation-right-endpoint-v2"
LEGACY_CALIBRATION_RULE_VERSIONS = ("i-gdp-saturation-pair-v1",)
SWEEP_MODULE_FILENAME = "celeba_public_svd_identity100_p_sweep_runner.py"
LEGACY_SWEEP_MODULE_SHA256 = (
    "3fc7a414ae8b7eb334412ff05069338397aaf85936809ea4a47af48cdb4a85de"
)
SCHEDULE_VERSION = "p-sweep-207-fits-v1"

LINKAGE_TO_I_TOLERANCE = 0.03
MINIMUM_SELECTABLE_V = 0.125
MINIMUM_USABLE_BALANCED_ACCURACY = 0.50


# The base implementation intentionally hard-locked its original 50-user
# master.  This versioned overlay changes that single capacity constant before
# any configuration or data context is created.
_base.MAX_PARTICIPANTS = MAX_PARTICIPANTS
_base.SCHEMA_VERSION = SCHEMA_VERSION
_unit.SCHEMA_VERSION = SCHEMA_VERSION


@dataclass(frozen=True)
class ExperimentConfig(_unit.ExperimentConfig):
    output_dir: str = (
        "/content/drive/MyDrive/I-GDP-PA/"
        "CelebA_PublicSVD_h400_identity100_p10to90_anchor_sweep_v3"
    )
    local_cache_dir: str = "/content/igdp_pa_celeba_p10to90_anchor_sweep_v3"
    max_participants: int = MAX_PARTICIPANTS

    def __post_init__(self) -> None:
        super().__post_init__()
        if self.max_participants != MAX_PARTICIPANTS:
            raise ValueError("This sweep fixes the master capacity at 90 participants")
        if self.n_participants not in P_SWEEP:
            raise ValueError(f"n_participants must be one of {P_SWEEP}")


# Make the reviewed helpers accept the versioned sweep configuration.
_base.ExperimentConfig = ExperimentConfig


def _utc_now() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _atomic_json(path: Path, payload: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    _base._atomic_json(path, dict(payload))


def _runtime_environment_lock(
    cfg: ExperimentConfig, out: Path
) -> dict[str, Any]:
    """Freeze the execution environment before a resumable stage starts."""

    lock = {
        **dict(_unit._runtime_environment_lock(cfg)),
        "schema_version": SCHEMA_VERSION,
        "deterministic": bool(cfg.deterministic),
        "module_sha256": dict(cfg.module_sha256),
    }
    _base._write_or_validate_json(
        out / "runtime_environment_lock.json", lock, "runtime environment lock"
    )
    return lock


def _begin_stage_runtime(
    path: Path,
    *,
    mode: str,
    planned_records: int,
    p: Optional[int] = None,
) -> tuple[dict[str, Any], float]:
    existing = json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {}
    now = _utc_now()
    runtime: dict[str, Any] = {
        "schema_version": SCHEMA_VERSION,
        "mode": mode,
        "p": p,
        "first_started_at_utc": existing.get("first_started_at_utc", now),
        "latest_attempt_started_at_utc": now,
        "latest_attempt_completed_at_utc": None,
        "latest_attempt_wall_seconds": None,
        "attempts": int(existing.get("attempts", 0)) + 1,
        "phase": "setup",
        "completed": False,
        "planned_records": int(planned_records),
        "completed_records_before_attempt": int(
            existing.get("completed_records", 0)
        ),
    }
    _atomic_json(path, runtime)
    return runtime, time.perf_counter()


def _fail_stage_runtime(
    path: Path,
    runtime: dict[str, Any],
    started: float,
    exc: BaseException,
    *,
    completed_records: int,
) -> None:
    runtime.update(
        {
            "latest_attempt_completed_at_utc": _utc_now(),
            "latest_attempt_wall_seconds": time.perf_counter() - started,
            "completed_records": int(completed_records),
            "phase": "failed",
            "last_error": f"{type(exc).__name__}: {exc}",
        }
    )
    _atomic_json(path, runtime)


def _expected_run_id(ctx: Any, specification: Mapping[str, Any]) -> str:
    identity = {
        "config": _base.stable_hash(ctx.cfg.to_dict()),
        "data": _base.stable_hash(ctx.data.manifest),
        "method": specification["method"],
        "outer": int(specification["outer"]),
        "draw": int(specification["draw"]),
        "run_role": specification["run_role"],
        "lambda": float(specification["level"]),
    }
    return _base.stable_hash(identity, 24)


def _expected_run_ids(
    ctx: Any, schedule: Sequence[Mapping[str, Any]]
) -> list[str]:
    run_ids = [_expected_run_id(ctx, item) for item in schedule]
    if len(run_ids) != len(set(run_ids)):
        raise RuntimeError("The frozen schedule contains duplicate run IDs")
    return run_ids


def _scheduled_records(
    store: Any, expected_run_ids: Sequence[str], *, require_complete: bool
) -> tuple[list[dict[str, Any]], list[str]]:
    records: list[dict[str, Any]] = []
    missing: list[str] = []
    for run_id in expected_run_ids:
        path = store.path(str(run_id))
        if not path.is_file():
            missing.append(str(run_id))
            continue
        record = json.loads(path.read_text(encoding="utf-8"))
        if str(record.get("run_id")) != str(run_id):
            raise RuntimeError(f"Run file {path} contains the wrong run ID")
        records.append(record)
    if require_complete and missing:
        raise RuntimeError(
            f"Missing {len(missing)} scheduled records; first missing ID is {missing[0]}"
        )
    return records, missing


def _aggregate_scheduled_records(
    store: Any, records: Sequence[Mapping[str, Any]]
) -> tuple[Path, Path]:
    """Aggregate only records named by the immutable stage schedule."""

    selected = [dict(row) for row in records]
    json_path = store.output_dir / f"{store.stem}.json"
    csv_path = store.output_dir / f"{store.stem}.csv"
    _atomic_json(
        json_path,
        {
            "schema_version": SCHEMA_VERSION,
            "record_count": len(selected),
            "record_ids": [str(row["run_id"]) for row in selected],
            "records": selected,
        },
    )

    def flatten(value: Mapping[str, Any], prefix: str = "") -> dict[str, Any]:
        output: dict[str, Any] = {}
        for key, item in value.items():
            name = f"{prefix}.{key}" if prefix else str(key)
            if isinstance(item, Mapping):
                output.update(flatten(item, name))
            elif isinstance(item, (list, tuple)):
                output[name] = json.dumps(item, separators=(",", ":"))
            else:
                output[name] = item
        return output

    flattened = [flatten(row) for row in selected]
    fields = sorted({key for row in flattened for key in row})
    with tempfile.NamedTemporaryFile(
        "w",
        encoding="utf-8",
        newline="",
        delete=False,
        dir=store.output_dir,
        suffix=".tmp",
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(flattened)
        temporary = Path(handle.name)
    os.replace(temporary, csv_path)
    return json_path, csv_path


def _active_assignment_hash(ctx: Any) -> str:
    manifest = ctx.data.manifest
    return _base.stable_hash(
        {
            "master_manifest_hash": manifest["master_manifest_hash"],
            "active_participant_identity_ids": manifest[
                "active_participant_identity_ids"
            ],
            "participant_design": manifest["participant_design"],
            "sizes": manifest["sizes"],
            "identity_counts": manifest["identity_counts"],
        }
    )


def _cross_image_summary(record: Mapping[str, Any]) -> tuple[float, tuple[float, float]]:
    linkage = record["best_attack"]["metrics"]["face_linkage"]["cross_image_identity"]
    interval = linkage["ci95_identity_cluster"]
    return float(linkage["top1"]), (float(interval[0]), float(interval[1]))


def _validation_balanced_accuracy(record: Mapping[str, Any]) -> float:
    return float(record["utility"]["validation_balanced_accuracy"])


def _calibration_adequacy(
    ctx: Any,
    records: Sequence[Mapping[str, Any]],
    svd_utility: Mapping[str, Any],
) -> dict[str, Any]:
    """Fail closed unless the validation-only no-noise controls are usable."""

    clean_rows = [
        row
        for row in records
        if row.get("protocol") == "c_gdp"
        and row.get("run_role") == "calibration-clean-adequacy"
    ]
    zero_anchor_rows = [
        row
        for row in records
        if row.get("protocol") == "pa_i_gdp"
        and math.isclose(float(row.get("anchor_v", math.nan)), 0.0, abs_tol=0.0)
    ]
    clean = clean_rows[0] if len(clean_rows) == 1 else {}
    zero_anchor = zero_anchor_rows[0] if len(zero_anchor_rows) == 1 else {}

    def attack_nrmse(
        record: Mapping[str, Any], *, name: str, role: str = "primary"
    ) -> float:
        candidates = [
            attack
            for attack in record.get("attacks", [])
            if attack.get("name") == name and attack.get("role") == role
        ]
        if len(candidates) != 1:
            return math.nan
        try:
            return float(candidates[0]["metrics"]["reduced"]["nrmse"])
        except (KeyError, TypeError, ValueError):
            return math.nan

    try:
        svd_ba = float(svd_utility["validation_balanced_accuracy"])
    except (KeyError, TypeError, ValueError):
        svd_ba = math.nan
    try:
        clean_ba = _validation_balanced_accuracy(clean)
    except (KeyError, TypeError, ValueError):
        clean_ba = math.nan
    c_exact_nrmse = attack_nrmse(clean, name="C-exact")
    pa_zero_mp_nrmse = attack_nrmse(zero_anchor, name="PA-MP")
    pa_zero_gpm_converged = bool(zero_anchor.get("gpm", {}).get("converged", False))
    checks = {
        "validation_only_test_pixels_unopened": not bool(
            ctx.data.manifest["test_pixels_opened"]
        ),
        "public_svd_validation_balanced_accuracy_at_least_0.70": math.isfinite(
            svd_ba
        )
        and svd_ba >= 0.70,
        "clean_c_gdp_validation_balanced_accuracy_at_least_0.70": math.isfinite(
            clean_ba
        )
        and clean_ba >= 0.70,
        "clean_c_gdp_exact_reduced_nrmse_at_most_1e-6": math.isfinite(
            c_exact_nrmse
        )
        and c_exact_nrmse <= _base.C_GDP_EXACT_NRMSE_MAX,
        "zero_anchor_pa_mp_reduced_nrmse_at_most_1e-6": math.isfinite(
            pa_zero_mp_nrmse
        )
        and pa_zero_mp_nrmse <= _base.C_GDP_EXACT_NRMSE_MAX,
        "zero_anchor_gpm_converged": pa_zero_gpm_converged,
    }
    return {
        "gate_version": "p-sweep-validation-controls-v1",
        "checks": checks,
        "svd_validation_balanced_accuracy": svd_ba,
        "clean_c_gdp_validation_balanced_accuracy": clean_ba,
        "svd_minus_clean_c_gdp_balanced_accuracy": svd_ba - clean_ba,
        "svd_minus_c_gdp_balanced_accuracy": svd_ba - clean_ba,
        "svd_vs_c_gdp_difference_is_gating": False,
        "clean_c_gdp_exact_reduced_nrmse": c_exact_nrmse,
        "c_gdp_exact_reduced_recovery": {
            "attack": "C-exact",
            "nrmse": c_exact_nrmse,
        },
        "zero_anchor_pa_mp_reduced_nrmse": pa_zero_mp_nrmse,
        "zero_anchor_gpm_converged": pa_zero_gpm_converged,
        "thresholds": {
            "minimum_validation_balanced_accuracy": 0.70,
            "maximum_exact_reduced_nrmse": _base.C_GDP_EXACT_NRMSE_MAX,
        },
        "adequacy_passed": all(checks.values()),
    }


def _intervals_overlap(
    left: tuple[float, float], right: tuple[float, float]
) -> bool:
    return max(left[0], right[0]) <= min(left[1], right[1])


def _select_anchor_upper(
    records: Sequence[Mapping[str, Any]],
) -> dict[str, Any]:
    """Select the right endpoint of the first linkage-saturation pair.

    Linkage, rather than a single stochastic utility fit, determines whether
    anchor noise has reached the I-GDP privacy regime.  Requiring two adjacent
    levels protects against selecting an isolated point estimate.  The right
    endpoint retains both the transition and a confirmed portion of the
    privacy plateau in the subsequent uniform-noise experiment.
    """

    i_rows = [row for row in records if row.get("protocol") == "i_gdp"]
    if len(i_rows) != 1:
        raise RuntimeError("Calibration requires exactly one I-GDP reference fit")
    i_row = i_rows[0]
    i_utility = _validation_balanced_accuracy(i_row)
    i_linkage, i_interval = _cross_image_summary(i_row)
    anchor_rows = sorted(
        [row for row in records if row.get("protocol") == "pa_i_gdp"],
        key=lambda row: float(row["anchor_v"]),
    )
    evidence: list[dict[str, Any]] = []
    for row in anchor_rows:
        linkage, interval = _cross_image_summary(row)
        utility = _validation_balanced_accuracy(row)
        linkage_gap = abs(linkage - i_linkage)
        chance_in_ci = interval[0] <= _base.LINKAGE_CHANCE <= interval[1]
        i_gdp_ci_overlap = _intervals_overlap(interval, i_interval)
        utility_usable = utility >= MINIMUM_USABLE_BALANCED_ACCURACY
        evidence.append(
            {
                "v": float(row["anchor_v"]),
                "run_id": str(row["run_id"]),
                "validation_balanced_accuracy": utility,
                "utility_gap_to_i_gdp": abs(utility - i_utility),
                "cross_image_top1": linkage,
                "linkage_gap_to_i_gdp": linkage_gap,
                "cross_image_ci95": list(interval),
                "chance_in_ci": chance_in_ci,
                "i_gdp_ci_overlap": i_gdp_ci_overlap,
                "utility_above_balanced_chance": utility_usable,
                "privacy_saturated": bool(
                    linkage_gap <= LINKAGE_TO_I_TOLERANCE
                    and chance_in_ci
                    and i_gdp_ci_overlap
                ),
                "gpm_converged": bool((row.get("gpm") or {}).get("converged", False)),
                "gpm_iterations": (row.get("gpm") or {}).get("iterations"),
                "gpm_termination_reason": (row.get("gpm") or {}).get(
                    "termination_reason"
                ),
            }
        )

    selected: Optional[float] = None
    selected_pair: Optional[list[float]] = None
    for first, second in zip(evidence, evidence[1:]):
        if float(first["v"]) < MINIMUM_SELECTABLE_V:
            continue
        saturated_pair = all(
            bool(item["privacy_saturated"])
            and bool(item["utility_above_balanced_chance"])
            for item in (first, second)
        )
        if saturated_pair:
            selected = float(second["v"])
            selected_pair = [float(first["v"]), float(second["v"])]
            break

    return {
        "rule_version": CALIBRATION_RULE_VERSION,
        "selected_anchor_v_upper": selected,
        "selected_saturation_pair": selected_pair,
        "right_censored": selected is None,
        "selection_target": "cross-image ten-way identity linkage",
        "utility_role": "reported response; balanced-chance usability guard only",
        "endpoint_rule": "right endpoint of earliest qualifying adjacent pair",
        "thresholds": {
            "linkage_to_i_gdp": LINKAGE_TO_I_TOLERANCE,
            "minimum_selectable_v": MINIMUM_SELECTABLE_V,
            "minimum_usable_balanced_accuracy": MINIMUM_USABLE_BALANCED_ACCURACY,
            "chance_linkage_accuracy": _base.LINKAGE_CHANCE,
        },
        "i_gdp_reference": {
            "run_id": str(i_row["run_id"]),
            "validation_balanced_accuracy": i_utility,
            "cross_image_top1": i_linkage,
            "cross_image_ci95": list(i_interval),
        },
        "candidate_evidence": evidence,
    }


def _module_provenance_compatible(
    calibration: Mapping[str, Any], current_modules: Mapping[str, str]
) -> bool:
    """Accept native provenance or an explicitly resolved legacy calibration."""

    record_modules = dict(calibration.get("module_sha256", {}))
    current = dict(current_modules)
    if record_modules == current:
        return True
    resolution = calibration.get("calibration_resolution") or {}
    return bool(
        resolution.get("resolution_version")
        == "completed-record-selection-only-v1"
        and dict(resolution.get("record_generation_module_sha256", {}))
        == record_modules
        and dict(resolution.get("resolver_module_sha256", {})) == current
        and resolution.get("changed_module") == SWEEP_MODULE_FILENAME
        and resolution.get("refits_performed") is False
    )


def _validate_resolution_module_change(
    previous_modules: Mapping[str, str], current_modules: Mapping[str, str]
) -> None:
    previous = dict(previous_modules)
    current = dict(current_modules)
    if not current:
        raise RuntimeError(
            "Resolving the completed calibration requires current module hashes"
        )
    if set(previous) != set(current):
        raise RuntimeError("Legacy and current calibration module sets differ")
    if previous.get(SWEEP_MODULE_FILENAME) != LEGACY_SWEEP_MODULE_SHA256:
        raise RuntimeError("The completed diagnostic is not the reviewed legacy build")
    changed = sorted(name for name in previous if previous[name] != current[name])
    if changed != [SWEEP_MODULE_FILENAME]:
        raise RuntimeError(
            "Selection-only resolution permits changing only the sweep orchestrator; "
            f"observed changes: {changed}"
        )


def _resolved_calibration_summary(
    base: Path,
    current_modules: Mapping[str, str],
) -> Optional[dict[str, Any]]:
    """Promote the reviewed legacy diagnostic using saved records only.

    The original diagnostic, runtime record, and per-fit files remain immutable.
    A new frozen manifest records both the record-generation modules and the
    selector module, making the rule change explicit rather than pretending the
    nine calibration fits were rerun.
    """

    out = base / "calibration_p010"
    frozen_path = out / "frozen_calibration_manifest.json"
    frozen_summary_path = out / "frozen_calibration_summary.json"
    if frozen_path.is_file():
        frozen = json.loads(frozen_path.read_text(encoding="utf-8"))
        resolution = frozen.get("calibration_resolution") or {}
        if not resolution:
            return None
        if not _module_provenance_compatible(frozen, current_modules):
            raise RuntimeError(
                "Resolved calibration was created by a different selector build"
            )
        if not frozen_summary_path.is_file():
            raise FileNotFoundError("Resolved calibration summary is unavailable")
        summary = json.loads(frozen_summary_path.read_text(encoding="utf-8"))
        return {**summary, "resumed_from_completed_stage": True}

    diagnostic_path = out / "calibration_diagnostic_manifest.json"
    runtime_path = out / "stage_runtime.json"
    if not diagnostic_path.is_file() or not runtime_path.is_file():
        return None
    diagnostic = json.loads(diagnostic_path.read_text(encoding="utf-8"))
    legacy_rule = (diagnostic.get("anchor_selection") or {}).get("rule_version")
    if legacy_rule not in LEGACY_CALIBRATION_RULE_VERSIONS:
        return None
    runtime = json.loads(runtime_path.read_text(encoding="utf-8"))
    if not bool(runtime.get("completed")) or runtime.get("phase") != "completed":
        raise RuntimeError("The legacy calibration diagnostic is not complete")
    if bool(diagnostic.get("calibration_passed")):
        raise RuntimeError("A passed calibration must not be promoted as a diagnostic")
    if diagnostic.get("calibration_status") != "right-censored":
        raise RuntimeError("Only the reviewed right-censored diagnostic can be resolved")
    if not bool((diagnostic.get("adequacy") or {}).get("adequacy_passed")):
        raise RuntimeError("The legacy calibration failed its adequacy controls")
    if bool(diagnostic.get("test_pixels_opened")):
        raise RuntimeError("The legacy calibration opened test pixels")
    if tuple(diagnostic.get("p_sweep", ())) != P_SWEEP:
        raise RuntimeError("The legacy calibration participant grid differs")
    if tuple(diagnostic.get("anchor_candidate_grid", ())) != ANCHOR_CALIBRATION_LEVELS:
        raise RuntimeError("The legacy anchor candidate grid differs")

    previous_modules = dict(diagnostic.get("module_sha256", {}))
    _validate_resolution_module_change(previous_modules, current_modules)
    expected_ids = [str(item) for item in diagnostic.get("expected_run_ids", [])]
    if len(expected_ids) != 2 + len(ANCHOR_CALIBRATION_LEVELS):
        raise RuntimeError("The legacy calibration expected-run-ID lock is invalid")
    store = _base.AtomicRunStore(out, stem="calibration_results")
    records, _ = _scheduled_records(store, expected_ids, require_complete=True)
    if _base.stable_hash(records) != diagnostic.get("record_hash"):
        raise RuntimeError("The legacy calibration record hash differs")
    selection = _select_anchor_upper(records)
    selected = selection.get("selected_anchor_v_upper")
    if selected is None:
        raise RuntimeError(
            "The completed calibration is still right-censored under the revised rule"
        )

    resolution = {
        "resolution_version": "completed-record-selection-only-v1",
        "source_diagnostic_path": str(diagnostic_path),
        "source_diagnostic_sha256": _base.file_sha256(diagnostic_path),
        "source_rule_version": legacy_rule,
        "resolved_rule_version": CALIBRATION_RULE_VERSION,
        "record_generation_module_sha256": previous_modules,
        "resolver_module_sha256": dict(current_modules),
        "changed_module": SWEEP_MODULE_FILENAME,
        "refits_performed": False,
        "reused_completed_records": len(records),
        "selected_anchor_v_upper": float(selected),
        "selected_saturation_pair": selection.get("selected_saturation_pair"),
    }
    frozen = {
        **diagnostic,
        "calibration_passed": True,
        "calibration_status": "frozen",
        "anchor_v_upper": float(selected),
        "anchor_lambda_upper": float(selected),
        "anchor_selection": selection,
        "calibration_resolution": resolution,
    }
    _base._write_or_validate_json(
        frozen_path, frozen, "resolved p=10 anchor calibration manifest"
    )
    source_summary_path = out / "calibration_summary.json"
    source_summary = (
        json.loads(source_summary_path.read_text(encoding="utf-8"))
        if source_summary_path.is_file()
        else {}
    )
    summary = {
        **source_summary,
        "mode": "calibration",
        "status": "frozen",
        "output_dir": str(out),
        "manifest_path": str(frozen_path),
        "source_manifest_path": str(diagnostic_path),
        "anchor_v_upper": float(selected),
        "selection": selection,
        "calibration_resolution": resolution,
        "reused_completed_calibration_records": True,
        "frozen_summary_path": str(frozen_summary_path),
    }
    _atomic_json(frozen_summary_path, summary)
    return summary


def _shared_artifact_paths(base_dir: Path) -> dict[str, Path]:
    shared = base_dir / "shared"
    return {
        "root": shared,
        "anchor": shared / "anchors" / "anchor_outer0.pt",
        "anchor_manifest": shared / "anchors" / "anchor_manifest.json",
        "deployment": shared / "deployment" / "gdp_parameters_outer0.pt",
        "deployment_manifest": shared
        / "deployment"
        / "gdp_parameters_manifest.json",
    }


def _ensure_shared_artifacts(
    cfg: ExperimentConfig, base_dir: Path
) -> dict[str, Any]:
    """Create or verify one anchor and one 90-participant GDP parameter bank."""

    paths = _shared_artifact_paths(base_dir)
    bank_cfg = replace(
        cfg,
        n_participants=MAX_PARTICIPANTS,
        mode="full",
        private_lambda_upper=PRIVATE_LAMBDA_UPPER,
        anchor_lambda_upper=1.0,
        anchor_v_upper=1.0,
        output_dir=str(paths["root"]),
    )
    if all(
        path.is_file()
        for path in (
            paths["anchor"],
            paths["anchor_manifest"],
            paths["deployment"],
            paths["deployment_manifest"],
        )
    ):
        anchor_expected = json.loads(
            paths["anchor_manifest"].read_text(encoding="utf-8")
        )
        deployment_expected = json.loads(
            paths["deployment_manifest"].read_text(encoding="utf-8")
        )
        for artifact, expected, label in (
            (paths["anchor"], anchor_expected, "anchor"),
            (paths["deployment"], deployment_expected, "GDP parameter bank"),
        ):
            if Path(str(expected.get("tensor_path", ""))) != artifact:
                raise RuntimeError(f"Persisted {label} path differs from the sweep lock")
            if _base.file_sha256(artifact) != expected.get("tensor_file_sha256"):
                raise RuntimeError(f"Persisted {label} file hash differs")
        anchor_payload = torch.load(
            paths["anchor"], map_location="cpu", weights_only=False
        )
        deployment_payload = torch.load(
            paths["deployment"], map_location="cpu", weights_only=False
        )
        anchor = anchor_payload["anchor"].to(torch.float64)
        rotations = [
            item.to(torch.float64) for item in deployment_payload["rotations"]
        ]
        translations = [
            item.to(torch.float64) for item in deployment_payload["translations"]
        ]
        anchor_spec = _unit._validate_loaded_anchor(
            anchor, bank_cfg, 0, anchor_expected
        )
        deployment_spec = _unit._validate_loaded_gdp_parameters(
            rotations, translations, bank_cfg, 0, deployment_expected
        )
        return {
            "paths": {key: str(value) for key, value in paths.items()},
            "anchor": anchor,
            "anchor_spec": anchor_spec,
            "rotations": rotations,
            "translations": translations,
            "deployment_spec": deployment_spec,
        }

    generator = _base.cp.make_generator(
        _base.named_seed(bank_cfg.seed, "gdp-parameters", 0), "cpu"
    )
    rotations, translations = _base.cp.sample_gdp_parameters(
        MAX_PARTICIPANTS,
        bank_cfg.reduced_dim,
        generator=generator,
        device="cpu",
    )
    raw_deployment_spec = _unit._gdp_parameter_diagnostics(
        rotations, translations, bank_cfg, 0
    )
    deployment_spec = _unit._persist_gdp_parameters(
        rotations,
        translations,
        raw_deployment_spec,
        paths["root"],
        bank_cfg,
    )
    anchor, raw_anchor_spec = _unit._make_centered_unit_isometric_anchor(bank_cfg, 0)
    anchor_spec = _unit._persist_anchor(
        anchor, raw_anchor_spec, paths["root"], bank_cfg
    )
    return {
        "paths": {key: str(value) for key, value in paths.items()},
        "anchor": anchor,
        "anchor_spec": anchor_spec,
        "rotations": rotations,
        "translations": translations,
        "deployment_spec": deployment_spec,
    }


def _active_deployment(
    ctx: Any, shared: Mapping[str, Any]
) -> dict[str, Any]:
    cfg = ctx.cfg
    p = int(cfg.n_participants)
    device = _base.resolve_device(cfg.device)
    rotations = list(shared["rotations"][:p])
    translations = list(shared["translations"][:p])
    active_spec = {
        **dict(shared["deployment_spec"]),
        "participants": p,
        "bank_participants": MAX_PARTICIPANTS,
        "active_participants": p,
        "active_rotations_sha256_float64_c_order": _unit._tensor_sequence_sha256(
            rotations
        ),
        "active_translations_sha256_float64_c_order": _unit._tensor_sequence_sha256(
            translations
        ),
        "prefix_invariant": True,
    }
    return {
        "rotations": [item.to(device) for item in rotations],
        "translations": [item.to(device) for item in translations],
        "anchor": shared["anchor"].to(device),
        "private_rms": float(torch.sqrt(ctx.data.train.reduced.double().square().mean())),
        "anchor_rms": float(shared["anchor_spec"]["entrywise_rms"]),
        "anchor_spec": dict(shared["anchor_spec"]),
        "deployment_spec": active_spec,
    }


def _annotate_record(
    record: Mapping[str, Any], store: Any, cfg: ExperimentConfig
) -> dict[str, Any]:
    annotated = dict(record)
    v = float(annotated.get("anchor_v", 0.0))
    annotated.update(
        {
            "schema_version": SCHEMA_VERSION,
            "n_participants": int(cfg.n_participants),
            "p_sweep": list(P_SWEEP),
            "anchor_mean_translation_sd": v / math.sqrt(cfg.anchor_rows),
            "anchor_noise_to_signal_frobenius": v
            * math.sqrt(cfg.anchor_rows - 1),
            "is_i_gdp_equivalence_control": annotated.get("run_role")
            == "i_gdp_equivalence_control",
        }
    )
    _atomic_json(store.path(str(annotated["run_id"])), annotated)
    return annotated


def _execute(
    ctx: Any,
    store: Any,
    deployment: Mapping[str, Any],
    **specification: Any,
) -> dict[str, Any]:
    record = _unit._execute_fit(ctx, store, deployment, **specification)
    return _annotate_record(record, store, ctx.cfg)


def _stage_config(
    cfg: ExperimentConfig, out: Path
) -> ExperimentConfig:
    out.mkdir(parents=True, exist_ok=True)
    resolved = replace(cfg, output_dir=str(out))
    _base._write_or_validate_json(
        out / "config.json", resolved.to_dict(), "resolved sweep configuration"
    )
    return resolved


def run_anchor_calibration(
    base_dir: str | Path,
    **overrides: Any,
) -> dict[str, Any]:
    """Run or resume the validation-only p=10 anchor calibration."""

    base = Path(base_dir)
    out = base / "calibration_p010"
    resolved = _resolved_calibration_summary(
        base, dict(overrides.get("module_sha256", {}))
    )
    if resolved is not None:
        return resolved
    cfg = ExperimentConfig.for_mode(
        "pilot",
        n_participants=10,
        max_participants=MAX_PARTICIPANTS,
        output_dir=str(out),
        anchor_v_upper=None,
        anchor_lambda_upper=None,
        private_lambda_upper=None,
        **overrides,
    )
    cfg = _stage_config(cfg, out)
    environment = _runtime_environment_lock(cfg, out)
    store = _base.AtomicRunStore(out, stem="calibration_results")
    schedule: list[dict[str, Any]] = [
        {
            "method": "c_gdp",
            "outer": 0,
            "draw": 0,
            "run_role": "calibration-clean-adequacy",
            "level": 0.0,
        },
        {
            "method": "i_gdp",
            "outer": 0,
            "draw": 0,
            "run_role": "calibration-i-gdp-reference",
            "level": 0.0,
        },
        *[
            {
                "method": "pa_i_gdp",
                "outer": 0,
                "draw": draw,
                "run_role": "calibration-anchor-level",
                "level": float(level),
            }
            for draw, level in enumerate(ANCHOR_CALIBRATION_LEVELS)
        ],
    ]
    runtime_path = out / "stage_runtime.json"

    # A finished calibration is immutable. Validate its locks and exact record
    # set without rematerializing CelebA or reloading the face model.
    existing_manifest_path = next(
        (
            path
            for path in (
                out / "frozen_calibration_manifest.json",
                out / "calibration_diagnostic_manifest.json",
            )
            if path.is_file()
        ),
        None,
    )
    if existing_manifest_path is not None and runtime_path.is_file():
        existing_runtime = json.loads(runtime_path.read_text(encoding="utf-8"))
        if bool(existing_runtime.get("completed")):
            existing_manifest = json.loads(
                existing_manifest_path.read_text(encoding="utf-8")
            )
            immutable = {
                "p_sweep": list(P_SWEEP),
                "config_hash": _base.stable_hash(cfg.to_dict()),
                "module_sha256": dict(cfg.module_sha256),
                "runtime_environment_hash": _base.stable_hash(environment),
                "schedule": schedule,
            }
            mismatches = {
                key: {
                    "expected": expected,
                    "actual": existing_manifest.get(key),
                }
                for key, expected in immutable.items()
                if existing_manifest.get(key) != expected
            }
            if mismatches:
                raise RuntimeError(
                    f"Completed calibration immutable provenance mismatch: {mismatches}"
                )
            expected_ids = list(existing_manifest.get("expected_run_ids", []))
            if len(expected_ids) != len(schedule):
                raise RuntimeError("Completed calibration expected-run-ID lock is invalid")
            records, _ = _scheduled_records(
                store, expected_ids, require_complete=True
            )
            if _base.stable_hash(records) != existing_manifest.get("record_hash"):
                raise RuntimeError("Completed calibration record hash differs")
            summary_path = out / "calibration_summary.json"
            if summary_path.is_file():
                summary = json.loads(summary_path.read_text(encoding="utf-8"))
                return {**summary, "resumed_from_completed_stage": True}

    runtime, started = _begin_stage_runtime(
        runtime_path,
        mode="calibration",
        planned_records=len(schedule),
        p=10,
    )
    try:
        ctx = _base._make_context(cfg, out, include_test=False)
        if ctx.data.manifest["test_pixels_opened"]:
            raise AssertionError("Calibration opened participant test pixels")
        runtime["phase"] = "controls-and-calibration-fits"
        _atomic_json(runtime_path, runtime)
        svd_utility = _base._svd_utility_control(ctx)
        shared = _ensure_shared_artifacts(cfg, base)
        deployment = _active_deployment(ctx, shared)
        expected_ids = _expected_run_ids(ctx, schedule)
        for specification in schedule:
            _execute(ctx, store, deployment, **specification)

        records, missing = _scheduled_records(
            store, expected_ids, require_complete=True
        )
        if missing:
            raise AssertionError("A completed calibration cannot have missing records")
        selection = _select_anchor_upper(records)
        selected = selection["selected_anchor_v_upper"]
        adequacy = _calibration_adequacy(ctx, records, svd_utility)
        calibration_passed = bool(
            selected is not None and adequacy["adequacy_passed"]
        )
        status = (
            "frozen"
            if calibration_passed
            else "inadequate-controls"
            if not adequacy["adequacy_passed"]
            else "right-censored"
        )
        manifest = {
            "schema_version": SCHEMA_VERSION,
            "stage": "validation-only-p10-anchor-calibration",
            "calibration_passed": calibration_passed,
            "calibration_status": status,
            "adequacy": adequacy,
            "svd_utility_control": dict(svd_utility),
            "test_pixels_opened": False,
            "p_sweep": list(P_SWEEP),
            "active_participants": 10,
            "master_capacity_participants": MAX_PARTICIPANTS,
            "private_lambda_upper": PRIVATE_LAMBDA_UPPER,
            "anchor_v_lower": 0.0,
            "anchor_v_upper": selected,
            "anchor_lambda_upper": selected,
            "anchor_candidate_grid": list(ANCHOR_CALIBRATION_LEVELS),
            "anchor_selection": selection,
            "anchor_equivalence_control_v": ANCHOR_EQUIVALENCE_V,
            "anchor_equivalence_translation_sd": ANCHOR_EQUIVALENCE_V
            / math.sqrt(cfg.anchor_rows),
            "anchor": dict(deployment["anchor_spec"]),
            "deployment_bank": dict(shared["deployment_spec"]),
            "deployment_active": dict(deployment["deployment_spec"]),
            "master_manifest_hash": ctx.data.manifest["master_manifest_hash"],
            "active_assignment_hash": _active_assignment_hash(ctx),
            "projection": ctx.data.manifest["projection"],
            "basis_matrix_sha256": ctx.data.manifest["projection"][
                "basis_matrix_sha256"
            ],
            "data_manifest_hash": _base.stable_hash(ctx.data.manifest),
            "linkage_manifest_hash": _base.stable_hash(
                ctx.linkage_manifest.to_dict()
            ),
            "config_hash": _base.stable_hash(cfg.to_dict()),
            "module_sha256": dict(cfg.module_sha256),
            "runtime_environment_hash": _base.stable_hash(environment),
            "schedule": schedule,
            "expected_run_ids": expected_ids,
            "record_ids": [str(row["run_id"]) for row in records],
            "record_hash": _base.stable_hash(records),
        }
        manifest_path = out / (
            "frozen_calibration_manifest.json"
            if calibration_passed
            else "calibration_diagnostic_manifest.json"
        )
        _base._write_or_validate_json(
            manifest_path, manifest, "p=10 anchor calibration manifest"
        )
        json_path, csv_path = _aggregate_scheduled_records(store, records)
        rendered = _unit._render_stage(
            {
                "mode": "calibration",
                "status": status,
                "output_dir": str(out),
                "records": records,
                "context": {
                    "test_pixels_opened": False,
                    "adequacy": adequacy,
                    "svd_utility_control": svd_utility,
                    "calibration_manifest": manifest,
                },
                "json_path": str(json_path),
                "csv_path": str(csv_path),
            },
            out,
        )
    except Exception as exc:
        _fail_stage_runtime(
            runtime_path,
            runtime,
            started,
            exc,
            completed_records=len(store.records()),
        )
        raise

    runtime.update(
        {
            "latest_attempt_completed_at_utc": _utc_now(),
            "latest_attempt_wall_seconds": time.perf_counter() - started,
            "completed_records": len(records),
            "summed_fit_runtime_seconds": float(
                sum(float(row.get("runtime_seconds", 0.0)) for row in records)
            ),
            "phase": "completed",
            "completed": True,
            "calibration_passed": calibration_passed,
        }
    )
    _atomic_json(runtime_path, runtime)
    summary = {
        "mode": "calibration",
        "status": status,
        "output_dir": str(out),
        "manifest_path": str(manifest_path),
        "anchor_v_upper": selected,
        "records": len(records),
        "expected_run_ids": expected_ids,
        "json_path": str(json_path),
        "csv_path": str(csv_path),
        "selection": selection,
        "adequacy": adequacy,
        "summary_path": rendered["summary_path"],
        "report_path": rendered["report_path"],
        "utility_plot": rendered["utility_plot"],
        "privacy_utility_plot": rendered["privacy_utility_plot"],
        "gpm_diagnostic_plot": rendered["gpm_diagnostic_plot"],
        "reconstruction_grids": rendered["reconstruction_grids"],
        "stage_runtime_path": str(runtime_path),
        "stage_runtime": runtime,
    }
    _atomic_json(out / "calibration_summary.json", summary)
    return summary


def _load_calibration(base: Path) -> dict[str, Any]:
    path = base / "calibration_p010" / "frozen_calibration_manifest.json"
    if not path.is_file():
        raise FileNotFoundError(
            "A passed p=10 calibration is required before the participant sweep"
        )
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if not manifest.get("calibration_passed") or manifest.get("test_pixels_opened"):
        raise RuntimeError("The calibration manifest is not a passed validation-only run")
    if tuple(manifest.get("p_sweep", ())) != P_SWEEP:
        raise RuntimeError("Calibration participant grid differs from the requested sweep")
    selection = manifest.get("anchor_selection", {})
    selected = manifest.get("anchor_v_upper")
    expected_constants = {
        "schema_version": SCHEMA_VERSION,
        "calibration_status": "frozen",
        "master_capacity_participants": MAX_PARTICIPANTS,
        "anchor_candidate_grid": list(ANCHOR_CALIBRATION_LEVELS),
        "anchor_equivalence_control_v": ANCHOR_EQUIVALENCE_V,
    }
    mismatches = {
        key: {"expected": value, "actual": manifest.get(key)}
        for key, value in expected_constants.items()
        if manifest.get(key) != value
    }
    if mismatches:
        raise RuntimeError(f"Frozen calibration constants differ: {mismatches}")
    if not bool(manifest.get("adequacy", {}).get("adequacy_passed")):
        raise RuntimeError("Frozen calibration does not pass its adequacy gate")
    if selection.get("rule_version") != CALIBRATION_RULE_VERSION:
        raise RuntimeError("Frozen calibration selection rule differs")
    if selected is None or float(selected) != float(
        selection.get("selected_anchor_v_upper", math.nan)
    ):
        raise RuntimeError("Frozen calibration selected bound is internally inconsistent")
    if float(manifest.get("anchor_lambda_upper", math.nan)) != float(selected):
        raise RuntimeError("Frozen calibration anchor bounds are internally inconsistent")
    expected_ids = [str(item) for item in manifest.get("expected_run_ids", [])]
    record_ids = [str(item) for item in manifest.get("record_ids", [])]
    if len(expected_ids) != 2 + len(ANCHOR_CALIBRATION_LEVELS):
        raise RuntimeError("Frozen calibration expected-run-ID lock has the wrong size")
    if expected_ids != record_ids or len(set(expected_ids)) != len(expected_ids):
        raise RuntimeError("Frozen calibration record IDs differ from its schedule lock")
    store = _base.AtomicRunStore(path.parent, stem="calibration_results")
    records, _ = _scheduled_records(store, expected_ids, require_complete=True)
    if _base.stable_hash(records) != manifest.get("record_hash"):
        raise RuntimeError("Frozen calibration record hash differs")
    resolution = manifest.get("calibration_resolution") or {}
    if resolution:
        source_path = Path(str(resolution.get("source_diagnostic_path", "")))
        if not source_path.is_file():
            raise FileNotFoundError("Resolved calibration source diagnostic is unavailable")
        if _base.file_sha256(source_path) != resolution.get(
            "source_diagnostic_sha256"
        ):
            raise RuntimeError("Resolved calibration source diagnostic hash differs")
        if dict(resolution.get("record_generation_module_sha256", {})) != dict(
            manifest.get("module_sha256", {})
        ):
            raise RuntimeError("Resolved calibration record-generation modules differ")
        recomputed_selection = _select_anchor_upper(records)
        if recomputed_selection != selection:
            raise RuntimeError("Resolved calibration selection does not reproduce")
    for key, label in (("anchor", "anchor"), ("deployment_bank", "GDP bank")):
        artifact = manifest.get(key, {})
        tensor_path = Path(str(artifact.get("tensor_path", "")))
        if not tensor_path.is_file():
            raise FileNotFoundError(f"Frozen calibration {label} tensor is unavailable")
        if _base.file_sha256(tensor_path) != artifact.get("tensor_file_sha256"):
            raise RuntimeError(f"Frozen calibration {label} tensor hash differs")
    return manifest


def _full_schedule(cfg: ExperimentConfig) -> list[dict[str, Any]]:
    schedule: list[dict[str, Any]] = [
        {"method": "c_gdp", "outer": 0, "draw": 0, "run_role": "clean", "level": 0.0},
        {"method": "i_gdp", "outer": 0, "draw": 0, "run_role": "clean", "level": 0.0},
    ]
    for draw in range(RANDOM_DRAWS_PER_FAMILY):
        schedule.extend(
            [
                {
                    "method": "c_gdp_an",
                    "outer": 0,
                    "draw": draw,
                    "run_role": "random",
                    "level": _base._lambda_draw(
                        cfg, PRIVATE_LAMBDA_UPPER, draw, "private"
                    ),
                },
                {
                    "method": "pa_i_gdp",
                    "outer": 0,
                    "draw": draw,
                    "run_role": "random",
                    "level": _base._lambda_draw(
                        cfg, float(cfg.anchor_v_upper), draw, "anchor"
                    ),
                },
            ]
        )
    schedule.extend(
        [
            {
                "method": "c_gdp_an",
                "outer": 0,
                "draw": 100,
                "run_role": "endpoint_zero",
                "level": 0.0,
            },
            {
                "method": "pa_i_gdp",
                "outer": 0,
                "draw": 100,
                "run_role": "endpoint_zero",
                "level": 0.0,
            },
            {
                "method": "c_gdp_an",
                "outer": 0,
                "draw": 101,
                "run_role": "endpoint_upper",
                "level": PRIVATE_LAMBDA_UPPER,
            },
            {
                "method": "pa_i_gdp",
                "outer": 0,
                "draw": 101,
                "run_role": "endpoint_upper",
                "level": float(cfg.anchor_v_upper),
            },
            {
                "method": "pa_i_gdp",
                "outer": 0,
                "draw": 102,
                "run_role": "i_gdp_equivalence_control",
                "level": ANCHOR_EQUIVALENCE_V,
            },
        ]
    )
    if len(schedule) != 207:
        raise AssertionError("The participant-sweep full schedule must contain 207 fits")
    return schedule


def run_full_for_p(
    p: int,
    base_dir: str | Path,
    **overrides: Any,
) -> dict[str, Any]:
    """Run or resume one 207-fit participant-count condition."""

    if int(p) not in P_SWEEP:
        raise ValueError(f"p must be one of {P_SWEEP}")
    base = Path(base_dir)
    calibration = _load_calibration(base)
    anchor_upper = float(calibration["anchor_v_upper"])
    out = base / f"p{int(p):03d}" / "full"
    cfg = ExperimentConfig.for_mode(
        "full",
        n_participants=int(p),
        max_participants=MAX_PARTICIPANTS,
        output_dir=str(out),
        private_lambda_upper=PRIVATE_LAMBDA_UPPER,
        anchor_lambda_upper=anchor_upper,
        anchor_v_upper=anchor_upper,
        **overrides,
    )
    cfg = _stage_config(cfg, out)
    environment = _runtime_environment_lock(cfg, out)
    store = _base.AtomicRunStore(out)
    schedule = _full_schedule(cfg)
    runtime_path = out / "stage_runtime.json"
    manifest_path = out / "full_run_manifest.json"
    completion_path = out / "completion_summary.json"
    calibration_path = base / "calibration_p010" / "frozen_calibration_manifest.json"
    calibration_sha256 = _base.file_sha256(calibration_path)

    # Completed stages can return without loading CelebA. All values that can
    # affect a run ID are checked against their immutable manifest first.
    if manifest_path.is_file() and runtime_path.is_file() and completion_path.is_file():
        existing_runtime = json.loads(runtime_path.read_text(encoding="utf-8"))
        if bool(existing_runtime.get("completed")):
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            immutable = {
                "schedule_version": SCHEDULE_VERSION,
                "p": int(p),
                "p_sweep": list(P_SWEEP),
                "schedule": schedule,
                "calibration_manifest_sha256": calibration_sha256,
                "config_hash": _base.stable_hash(cfg.to_dict()),
                "module_sha256": dict(cfg.module_sha256),
                "runtime_environment_hash": _base.stable_hash(environment),
            }
            mismatches = {
                key: {"expected": expected, "actual": manifest.get(key)}
                for key, expected in immutable.items()
                if manifest.get(key) != expected
            }
            if mismatches:
                raise RuntimeError(
                    f"Completed p={p} immutable provenance mismatch: {mismatches}"
                )
            expected_ids = list(manifest.get("expected_run_ids", []))
            if len(expected_ids) != len(schedule) or len(set(expected_ids)) != len(
                schedule
            ):
                raise RuntimeError(f"Completed p={p} expected-run-ID lock is invalid")
            records, _ = _scheduled_records(
                store, expected_ids, require_complete=True
            )
            completion = json.loads(completion_path.read_text(encoding="utf-8"))
            if completion.get("record_hash") != _base.stable_hash(records):
                raise RuntimeError(f"Completed p={p} record hash differs")
            aggregate_path = Path(str(completion.get("json_path", "")))
            report_paths = [
                Path(str(completion.get(key, "")))
                for key in (
                    "csv_path",
                    "summary_path",
                    "report_path",
                    "utility_plot",
                    "privacy_utility_plot",
                    "gpm_diagnostic_plot",
                )
            ]
            if aggregate_path.is_file() and all(path.is_file() for path in report_paths):
                aggregate = json.loads(aggregate_path.read_text(encoding="utf-8"))
                if aggregate.get("record_ids") == expected_ids:
                    return {**completion, "resumed_from_completed_stage": True}

    runtime, started = _begin_stage_runtime(
        runtime_path,
        mode="full",
        planned_records=len(schedule),
        p=int(p),
    )
    expected_ids: list[str] = []
    try:
        if not _module_provenance_compatible(calibration, cfg.module_sha256):
            raise RuntimeError(
                "Full run modules differ from both the calibration records and "
                "the reviewed selection-only resolver"
            )
        # Verify allocation and reducer compatibility without touching test pixels.
        # Only after these fail-closed checks do we build the full test context.
        validation_data = _base.prepare_celeba_data(cfg, include_test=False)
        if (
            validation_data.manifest["master_manifest_hash"]
            != calibration["master_manifest_hash"]
        ):
            raise RuntimeError("Full run and calibration master allocations differ")
        if (
            validation_data.manifest["projection"]["basis_matrix_sha256"]
            != calibration["projection"]["basis_matrix_sha256"]
        ):
            raise RuntimeError("Full run and calibration public SVD bases differ")
        del validation_data
        gc.collect()
        ctx = _base._make_context(cfg, out, include_test=True)
        if len(ctx.linkage_manifest.query_identities) != 100:
            raise AssertionError(
                "Full mode must evaluate all 100 target identity clusters"
            )
        if (
            ctx.data.manifest["master_manifest_hash"]
            != calibration["master_manifest_hash"]
        ):
            raise RuntimeError("Full run and calibration master allocations differ")
        if (
            ctx.data.manifest["projection"]["basis_matrix_sha256"]
            != calibration["projection"]["basis_matrix_sha256"]
        ):
            raise RuntimeError("Full run and calibration public SVD bases differ")
        shared = _ensure_shared_artifacts(cfg, base)
        deployment = _active_deployment(ctx, shared)
        expected_ids = _expected_run_ids(ctx, schedule)
        provenance = {
            "master_manifest_hash": ctx.data.manifest["master_manifest_hash"],
            "active_assignment_hash": _active_assignment_hash(ctx),
            "basis_matrix_sha256": ctx.data.manifest["projection"][
                "basis_matrix_sha256"
            ],
            "data_manifest_hash": _base.stable_hash(ctx.data.manifest),
            "linkage_manifest_hash": _base.stable_hash(
                ctx.linkage_manifest.to_dict()
            ),
            "config_hash": _base.stable_hash(cfg.to_dict()),
            "module_sha256": dict(cfg.module_sha256),
            "runtime_environment_hash": _base.stable_hash(environment),
            "calibration_manifest_sha256": calibration_sha256,
        }
        manifest = {
            "schema_version": SCHEMA_VERSION,
            "schedule_version": SCHEDULE_VERSION,
            "p": int(p),
            "p_sweep": list(P_SWEEP),
            "planned_fits": len(schedule),
            "random_draws_per_noisy_family": RANDOM_DRAWS_PER_FAMILY,
            "anchor_v_upper": anchor_upper,
            "anchor_equivalence_control_v": ANCHOR_EQUIVALENCE_V,
            "private_lambda_upper": PRIVATE_LAMBDA_UPPER,
            "uniform_noise_seed_rule": "named_seed(seed,uniform-noise,family,draw)",
            "participant_noise_seed_rule": (
                "named_seed(seed,noise,family,outer,draw,split,participant)"
            ),
            "schedule": schedule,
            "expected_run_ids": expected_ids,
            "calibration_manifest_sha256": calibration_sha256,
            "master_manifest_hash": provenance["master_manifest_hash"],
            "active_assignment_hash": provenance["active_assignment_hash"],
            "basis_matrix_sha256": provenance["basis_matrix_sha256"],
            "data_manifest_hash": provenance["data_manifest_hash"],
            "linkage_manifest_hash": provenance["linkage_manifest_hash"],
            "config_hash": provenance["config_hash"],
            "module_sha256": provenance["module_sha256"],
            "runtime_environment_hash": provenance["runtime_environment_hash"],
            "provenance": provenance,
            "shared_anchor": dict(deployment["anchor_spec"]),
            "shared_deployment": dict(deployment["deployment_spec"]),
        }
        _base._write_or_validate_json(manifest_path, manifest, "full schedule")
        runtime["phase"] = "scheduled-fits"
        runtime["expected_run_ids_hash"] = _base.stable_hash(expected_ids)
        _atomic_json(runtime_path, runtime)
        for specification in schedule:
            _execute(ctx, store, deployment, **specification)
    except Exception as exc:
        completed_records = (
            len(_scheduled_records(store, expected_ids, require_complete=False)[0])
            if expected_ids
            else 0
        )
        _fail_stage_runtime(
            runtime_path,
            runtime,
            started,
            exc,
            completed_records=completed_records,
        )
        raise

    try:
        records, missing = _scheduled_records(
            store, expected_ids, require_complete=True
        )
        if missing:
            raise AssertionError("A completed full stage cannot have missing records")
        all_ids = {str(row["run_id"]) for row in store.records()}
        unexpected_ids = sorted(all_ids - set(expected_ids))
        json_path, csv_path = _aggregate_scheduled_records(store, records)
        rendered = _unit._render_stage(
            {
                "mode": "full",
                "status": {"completed": len(records), "planned": len(schedule)},
                "output_dir": str(out),
                "records": records,
                "context": {
                    "calibration": calibration,
                    "provenance": provenance,
                    "test_pixels_opened": True,
                    "unexpected_run_ids_ignored": unexpected_ids,
                },
                "json_path": str(json_path),
                "csv_path": str(csv_path),
            },
            out,
        )
        record_hash = _base.stable_hash(records)
    except Exception as exc:
        completed_records = len(
            _scheduled_records(store, expected_ids, require_complete=False)[0]
        )
        _fail_stage_runtime(
            runtime_path,
            runtime,
            started,
            exc,
            completed_records=completed_records,
        )
        raise
    runtime.update(
        {
            "latest_attempt_completed_at_utc": _utc_now(),
            "latest_attempt_wall_seconds": time.perf_counter() - started,
            "completed_records": len(records),
            "summed_fit_runtime_seconds": float(
                sum(float(row.get("runtime_seconds", 0.0)) for row in records)
            ),
            "unexpected_records_ignored": len(unexpected_ids),
            "phase": "completed",
            "completed": not missing and len(records) == len(expected_ids),
        }
    )
    _atomic_json(runtime_path, runtime)
    result = {
        "mode": "full",
        "p": int(p),
        "status": {"completed": len(records), "planned": len(expected_ids)},
        "output_dir": str(out),
        "manifest_path": str(manifest_path),
        "expected_run_ids": expected_ids,
        "record_hash": record_hash,
        "unexpected_run_ids_ignored": unexpected_ids,
        "json_path": str(json_path),
        "csv_path": str(csv_path),
        "summary_path": rendered["summary_path"],
        "report_path": rendered["report_path"],
        "utility_plot": rendered["utility_plot"],
        "privacy_utility_plot": rendered["privacy_utility_plot"],
        "gpm_diagnostic_plot": rendered["gpm_diagnostic_plot"],
        "reconstruction_grid": rendered["reconstruction_grid"],
        "reconstruction_grids": rendered["reconstruction_grids"],
        "stage_runtime_path": str(runtime_path),
        "stage_runtime": runtime,
    }
    _atomic_json(completion_path, result)
    return result


def run_sequential_sweep(
    base_dir: str | Path,
    **overrides: Any,
) -> dict[str, Any]:
    """Calibrate at p=10, then run p=10,30,50,70,90 in order."""

    base = Path(base_dir)
    base.mkdir(parents=True, exist_ok=True)
    calibration = run_anchor_calibration(base, **overrides)
    if calibration["status"] != "frozen":
        raise RuntimeError(
            "Calibration was not frozen; inspect its adequacy and saturation evidence"
        )
    status_path = base / "sequential_sweep_status.json"
    existing = (
        json.loads(status_path.read_text(encoding="utf-8"))
        if status_path.is_file()
        else {}
    )
    status: dict[str, Any] = {
        "schema_version": SCHEMA_VERSION,
        "p_sweep": list(P_SWEEP),
        "anchor_v_upper": calibration["anchor_v_upper"],
        "anchor_equivalence_control_v": ANCHOR_EQUIVALENCE_V,
        "first_started_at_utc": existing.get("first_started_at_utc", _utc_now()),
        "latest_attempt_started_at_utc": _utc_now(),
        "completed_p": list(existing.get("completed_p", [])),
        "active_p": None,
        "completed": False,
    }
    _atomic_json(status_path, status)
    summaries: dict[str, Any] = {}
    for p in P_SWEEP:
        status["active_p"] = p
        _atomic_json(status_path, status)
        summary = run_full_for_p(p, base, **overrides)
        summaries[str(p)] = summary
        if summary["status"]["completed"] == summary["status"]["planned"]:
            status["completed_p"] = sorted(
                set(map(int, status["completed_p"])) | {int(p)}
            )
        status["active_p"] = None
        _atomic_json(status_path, status)
    status.update(
        {
            "latest_attempt_completed_at_utc": _utc_now(),
            "completed": tuple(status["completed_p"]) == P_SWEEP,
        }
    )
    _atomic_json(status_path, status)
    return {
        "mode": "sequential-p-sweep",
        "calibration": calibration,
        "status": status,
        "summaries": summaries,
        "status_path": str(status_path),
    }


def run_static_self_tests() -> dict[str, Any]:
    def calibration_record(
        protocol: str,
        run_id: str,
        balanced_accuracy: float,
        linkage_top1: float,
        linkage_ci95: tuple[float, float],
        *,
        anchor_v: Optional[float] = None,
    ) -> dict[str, Any]:
        record: dict[str, Any] = {
            "protocol": protocol,
            "run_id": run_id,
            "utility": {"validation_balanced_accuracy": balanced_accuracy},
            "best_attack": {
                "metrics": {
                    "face_linkage": {
                        "cross_image_identity": {
                            "top1": linkage_top1,
                            "ci95_identity_cluster": list(linkage_ci95),
                        }
                    }
                }
            },
            "gpm": {},
        }
        if anchor_v is not None:
            record["anchor_v"] = anchor_v
        return record

    observed_calibration = [
        calibration_record(
            "i_gdp",
            "i-reference",
            0.6333139578,
            0.1070588235,
            (0.0658823529, 0.1564705882),
        ),
        calibration_record(
            "pa_i_gdp", "v0", 0.8163310454, 0.8141176471,
            (0.7529117647, 0.8752941176), anchor_v=0.0,
        ),
        calibration_record(
            "pa_i_gdp", "v00625", 0.6055646706, 0.1376470588,
            (0.0917647059, 0.1870588235), anchor_v=0.0625,
        ),
        calibration_record(
            "pa_i_gdp", "v0125", 0.6423838973, 0.1105882353,
            (0.0623529412, 0.1659117647), anchor_v=0.125,
        ),
        calibration_record(
            "pa_i_gdp", "v025", 0.5912454689, 0.0788235294,
            (0.0423529412, 0.1188235294), anchor_v=0.25,
        ),
        calibration_record(
            "pa_i_gdp", "v05", 0.6060497810, 0.0905882353,
            (0.0482352941, 0.1388235294), anchor_v=0.5,
        ),
    ]
    observed_selection = _select_anchor_upper(observed_calibration)
    cfg = ExperimentConfig.for_mode(
        "full",
        n_participants=90,
        max_participants=90,
        private_lambda_upper=PRIVATE_LAMBDA_UPPER,
        anchor_lambda_upper=0.25,
        anchor_v_upper=0.25,
        device="cpu",
    )
    schedule = _full_schedule(cfg)
    checks = {
        "schema_v3": SCHEMA_VERSION == 3,
        "p_grid": P_SWEEP == (10, 30, 50, 70, 90),
        "master_capacity_90": cfg.max_participants == 90,
        "identity_count_100": cfg.identities_per_participant == 100,
        "split_70_15_15": _base.SPLIT_IDENTITY_COUNTS
        == {"train": 70, "val": 15, "test": 15},
        "schedule_207": len(schedule) == 207,
        "equivalence_control_once": sum(
            item["run_role"] == "i_gdp_equivalence_control" for item in schedule
        )
        == 1,
        "equivalence_v_sqrt_r": math.isclose(
            ANCHOR_EQUIVALENCE_V, math.sqrt(cfg.anchor_rows)
        ),
        "calibration_lower_zero": ANCHOR_CALIBRATION_LEVELS[0] == 0.0,
        "calibration_extends_old_bound": ANCHOR_CALIBRATION_LEVELS[-1] > 0.0625,
        "privacy_rule_versioned": CALIBRATION_RULE_VERSION
        == "i-gdp-linkage-saturation-right-endpoint-v2",
        "observed_calibration_selects_025": math.isclose(
            float(observed_selection["selected_anchor_v_upper"]), 0.25
        ),
        "observed_pair_is_0125_025": observed_selection[
            "selected_saturation_pair"
        ]
        == [0.125, 0.25],
        "utility_not_matched_to_i_gdp": "utility_to_i_gdp"
        not in observed_selection["thresholds"],
    }
    report = {"checks": checks, "all_passed": all(checks.values())}
    if not report["all_passed"]:
        raise AssertionError(report)
    return report


make_config = _base.make_config
print_config = _base.print_config
prepare_celeba_data = _base.prepare_celeba_data
resolve_device = _base.resolve_device


__all__ = [
    "ExperimentConfig",
    "P_SWEEP",
    "ANCHOR_CALIBRATION_LEVELS",
    "ANCHOR_EQUIVALENCE_V",
    "run_anchor_calibration",
    "run_full_for_p",
    "run_sequential_sweep",
    "run_static_self_tests",
    "resolve_device",
]


In [ ]:

# 4. Import reviewed implementation and hash all embedded modules
import hashlib
import importlib
import json
from pathlib import Path

import core_protocols
import celeba_public_svd_identity100_runner

importlib.reload(core_protocols)
for sentinel in (
    "_unit_anchor_base_config",
    "_unit_anchor_original_diag",
    "_unit_anchor_original_execute_fit",
    "_unit_anchor_original_load_frozen_pilot",
    "_unit_anchor_original_render_stage",
    "_unit_anchor_original_static_tests",
):
    if hasattr(celeba_public_svd_identity100_runner, sentinel):
        delattr(celeba_public_svd_identity100_runner, sentinel)
importlib.reload(celeba_public_svd_identity100_runner)

import celeba_public_svd_identity100_unit_isometric_runner
importlib.reload(celeba_public_svd_identity100_unit_isometric_runner)

import celeba_public_svd_identity100_p_sweep_runner
importlib.reload(celeba_public_svd_identity100_p_sweep_runner)

module_hashes = {
    name: hashlib.sha256(Path(name).read_bytes()).hexdigest()
    for name in (
        "core_protocols.py",
        "celeba_public_svd_identity100_runner.py",
        "celeba_public_svd_identity100_unit_isometric_runner.py",
        "celeba_public_svd_identity100_p_sweep_runner.py",
    )
}
common_overrides = {
    "cache_dir": str(CACHE_ROOT),
    "local_cache_dir": str(LOCAL_ROOT),
    "device": DEVICE,
    "svd_q": int(SVD_Q),
    "svd_niter": int(SVD_NITER),
    "module_sha256": module_hashes,
}
print(json.dumps({"module_sha256": module_hashes, "overrides": common_overrides}, indent=2))


## 5. Algebraic and allocation acceptance tests


In [ ]:

self_test_device = str(celeba_public_svd_identity100_p_sweep_runner.resolve_device(DEVICE))
core_self_tests = core_protocols.run_self_tests(device=self_test_device)
sweep_self_tests = celeba_public_svd_identity100_p_sweep_runner.run_static_self_tests()
self_test_report = {
    "resolved_device": self_test_device,
    "core_protocols": core_self_tests,
    "participant_sweep": sweep_self_tests,
    "all_passed": bool(core_self_tests.get("all_passed", False))
    and bool(sweep_self_tests.get("all_passed", False)),
}
if not self_test_report["all_passed"]:
    raise AssertionError(self_test_report)
print(json.dumps(self_test_report, indent=2, default=str))


## 6. Execute with atomic resume


In [ ]:
runner = celeba_public_svd_identity100_p_sweep_runner
calibration_result = runner.run_anchor_calibration(RESULTS_ROOT, **common_overrides)
print(json.dumps(calibration_result, indent=2, default=str))
if calibration_result["status"] != "frozen":
    raise RuntimeError(
        "Calibration was not frozen. Inspect its saved adequacy and saturation evidence before running test-data fits."
    )


## Full experiment: \(p=10\)


In [ ]:
if EXECUTE_FULL_SWEEP:
    p010_result = runner.run_full_for_p(10, RESULTS_ROOT, **common_overrides)
    print(json.dumps(p010_result, indent=2, default=str))
else:
    p010_result = {"status": "skipped-by-control", "p": 10}


## Full experiment: \(p=30\)


In [ ]:
if EXECUTE_FULL_SWEEP:
    p030_result = runner.run_full_for_p(30, RESULTS_ROOT, **common_overrides)
    print(json.dumps(p030_result, indent=2, default=str))
else:
    p030_result = {"status": "skipped-by-control", "p": 30}


## Full experiment: \(p=50\)


In [ ]:
if EXECUTE_FULL_SWEEP:
    p050_result = runner.run_full_for_p(50, RESULTS_ROOT, **common_overrides)
    print(json.dumps(p050_result, indent=2, default=str))
else:
    p050_result = {"status": "skipped-by-control", "p": 50}


## Full experiment: \(p=70\)


In [ ]:
if EXECUTE_FULL_SWEEP and EXECUTE_EXPLORATORY_P70_P90:
    p070_result = runner.run_full_for_p(70, RESULTS_ROOT, **common_overrides)
    print(json.dumps(p070_result, indent=2, default=str))
else:
    p070_result = {"status": "skipped-by-control", "p": 70}


## Full experiment: \(p=90\)


In [ ]:
if EXECUTE_FULL_SWEEP and EXECUTE_EXPLORATORY_P70_P90:
    p090_result = runner.run_full_for_p(90, RESULTS_ROOT, **common_overrides)
    print(json.dumps(p090_result, indent=2, default=str))
else:
    p090_result = {"status": "skipped-by-control", "p": 90}


## Persisted completion summary


In [ ]:
completion = {
    "calibration": calibration_result,
    "participant_results": {
        "10": p010_result,
        "30": p030_result,
        "50": p050_result,
        "70": p070_result,
        "90": p090_result,
    },
}
summary_path = RESULTS_ROOT / "notebook_completion_summary.json"
runner._atomic_json(summary_path, completion)
print(json.dumps({
    "summary_path": str(summary_path),
    "calibrated_anchor_v_upper": calibration_result["anchor_v_upper"],
    "p_status": {
        key: value.get("status") for key, value in completion["participant_results"].items()
    },
}, indent=2, default=str))


## Interpretation

The calibration and every participant-count condition are independently resumable from Drive. The primary participant-count comparison uses identical normalized scalar noise draws and participant-indexed Gaussian noise tensors across \(p\), while the data, GDP parameter bank, and deployment are nested prefixes. Linkage remains the same participant-0-to-participant-1 ten-way threat model with 100 protected identity clusters; increasing \(p\) changes collaboration and training data, not the colluding coalition size.
